# Module 3 — MCP Architecture & Cross-LLM Evaluation
## Version B Runtime — DeepSeek V4 Flash 0731

**Purpose:** Restore persistent BM25/FTS shards + MCP + DeepSeek V4 Flash 0731 evaluation.

**Repository note**
- This notebook preserves the executed experiment outputs for auditability and portfolio review.
- API credentials are loaded from environment variables or Google Colab secrets; no literal API keys are stored in this notebook.
- Large corpora and persistent BM25/DuckDB artifacts are intentionally excluded from Git and are accessed remotely or through the documented Kaggle artifact.
- Frozen benchmark, Top-K, MCP search limits and system-prompt controls are preserved from the executed experiment.

> Reproducibility dependencies and required secrets are documented in `README.md` and `requirements-module3.txt`.


```text
SECTION 0 — Runtime Environment
Cell 0 — Install Runtime Libraries
Cell 1 — Imports + Global Runtime Config

SECTION 1 — Restore Persistent Knowledge Base
Cell 2 — Kaggle Authentication + Dataset Access
Cell 3 — Download Persistent BM25 Artifact
Cell 4 — Validate Restored BM25 Artifact

SECTION 2 — Reconstruct Frozen Version B Retrieval
Cell 5 — Shard Registry + Deterministic Dynamic Router
Cell 6 — Dynamic Routed BM25 Retrieval Engine
Cell 7 — Three-Route Retrieval Validation

SECTION 3 — MCP Knowledge Service
Cell 8 — Unified MCP Telecom Knowledge Search Tool
Cell 9 — Three-Route MCP Tool Validation

SECTION 4 — DeepSeek V4 + MCP Orchestration
Cell 10 — DeepSeek V4 Setup + Frozen Shared System Prompt
Cell 11 — End-to-End MCP Orchestration
Cell 12 — Three-Route End-to-End Pilot

SECTION 5 — Version B Evaluation
Cell 13 — Module 3 Evaluation Benchmark v2


============================================================
VERSION B — FROZEN RUNTIME ARCHITECTURE
============================================================

Persistent artifact
-------------------
21 DuckDB BM25/FTS shards
1,780,938 indexed records
~69 GB
restored from Kaggle without rebuilding the corpus or indexes


End-to-end path
---------------
User Question
→ DeepSeek V4 Flash 0731
→ ONE MCP tool: search_telecom_knowledge(query, top_k=5)
→ deterministic 3GPP / TCC / Hybrid router
→ selected persistent BM25 shards
→ global BM25 ranking + bounded evidence selection
→ DeepSeek V4 Flash 0731
→ final answer


A/B routing control
-------------------
Version B uses the same high-level route semantics as Version A.

3GPP route
→ dedicated source_family == "3GPP" shards

TCC route
→ selected source_family == "TCC" collections

Hybrid route
→ dedicated 3GPP + selected non-3GPP TCC evidence

TCC 3GPP-TSG contribution material is excluded from Hybrid retrieval.


Public MCP contract
-------------------
The LLM-facing tool exposes only:
- query
- top_k

The LLM does NOT select:
- retrieval profile
- source family
- collection
- shard

Internal source/search groups may remain as implementation primitives;
they are not public orchestration controls.


Frozen controls
---------------
Provider: OpenRouter
Maximum MCP searches: 3
Top-K: 5
Maximum evidence excerpt: 2,500 characters/source
Maximum output tokens: 1,800
Target answer: approximately 300–500 words
Benchmark: 5 × 3GPP | 2 × TCC | 1 × Hybrid
Benchmark SHA-256: d40c0090c371f0a99ea6057bbb3fa8024e6b7d4174e037e7a6cf9fef9b9667f5


Validation order
----------------
restore artifact
→ validate all 21 shards
→ reconstruct deterministic router
→ validate 3GPP / TCC / Hybrid retrieval directly
→ validate the same routes through real FastMCP Client
→ connect the LLM
→ run three-route end-to-end pilot
→ run frozen Benchmark v2

Hybrid benchmark success accepts either one explicit Hybrid search or
complementary 3GPP+TCC searches whose accumulated evidence covers both
source families.

Do not tune the router or BM25 retrieval engine after observing formal
benchmark failures.
```


# **SECTION 0 — Runtime Environment**

## **Cell 0 — Install Runtime Libraries**

In [1]:
# ============================================================
# CELL 0 — INSTALL RUNTIME LIBRARIES
# ============================================================


# ============================================================
# CORE RUNTIME DEPENDENCIES
# ============================================================

!pip install -q \
    duckdb \
    pandas \
    fastmcp \
    anthropic \
    kagglehub


# ============================================================
# KAGGLE CLI
# ============================================================

# Keep the Kaggle CLI available for authentication /
# dataset-access checks and lightweight file inspection.
!pip install -q --upgrade kaggle


# ============================================================
# VERIFY RUNTIME INSTALLATION
# ============================================================

from importlib.metadata import (
    version,
    PackageNotFoundError
)


RUNTIME_PACKAGES = [
    "duckdb",
    "pandas",
    "fastmcp",
    "anthropic",
    "kagglehub",
    "kaggle"
]


print("=" * 80)
print("VERSION B — RUNTIME DEPENDENCY CHECK")
print("=" * 80)


for package in RUNTIME_PACKAGES:

    try:

        package_version = (
            version(package)
        )

        print(
            f"{package:<15} : "
            f"{package_version}"
        )

    except PackageNotFoundError:

        raise RuntimeError(
            f"Required runtime package "
            f"'{package}' is not installed."
        )


print("=" * 80)

print(
    "✓ Version B runtime dependencies installed."
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 858.1/858.1 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.2/237.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.9/357.9 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.0/273.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.4/196.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Observation — Runtime Environment**

The Runtime notebook installs only the libraries required to restore, search and orchestrate the persisted Version B Knowledge Base.

***Key Decision:*** Keep the Runtime environment separate from the corpus-build toolchain.

## **Cell 1 — Imports + Global Runtime Config**

In [2]:
# ============================================================
# CELL 1 — IMPORTS + GLOBAL RUNTIME CONFIGURATION
# ============================================================

import os
import re
import json
import math
import time
import shutil
import subprocess

import duckdb
import pandas as pd
import kagglehub

from pathlib import Path
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)


# ============================================================
# PERSISTED KNOWLEDGE BASE
# ============================================================

KAGGLE_DATASET = (
    "cliffordimaguezegie/"
    "telecom-bm25-indexed-knowledge-base"
)

EXPECTED_SHARDS = 21
EXPECTED_RECORDS = 1_780_938


# ============================================================
# LOCAL RUNTIME PATHS
# ============================================================

WORK_DIR = Path(
    "/tmp/telecom_bm25_runtime"
)

SHARD_DIR = (
    WORK_DIR
    / "bm25_shards"
)

SHARD_MANIFEST_PATH = (
    SHARD_DIR
    / "bm25_shard_manifest.csv"
)


WORK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SHARD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# DUCKDB / BM25 CONFIGURATION
# ============================================================

FTS_TABLE = (
    "telecom_shard"
)

FTS_SCHEMA = (
    "fts_main_telecom_shard"
)

BM25_K = 1.2
BM25_B = 0.75


# ============================================================
# FROZEN OPTIMIZED RETRIEVAL CONFIGURATION
# ============================================================

MAX_CANDIDATE_TERMS = 3

MAX_SEARCH_WORKERS = 6

TOP_K_RESULTS = 5


# ============================================================
# MCP RUNTIME LIMITS
# ============================================================

MCP_EXCERPT_CHARS = 2500

MAX_RETRIEVED_SOURCES = 5

MAX_MCP_SEARCHES = 3


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 80)
print("VERSION B — TELECOM AI MCP RUNTIME CONFIGURATION")
print("=" * 80)

print(
    f"Kaggle Dataset       : "
    f"{KAGGLE_DATASET}"
)

print(
    f"Runtime Directory    : "
    f"{WORK_DIR}"
)

print(
    f"BM25 Shard Directory : "
    f"{SHARD_DIR}"
)

print(
    f"Expected Shards      : "
    f"{EXPECTED_SHARDS}"
)

print(
    f"Expected Records     : "
    f"{EXPECTED_RECORDS:,}"
)

print(
    f"Top-K Results        : "
    f"{TOP_K_RESULTS}"
)

print(
    f"High-IDF Terms       : "
    f"{MAX_CANDIDATE_TERMS}"
)

print(
    f"Max Search Workers   : "
    f"{MAX_SEARCH_WORKERS}"
)

print(
    f"MCP Excerpt          : "
    f"{MCP_EXCERPT_CHARS:,} chars/source"
)

print(
    f"Max MCP Searches     : "
    f"{MAX_MCP_SEARCHES}"
)

print("=" * 80)

print(
    "✓ Version B runtime configuration loaded."
)

VERSION B — TELECOM AI MCP RUNTIME CONFIGURATION
Kaggle Dataset       : cliffordimaguezegie/telecom-bm25-indexed-knowledge-base
Runtime Directory    : /tmp/telecom_bm25_runtime
BM25 Shard Directory : /tmp/telecom_bm25_runtime/bm25_shards
Expected Shards      : 21
Expected Records     : 1,780,938
Top-K Results        : 5
High-IDF Terms       : 3
Max Search Workers   : 6
MCP Excerpt          : 2,500 chars/source
Max MCP Searches     : 3
✓ Version B runtime configuration loaded.


**Observation — Runtime Configuration**

The frozen Version B retrieval and MCP parameters are defined centrally without performing any external authentication.

***Key Decision:*** Separate runtime configuration from external-service authentication.

# **SECTION 1 — Restore Persistent Knowledge Base**

## **Cell 2 — Kaggle Authentication + Dataset Access**

In [3]:
# ============================================================
# CELL 2 — KAGGLE AUTHENTICATION + DATASET ACCESS
# ============================================================

from google.colab import userdata


# ============================================================
# LOAD KAGGLE API TOKEN
# ============================================================

print("=" * 80)
print("KAGGLE AUTHENTICATION + DATASET ACCESS")
print("=" * 80)


try:

    KAGGLE_API_TOKEN = (
        userdata.get(
            "KAGGLE_API_TOKEN"
        )
    )

except Exception as exc:

    raise RuntimeError(
        "Unable to read KAGGLE_API_TOKEN "
        "from Google Colab Secrets."
    ) from exc


if not KAGGLE_API_TOKEN:

    raise RuntimeError(
        "KAGGLE_API_TOKEN was not found "
        "in Google Colab Secrets."
    )


# ============================================================
# CONFIGURE KAGGLE AUTHENTICATION
# ============================================================

os.environ[
    "KAGGLE_API_TOKEN"
] = KAGGLE_API_TOKEN


print(
    "Colab Secret         : ✓ LOADED"
)


# ============================================================
# VERIFY KAGGLE CLI
# ============================================================

version_check = subprocess.run(
    [
        "kaggle",
        "--version"
    ],
    capture_output=True,
    text=True
)


if version_check.returncode != 0:

    raise RuntimeError(
        "Kaggle CLI is unavailable. "
        "Check the Cell 0 installation."
    )


print(
    f"Kaggle CLI           : "
    f"{version_check.stdout.strip()}"
)


# ============================================================
# VERIFY PERSISTED DATASET ACCESS
# ============================================================

dataset_check = subprocess.run(
    [
        "kaggle",
        "datasets",
        "files",
        KAGGLE_DATASET,
        "--page-size",
        "50"
    ],
    capture_output=True,
    text=True
)


if dataset_check.returncode != 0:

    raise RuntimeError(
        "Kaggle authentication succeeded, "
        "but the persisted Version B Knowledge Base "
        "could not be accessed.\n\n"
        f"{dataset_check.stderr}"
    )


print(
    "Dataset Access       : ✓ AVAILABLE"
)

print(
    f"Dataset              : "
    f"{KAGGLE_DATASET}"
)


# ============================================================
# VERIFY MANIFEST IS PRESENT
# ============================================================

dataset_listing = (
    dataset_check.stdout
)


manifest_available = (
    "bm25_shard_manifest.csv"
    in dataset_listing
)


if not manifest_available:

    raise RuntimeError(
        "bm25_shard_manifest.csv was not found "
        "in the persisted Kaggle dataset."
    )


print(
    "Shard Manifest       : ✓ AVAILABLE"
)


print("=" * 80)

print(
    "✓ Kaggle authentication validated."
)

print(
    "✓ Persistent Version B Knowledge Base "
    "is accessible."
)

KAGGLE AUTHENTICATION + DATASET ACCESS
Colab Secret         : ✓ LOADED
Kaggle CLI           : Kaggle CLI 2.2.4
Dataset Access       : ✓ AVAILABLE
Dataset              : cliffordimaguezegie/telecom-bm25-indexed-knowledge-base
Shard Manifest       : ✓ AVAILABLE
✓ Kaggle authentication validated.
✓ Persistent Version B Knowledge Base is accessible.


**Observation — Kaggle Access**

Kaggle authentication and access to the persisted Version B Knowledge Base were successfully validated.

***Key Decision:*** Verify remote artifact access before downloading the full Knowledge Base.

## **Cell 3 — Download Persistent BM25 Artifact**

In [4]:
# ============================================================
# CELL 3 — DOWNLOAD PERSISTENT BM25 ARTIFACT
# ============================================================


# ============================================================
# DOWNLOAD CONFIGURATION
# ============================================================

# Set True only when the local runtime copy should be
# replaced even if a complete-looking artifact is present.
FORCE_KAGGLE_DOWNLOAD = False

MIN_DOWNLOAD_DISK_GB = 80


# ============================================================
# CURRENT LOCAL STATE
# ============================================================

existing_shards = sorted(
    SHARD_DIR.glob(
        "*.duckdb"
    )
)

manifest_present = (
    SHARD_MANIFEST_PATH.exists()
)


print("=" * 80)
print("DOWNLOAD PERSISTENT BM25 ARTIFACT")
print("=" * 80)

print(
    f"Kaggle Dataset        : "
    f"{KAGGLE_DATASET}"
)

print(
    f"Destination           : "
    f"{SHARD_DIR}"
)

print(
    f"Expected Shards       : "
    f"{EXPECTED_SHARDS}"
)

print(
    f"Existing Shards       : "
    f"{len(existing_shards)}"
)

print(
    f"Manifest Present      : "
    f"{manifest_present}"
)

print(
    f"Force Download        : "
    f"{FORCE_KAGGLE_DOWNLOAD}"
)


# ============================================================
# LOCAL ARTIFACT DECISION
# ============================================================

artifact_already_present = (

    len(existing_shards)
    == EXPECTED_SHARDS

    and manifest_present
)


if (
    artifact_already_present
    and not FORCE_KAGGLE_DOWNLOAD
):

    print(
        "\n✓ Complete-looking BM25 artifact "
        "already exists locally."
    )

    print(
        "✓ Kaggle download skipped."
    )

    print(
        "✓ Deep artifact validation will "
        "be performed in Cell 4."
    )


else:

    # ========================================================
    # DISK SPACE CHECK
    # ========================================================

    disk_usage = shutil.disk_usage(
        WORK_DIR
    )

    free_disk_gb = (
        disk_usage.free
        / (1024 ** 3)
    )


    print(
        f"\nFree Disk             : "
        f"{free_disk_gb:.2f} GB"
    )


    if (
        free_disk_gb
        < MIN_DOWNLOAD_DISK_GB
    ):

        raise RuntimeError(
            "Insufficient free disk space for "
            "the persistent BM25 artifact. "
            f"At least {MIN_DOWNLOAD_DISK_GB} GB "
            "is required before download."
        )


    # ========================================================
    # REMOVE PARTIAL / STALE LOCAL ARTIFACT
    # ========================================================

    if SHARD_DIR.exists():

        print(
            "\nRemoving incomplete or stale "
            "local artifact..."
        )

        shutil.rmtree(
            SHARD_DIR
        )


    print(
        "\nDownloading persistent "
        "Version B Knowledge Base..."
    )

    print(
        "KaggleHub download progress "
        "will appear below."
    )

    print("-" * 80)


    # ========================================================
    # KAGGLEHUB DOWNLOAD
    # ========================================================

    download_start = (
        time.perf_counter()
    )


    downloaded_path = (
        kagglehub.dataset_download(
            KAGGLE_DATASET,
            output_dir=str(
                SHARD_DIR
            )
        )
    )


    download_elapsed = (
        time.perf_counter()
        - download_start
    )


    print("-" * 80)

    print(
        "✓ KaggleHub download completed."
    )

    print(
        f"Downloaded Path        : "
        f"{downloaded_path}"
    )

    print(
        f"Download Time          : "
        f"{download_elapsed / 60:.2f} min"
    )


# ============================================================
# LIGHT POST-DOWNLOAD CHECK
# ============================================================

downloaded_shards = sorted(
    SHARD_DIR.glob(
        "*.duckdb"
    )
)

manifest_present = (
    SHARD_MANIFEST_PATH.exists()
)


print(
    "\n" + "=" * 80
)

print(
    "POST-DOWNLOAD ARTIFACT CHECK"
)

print("=" * 80)


print(
    f"DuckDB Shards Found   : "
    f"{len(downloaded_shards)}"
)

print(
    f"Manifest Present      : "
    f"{manifest_present}"
)


if (
    len(downloaded_shards)
    != EXPECTED_SHARDS
):

    raise RuntimeError(
        "Persistent artifact is incomplete: "
        f"expected {EXPECTED_SHARDS} DuckDB shards, "
        f"found {len(downloaded_shards)}."
    )


if not manifest_present:

    raise RuntimeError(
        "The canonical BM25 shard manifest "
        "was not found after artifact restoration."
    )


# ============================================================
# ARTIFACT SIZE
# ============================================================

artifact_size_gb = sum(

    shard.stat().st_size

    for shard
    in downloaded_shards

) / (1024 ** 3)


# ============================================================
# FINAL DISK STATE
# ============================================================

disk_usage_after = shutil.disk_usage(
    WORK_DIR
)

free_disk_after_gb = (
    disk_usage_after.free
    / (1024 ** 3)
)


print(
    f"Artifact Size         : "
    f"{artifact_size_gb:.2f} GB"
)

print(
    f"Free Disk Remaining   : "
    f"{free_disk_after_gb:.2f} GB"
)

print("=" * 80)


print(
    f"✓ {EXPECTED_SHARDS} persistent "
    "DuckDB BM25 shards available."
)

print(
    "✓ Canonical shard manifest available."
)

print(
    "✓ Persistent Knowledge Base restored."
)

print(
    "✓ Ready for Cell 4 — "
    "Validate Restored BM25 Artifact."
)

DOWNLOAD PERSISTENT BM25 ARTIFACT
Kaggle Dataset        : cliffordimaguezegie/telecom-bm25-indexed-knowledge-base
Destination           : /tmp/telecom_bm25_runtime/bm25_shards
Expected Shards       : 21
Existing Shards       : 0
Manifest Present      : False
Force Download        : False

Free Disk             : 205.65 GB

Removing incomplete or stale local artifact...

KaggleHub download progress will appear below.
--------------------------------------------------------------------------------


100%|██████████| 27.5G/27.5G [11:37<00:00, 42.3MB/s]

Extracting files...


--------------------------------------------------------------------------------
✓ KaggleHub download completed.
Downloaded Path        : /tmp/telecom_bm25_runtime/bm25_shards
Download Time          : 18.38 min

POST-DOWNLOAD ARTIFACT CHECK
DuckDB Shards Found   : 21
Manifest Present      : True
Artifact Size         : 69.00 GB
Free Disk Remaining   : 136.65 GB
✓ 21 persistent DuckDB BM25 shards available.
✓ Canonical shard manifest available.
✓ Persistent Knowledge Base restored.
✓ Ready for Cell 4 — Validate Restored BM25 Artifact.


**Observation — Knowledge Base Restoration**

The persistent 21-shard BM25 Knowledge Base can be restored directly from Kaggle without rebuilding the source corpus or indexes.

***Key Decision:*** Reuse the persisted Knowledge Base as the normal Version B runtime path.

## **Cell 4 — Validate Restored BM25 Artifact**

In [5]:
# ============================================================
# CELL 4 — VALIDATE RESTORED BM25 ARTIFACT
# ============================================================


# ============================================================
# ARTIFACT PRESENCE
# ============================================================

print("=" * 90)
print("VALIDATE RESTORED BM25 ARTIFACT")
print("=" * 90)


if not SHARD_DIR.exists():

    raise FileNotFoundError(
        f"Shard directory does not exist: "
        f"{SHARD_DIR}"
    )


if not SHARD_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "BM25 shard manifest does not exist: "
        f"{SHARD_MANIFEST_PATH}"
    )


SHARD_FILES = sorted(
    SHARD_DIR.glob(
        "*.duckdb"
    )
)


print(
    f"Shard Directory       : "
    f"{SHARD_DIR}"
)

print(
    f"DuckDB Shards Found   : "
    f"{len(SHARD_FILES)}"
)

print(
    f"Manifest              : "
    f"{SHARD_MANIFEST_PATH.name}"
)


# ============================================================
# BASIC SHARD COUNT VALIDATION
# ============================================================

if (
    len(SHARD_FILES)
    != EXPECTED_SHARDS
):

    raise RuntimeError(
        "Shard-count validation failed: "
        f"expected {EXPECTED_SHARDS}, "
        f"found {len(SHARD_FILES)}."
    )


print(
    "Shard Count          : ✓ PASS"
)


# ============================================================
# LOAD PERSISTED SHARD MANIFEST
# ============================================================

SHARD_MANIFEST = pd.read_csv(
    SHARD_MANIFEST_PATH
)


required_manifest_columns = {
    "shard_name",
    "source_family",
    "collection",
    "records"
}


missing_columns = (
    required_manifest_columns
    - set(
        SHARD_MANIFEST.columns
    )
)


if missing_columns:

    raise RuntimeError(
        "Persisted shard manifest is missing "
        "required columns: "
        f"{sorted(missing_columns)}"
    )


manifest_shard_count = len(
    SHARD_MANIFEST
)

manifest_total_records = int(
    SHARD_MANIFEST[
        "records"
    ].sum()
)


print(
    f"Manifest Shards      : "
    f"{manifest_shard_count}"
)

print(
    f"Manifest Records     : "
    f"{manifest_total_records:,}"
)


# ============================================================
# MANIFEST VALIDATION
# ============================================================

if (
    manifest_shard_count
    != EXPECTED_SHARDS
):

    raise RuntimeError(
        "Manifest shard count does not match "
        "the frozen Version B architecture: "
        f"{manifest_shard_count} != "
        f"{EXPECTED_SHARDS}"
    )


if (
    manifest_total_records
    != EXPECTED_RECORDS
):

    raise RuntimeError(
        "Manifest record total does not match "
        "the frozen Version B corpus: "
        f"{manifest_total_records:,} != "
        f"{EXPECTED_RECORDS:,}"
    )


if (
    SHARD_MANIFEST[
        "shard_name"
    ].duplicated().any()
):

    raise RuntimeError(
        "Duplicate shard names were found "
        "in the persisted manifest."
    )


print(
    "Manifest Validation  : ✓ PASS"
)


# ============================================================
# VERIFY MANIFEST ↔ LOCAL SHARD MAPPING
# ============================================================

expected_shard_files = {
    f"{shard_name}.duckdb"

    for shard_name
    in SHARD_MANIFEST[
        "shard_name"
    ]
}


actual_shard_files = {
    shard_path.name

    for shard_path
    in SHARD_FILES
}


missing_shards = (
    expected_shard_files
    - actual_shard_files
)

unexpected_shards = (
    actual_shard_files
    - expected_shard_files
)


if missing_shards:

    raise RuntimeError(
        "Manifest shard files are missing "
        "from the restored artifact: "
        f"{sorted(missing_shards)}"
    )


if unexpected_shards:

    raise RuntimeError(
        "Unexpected DuckDB shard files were "
        "found in the restored artifact: "
        f"{sorted(unexpected_shards)}"
    )


print(
    "Shard File Mapping   : ✓ PASS"
)


# ============================================================
# LOAD DUCKDB FTS EXTENSION
# ============================================================

fts_test_conn = duckdb.connect()


try:

    fts_test_conn.execute(
        "INSTALL fts"
    )

    fts_test_conn.execute(
        "LOAD fts"
    )

finally:

    fts_test_conn.close()


print(
    "DuckDB FTS Extension : ✓ READY"
)


# ============================================================
# SHARD-BY-SHARD READABILITY + FTS VALIDATION
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "SHARD READABILITY + FTS VALIDATION"
)

print("=" * 90)


validation_rows = []

total_records = 0
total_fts_documents = 0


for shard_number, row in (
    SHARD_MANIFEST
    .reset_index(drop=True)
    .iterrows()
):

    shard_name = str(
        row["shard_name"]
    )

    expected_records = int(
        row["records"]
    )

    shard_path = (
        SHARD_DIR
        / f"{shard_name}.duckdb"
    )

    conn = None


    try:

        # ====================================================
        # OPEN SHARD READ-ONLY
        # ====================================================

        conn = duckdb.connect(
            str(shard_path),
            read_only=True
        )


        conn.execute(
            "LOAD fts"
        )


        # ====================================================
        # VALIDATE DOCUMENT TABLE
        # ====================================================

        table_ready = (
            conn.execute(
                f"""
                SELECT COUNT(*)

                FROM information_schema.tables

                WHERE table_schema = 'main'
                  AND table_name = '{FTS_TABLE}'
                """
            ).fetchone()[0]
            > 0
        )


        if not table_ready:

            raise RuntimeError(
                f"Document table "
                f"'{FTS_TABLE}' not found."
            )


        # ====================================================
        # RECORD COUNT
        # ====================================================

        record_count = int(
            conn.execute(
                f"""
                SELECT COUNT(*)
                FROM "{FTS_TABLE}"
                """
            ).fetchone()[0]
        )


        if (
            record_count
            != expected_records
        ):

            raise RuntimeError(
                "Shard record count does not "
                "match persisted manifest: "
                f"{record_count:,} != "
                f"{expected_records:,}"
            )


        # ====================================================
        # VALIDATE FTS SCHEMA
        # ====================================================

        fts_ready = (
            conn.execute(
                f"""
                SELECT COUNT(*)

                FROM information_schema.schemata

                WHERE schema_name =
                      '{FTS_SCHEMA}'
                """
            ).fetchone()[0]
            > 0
        )


        if not fts_ready:

            raise RuntimeError(
                "Persistent FTS schema "
                f"'{FTS_SCHEMA}' not found."
            )


        # ====================================================
        # FTS INTERNAL STATS
        # ====================================================

        stats_row = (
            conn.execute(
                f"""
                SELECT
                    num_docs,
                    avgdl

                FROM "{FTS_SCHEMA}"."stats"

                LIMIT 1
                """
            ).fetchone()
        )


        if stats_row is None:

            raise RuntimeError(
                "FTS stats table is empty."
            )


        fts_num_docs = int(
            stats_row[0]
        )

        fts_avgdl = float(
            stats_row[1]
        )


        # ====================================================
        # TABLE ↔ FTS CONSISTENCY
        # ====================================================

        if (
            fts_num_docs
            != record_count
        ):

            raise RuntimeError(
                "FTS document count does not "
                "match table count: "
                f"{fts_num_docs:,} != "
                f"{record_count:,}"
            )


        # ====================================================
        # ACCUMULATE RESULTS
        # ====================================================

        total_records += (
            record_count
        )

        total_fts_documents += (
            fts_num_docs
        )


        validation_rows.append({

            "shard_name":
                shard_name,

            "source_family":
                row["source_family"],

            "collection":
                row["collection"],

            "records":
                record_count,

            "fts_num_docs":
                fts_num_docs,

            "avgdl":
                round(
                    fts_avgdl,
                    2
                ),

            "status":
                "PASS"
        })


        print(
            f"[{shard_number + 1:02d}/"
            f"{EXPECTED_SHARDS}] "
            f"{shard_name:<38} "
            f"{record_count:>10,} records | "
            f"FTS ✓"
        )


    except Exception as exc:

        raise RuntimeError(
            f"Validation failed for "
            f"{shard_name}: {exc}"
        ) from exc


    finally:

        if conn is not None:

            conn.close()


# ============================================================
# GLOBAL ARTIFACT VALIDATION
# ============================================================

artifact_validation_df = pd.DataFrame(
    validation_rows
)

# Runtime alias retained for later inspection if required.
ARTIFACT_VALIDATION = (
    artifact_validation_df
)


print(
    "\n" + "=" * 90
)

print(
    "GLOBAL ARTIFACT VALIDATION"
)

print("=" * 90)


print(
    f"Validated Shards      : "
    f"{len(artifact_validation_df)}/"
    f"{EXPECTED_SHARDS}"
)

print(
    f"Total Records         : "
    f"{total_records:,}"
)

print(
    f"FTS Documents         : "
    f"{total_fts_documents:,}"
)

print(
    f"Expected Records      : "
    f"{EXPECTED_RECORDS:,}"
)


# ============================================================
# STRICT GLOBAL CHECKS
# ============================================================

if (
    total_records
    != EXPECTED_RECORDS
):

    raise RuntimeError(
        "Restored artifact record total does "
        "not match the frozen Version B corpus: "
        f"{total_records:,} != "
        f"{EXPECTED_RECORDS:,}"
    )


if (
    total_fts_documents
    != EXPECTED_RECORDS
):

    raise RuntimeError(
        "Global FTS document count does "
        "not match the frozen Version B corpus: "
        f"{total_fts_documents:,} != "
        f"{EXPECTED_RECORDS:,}"
    )


all_passed = (

    len(artifact_validation_df)
    == EXPECTED_SHARDS

    and (
        artifact_validation_df[
            "status"
        ]
        == "PASS"
    ).all()
)


if not all_passed:

    raise RuntimeError(
        "One or more restored BM25 shards "
        "failed validation."
    )


# ============================================================
# VALIDATED ARTIFACT SIZE
# ============================================================

artifact_size_gb = sum(

    shard_path.stat().st_size

    for shard_path
    in SHARD_FILES

) / (1024 ** 3)


print(
    f"Artifact Size         : "
    f"{artifact_size_gb:.2f} GB"
)

print(
    f"FTS-Ready Shards      : "
    f"{(
        artifact_validation_df['status']
        == 'PASS'
    ).sum()}/"
    f"{EXPECTED_SHARDS}"
)


print("=" * 90)


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "✓ RESTORED BM25 ARTIFACT VALIDATION PASSED"
)

print(
    f"✓ {EXPECTED_SHARDS}/{EXPECTED_SHARDS} "
    "DuckDB shards readable."
)

print(
    f"✓ {total_records:,} records confirmed."
)

print(
    "✓ FTS/BM25 metadata retained "
    "in every shard."
)

print(
    "✓ Manifest and local shard "
    "mapping confirmed."
)

print(
    "✓ Knowledge Base ready for "
    "optimized retrieval reconstruction."
)

print("=" * 90)

VALIDATE RESTORED BM25 ARTIFACT
Shard Directory       : /tmp/telecom_bm25_runtime/bm25_shards
DuckDB Shards Found   : 21
Manifest              : bm25_shard_manifest.csv
Shard Count          : ✓ PASS
Manifest Shards      : 21
Manifest Records     : 1,780,938
Manifest Validation  : ✓ PASS
Shard File Mapping   : ✓ PASS
DuckDB FTS Extension : ✓ READY

SHARD READABILITY + FTS VALIDATION
[01/21] tcc_3gpp_tsg_00                           143,679 records | FTS ✓
[02/21] tcc_3gpp_tsg_01                           142,683 records | FTS ✓
[03/21] tcc_3gpp_tsg_02                           143,196 records | FTS ✓
[04/21] tcc_3gpp_tsg_03                           143,114 records | FTS ✓
[05/21] tcc_uspto_00                               23,986 records | FTS ✓
[06/21] tcc_uspto_01                               23,888 records | FTS ✓
[07/21] tcc_uspto_02                               23,714 records | FTS ✓
[08/21] tcc_uspto_03                               23,716 records | FTS ✓
[09/21] tcc_ieee_access

**Observation — Artifact Validation**

All 21 DuckDB shards, 1,780,938 records and persistent BM25/FTS indexes are validated before retrieval is enabled.

***Key Decision:*** Treat artifact integrity as a strict runtime gate before reconstructing the search layer.

# **SECTION 2 — Reconstruct Optimized Retrieval**

## **Cell 5 — Shard Registry + Deterministic Dynamic Router**


In [6]:
# ============================================================
# CELL 5 — VERSION B SHARD REGISTRY + DYNAMIC ROUTER
# ============================================================

# Purpose:
#
# Reconstruct the persistent BM25 shard registry and expose the
# SAME deterministic source-routing semantics used by Version A:
#
#     3gpp
#     tcc
#     hybrid
#
# Important experimental control:
#
#     Version A and Version B use the same routing logic.
#     The architectural difference remains retrieval:
#
#         Version A → remote raw-corpus retrieval
#         Version B → persistent DuckDB BM25/FTS retrieval
#
# Version B keeps search profiles only as INTERNAL retrieval
# primitives. The LLM will not select a profile.


# ============================================================
# RUNTIME MANIFEST
# ============================================================

RUNTIME_MANIFEST = (
    SHARD_MANIFEST
    .copy()
    .reset_index(drop=True)
)


for column in [
    "source_family",
    "collection",
    "shard_name"
]:
    RUNTIME_MANIFEST[
        column
    ] = (
        RUNTIME_MANIFEST[
            column
        ]
        .fillna("")
        .astype(str)
    )


# ============================================================
# CANONICAL SHARD PATHS
# ============================================================

RUNTIME_MANIFEST[
    "shard_path"
] = (
    RUNTIME_MANIFEST[
        "shard_name"
    ].apply(
        lambda shard_name:
            SHARD_DIR
            / f"{shard_name}.duckdb"
    )
)


ALL_SHARD_NAMES = sorted(
    RUNTIME_MANIFEST[
        "shard_name"
    ].tolist()
)


# ============================================================
# SHARD HELPERS
# ============================================================

def get_shard_names(
    mask
):
    """
    Return a deterministic list of shard names for a
    manifest selection condition.
    """

    return sorted(
        RUNTIME_MANIFEST.loc[
            mask,
            "shard_name"
        ]
        .astype(str)
        .tolist()
    )


def dedupe_preserve_order(
    values
):
    """
    Remove duplicates while preserving deterministic order.
    """

    return list(
        dict.fromkeys(
            values
        )
    )


# ============================================================
# COLLECTION → SHARD REGISTRY
# ============================================================

COLLECTION_SHARDS = {

    str(collection):
        sorted(
            group[
                "shard_name"
            ]
            .astype(str)
            .tolist()
        )

    for collection, group
    in RUNTIME_MANIFEST.groupby(
        "collection",
        dropna=False
    )
}


# ============================================================
# DEDICATED 3GPP SHARDS
# ============================================================
#
# IMPORTANT:
# Version A uses dedicated GSMA/3GPP specifications for its
# 3GPP route. For fair A/B comparison, Version B must therefore
# use ONLY source_family == "3GPP" for the 3GPP branch.
#
# TCC 3GPP-TSG material is NOT included in this branch.
# ============================================================

DEDICATED_3GPP_SHARDS = (
    get_shard_names(
        RUNTIME_MANIFEST[
            "source_family"
        ] == "3GPP"
    )
)


if len(
    DEDICATED_3GPP_SHARDS
) != 2:

    raise RuntimeError(
        "Expected exactly 2 dedicated 3GPP BM25 shards, "
        f"found {len(DEDICATED_3GPP_SHARDS)}."
    )


# ============================================================
# INTERNAL SEARCH PRIMITIVES
# ============================================================
#
# These are NOT exposed to the LLM.
#
# IETF intentionally uses only IETF-RFCs + IETF-Drafts to
# match Version A's TCC collection selection.
# ============================================================

SEARCH_PROFILES = {

    "all":
        sorted(
            ALL_SHARD_NAMES
        ),

    "3gpp":
        list(
            DEDICATED_3GPP_SHARDS
        ),

    "ietf":
        sorted(
            COLLECTION_SHARDS.get(
                "IETF-RFCs",
                []
            )
            +
            COLLECTION_SHARDS.get(
                "IETF-Drafts",
                []
            )
        ),

    "research":
        sorted(
            COLLECTION_SHARDS.get(
                "IEEE-Access",
                []
            )
            +
            COLLECTION_SHARDS.get(
                "OpenAlex",
                []
            )
        ),

    "patents":
        sorted(
            COLLECTION_SHARDS.get(
                "USPTO",
                []
            )
            +
            COLLECTION_SHARDS.get(
                "EPO",
                []
            )
        ),

    "knowledge":
        sorted(
            COLLECTION_SHARDS.get(
                "Wikipedia-Telecom",
                []
            )
            +
            COLLECTION_SHARDS.get(
                "Wikidata-Telecom",
                []
            )
        )
}


EXPECTED_PROFILE_COUNTS = {
    "all": 21,
    "3gpp": 2,
    "ietf": 2,
    "research": 3,
    "patents": 5,
    "knowledge": 2
}


for profile_name, expected_count in (
    EXPECTED_PROFILE_COUNTS.items()
):

    actual_count = len(
        SEARCH_PROFILES[
            profile_name
        ]
    )

    if actual_count != expected_count:

        raise RuntimeError(
            f"Internal profile '{profile_name}' expected "
            f"{expected_count} shards, found {actual_count}."
        )


# ============================================================
# QUERY NORMALIZATION
# ============================================================

def normalize_query(
    query
):
    """
    Normalize query text for deterministic source routing.
    """

    if (
        not isinstance(
            query,
            str
        )
        or
        not query.strip()
    ):

        raise ValueError(
            "Query must not be empty."
        )

    return re.sub(
        r"\s+",
        " ",
        query.strip().lower()
    )


def contains_signal(
    normalized_query,
    signal
):
    """
    Match complete telecom terms or phrases rather than
    arbitrary substrings.
    """

    pattern = (
        r"(?<![a-z0-9])"
        +
        re.escape(
            signal.lower()
        )
        +
        r"(?![a-z0-9])"
    )

    return (
        re.search(
            pattern,
            normalized_query
        )
        is not None
    )


# ============================================================
# DYNAMIC SOURCE SIGNALS
# EXACT VERSION A ROUTING SEMANTICS
# ============================================================

GPP_SOURCE_SIGNALS = {

    "3gpp",
    "3gpp ts",
    "3gpp tr",

    "5g standalone",
    "5g sa",
    "5gs",
    "5g core",
    "ng-ran",

    "amf",
    "smf",
    "upf",
    "ausf",
    "udm",
    "nssf",
    "pcf",

    "pdu session",
    "registration",
    "mobility management",

    "s-nssai",
    "network slicing",
    "network slice",

    "5qi",
    "qos flow",

    "ngap",
    "xnap",
    "xn interface",

    "rrc",
    "radio link failure",
    "rlf",

    "n1 interface",
    "n2 interface",
    "n3 interface",
    "n4 interface",

    "pfcp"
}


TCC_IETF_SIGNALS = {
    "ietf",
    "rfc",
    "quic",
    "http",
    "http3",
    "http/3",
    "tls",
    "tcp",
    "udp",
    "dns"
}


TCC_RESEARCH_SIGNALS = {
    "research",
    "paper",
    "study",
    "ieee",
    "openalex"
}


TCC_PATENT_SIGNALS = {
    "patent",
    "invention",
    "uspto",
    "epo"
}


TCC_KNOWLEDGE_SIGNALS = {
    "wikipedia",
    "wikidata",
    "general telecom",
    "general telecommunications"
}


# ============================================================
# 3GPP DOMAIN → SPECIFICATION INFERENCE
# ============================================================

GPP_SPEC_RULES = [

    {
        "name":
            "5GS architecture",

        "signals": {
            "amf",
            "smf",
            "upf",
            "nssf",
            "s-nssai",
            "network slice",
            "network slicing",
            "5qi",
            "qos flow",
            "pdu session",
            "5g core",
            "5g standalone",
            "5g sa"
        },

        "specs": [
            "23.501"
        ]
    },

    {
        "name":
            "5GS procedures",

        "signals": {
            "registration",
            "mobility management",
            "pdu session",
            "handover",
            "inter-gnb handover",
            "service request",
            "session release",
            "pdu session release"
        },

        "specs": [
            "23.502"
        ]
    },

    {
        "name":
            "5GS policy and QoS",

        "signals": {
            "policy control",
            "pcf",
            "qos policy",
            "5qi"
        },

        "specs": [
            "23.503"
        ]
    },

    {
        "name":
            "5G security",

        "signals": {
            "authentication",
            "ausf",
            "udm",
            "security",
            "5g aka",
            "aka"
        },

        "specs": [
            "33.501"
        ]
    },

    {
        "name":
            "5GS NAS",

        "signals": {
            "nas",
            "5gmm",
            "5gsm"
        },

        "specs": [
            "24.501"
        ]
    },

    {
        "name":
            "PFCP and N4",

        "signals": {
            "pfcp",
            "n4",
            "n4 interface"
        },

        "specs": [
            "29.244"
        ]
    },

    {
        "name":
            "NR RRC",

        "signals": {
            "rrc",
            "radio link failure",
            "rlf",
            "rrc re-establishment"
        },

        "specs": [
            "38.331"
        ]
    },

    {
        "name":
            "NR architecture",

        "signals": {
            "nr architecture",
            "gnb",
            "ng-ran",
            "handover",
            "inter-gnb handover"
        },

        "specs": [
            "38.300"
        ]
    },

    {
        "name":
            "NGAP",

        "signals": {
            "ngap",
            "n2",
            "n2 interface"
        },

        "specs": [
            "38.413"
        ]
    },

    {
        "name":
            "XnAP",

        "signals": {
            "xnap",
            "xn",
            "xn interface",
            "inter-gnb handover"
        },

        "specs": [
            "38.423"
        ]
    }
]


MAX_GPP_SPEC_CANDIDATES = 3


def extract_explicit_3gpp_specs(
    query
):
    """
    Extract explicit 3GPP specification numbers.
    """

    normalized = normalize_query(
        query
    )

    matches = re.findall(
        r"\b"
        r"(?:3gpp\s*)?"
        r"(?:ts\s*|tr\s*)?"
        r"(\d{2}\.\d{3})"
        r"\b",
        normalized
    )

    return dedupe_preserve_order(
        matches
    )


def infer_3gpp_spec_candidates(
    query,
    max_specs=MAX_GPP_SPEC_CANDIDATES
):
    """
    Infer likely 3GPP specifications from telecom concepts.
    This is routing metadata only in Version B; BM25 still
    searches the selected persistent shard set.
    """

    normalized = normalize_query(
        query
    )

    candidates = []

    reasons = []


    for spec in extract_explicit_3gpp_specs(
        query
    ):

        if spec not in candidates:

            candidates.append(
                spec
            )

            reasons.append(
                f"explicit:{spec}"
            )


    for rule in GPP_SPEC_RULES:

        matched_signals = [
            signal
            for signal in rule[
                "signals"
            ]
            if contains_signal(
                normalized,
                signal
            )
        ]

        if not matched_signals:
            continue


        for spec in rule[
            "specs"
        ]:

            if spec not in candidates:

                candidates.append(
                    spec
                )


        reasons.append(
            (
                f"{rule['name']}: "
                +
                ", ".join(
                    matched_signals
                )
            )
        )


        if len(candidates) >= max_specs:
            break


    return {
        "specs":
            candidates[
                :max_specs
            ],

        "reasons":
            reasons
    }


# ============================================================
# TCC COLLECTION ROUTING
# EXACT VERSION A COLLECTION SEMANTICS
# ============================================================

def select_tcc_collections(
    query
):
    """
    Identify TCC collections relevant to the query.
    """

    normalized = normalize_query(
        query
    )

    collections = []

    reasons = []


    ietf_hits = [
        signal
        for signal in TCC_IETF_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]


    if ietf_hits:

        collections.extend([
            "IETF-RFCs",
            "IETF-Drafts"
        ])

        reasons.append(
            "IETF: "
            +
            ", ".join(
                ietf_hits
            )
        )


    research_hits = [
        signal
        for signal in TCC_RESEARCH_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]


    if research_hits:

        collections.extend([
            "IEEE-Access",
            "OpenAlex"
        ])

        reasons.append(
            "Research: "
            +
            ", ".join(
                research_hits
            )
        )


    patent_hits = [
        signal
        for signal in TCC_PATENT_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]


    if patent_hits:

        collections.extend([
            "USPTO",
            "EPO"
        ])

        reasons.append(
            "Patents: "
            +
            ", ".join(
                patent_hits
            )
        )


    knowledge_hits = [
        signal
        for signal in TCC_KNOWLEDGE_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]


    if knowledge_hits:

        collections.extend([
            "Wikipedia-Telecom",
            "Wikidata-Telecom"
        ])

        reasons.append(
            "General knowledge: "
            +
            ", ".join(
                knowledge_hits
            )
        )


    gpp_hits = [
        signal
        for signal in GPP_SOURCE_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]


    if gpp_hits:

        collections.append(
            "3GPP-TSG"
        )

        reasons.append(
            "TCC 3GPP contribution material"
        )


    if not collections:

        collections = [
            "Wikipedia-Telecom"
        ]

        reasons.append(
            "generic TCC fallback"
        )


    return {
        "collections":
            dedupe_preserve_order(
                collections
            ),

        "reasons":
            reasons
    }


# ============================================================
# DYNAMIC SOURCE ROUTER
# EXACT VERSION A DECISION SEMANTICS
# ============================================================

def route_telecom_query(
    query
):
    """
    Dynamically route to:

        3gpp
        tcc
        hybrid

    HYBRID:
        dedicated 3GPP BM25 shards
        +
        selected non-3GPP TCC collections.

    TCC 3GPP-TSG is removed from HYBRID to avoid duplicate
    standards-domain evidence.
    """

    normalized = normalize_query(
        query
    )


    gpp_hits = [
        signal
        for signal in GPP_SOURCE_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]


    tcc_signals = (
        TCC_IETF_SIGNALS
        |
        TCC_RESEARCH_SIGNALS
        |
        TCC_PATENT_SIGNALS
        |
        TCC_KNOWLEDGE_SIGNALS
    )


    tcc_hits = [
        signal
        for signal in tcc_signals
        if contains_signal(
            normalized,
            signal
        )
    ]


    explicit_specs = (
        extract_explicit_3gpp_specs(
            query
        )
    )


    if explicit_specs:

        route = "3gpp"

    elif gpp_hits and tcc_hits:

        route = "hybrid"

    elif gpp_hits:

        route = "3gpp"

    elif tcc_hits:

        route = "tcc"

    else:

        route = "tcc"


    gpp_selection = (
        infer_3gpp_spec_candidates(
            query
        )
    )


    tcc_selection = (
        select_tcc_collections(
            query
        )
    )


    if route == "hybrid":

        tcc_selection[
            "collections"
        ] = [
            collection
            for collection
            in tcc_selection[
                "collections"
            ]
            if collection
            != "3GPP-TSG"
        ]


        tcc_selection[
            "reasons"
        ] = [
            reason
            for reason
            in tcc_selection[
                "reasons"
            ]
            if reason
            != "TCC 3GPP contribution material"
        ]


        if not tcc_selection[
            "collections"
        ]:

            tcc_selection[
                "collections"
            ] = [
                "Wikipedia-Telecom"
            ]

            tcc_selection[
                "reasons"
            ].append(
                "hybrid TCC fallback"
            )


    return {

        "route":
            route,

        "gpp_signal_hits":
            gpp_hits,

        "tcc_signal_hits":
            tcc_hits,

        "gpp_specs":
            gpp_selection[
                "specs"
            ],

        "gpp_reasons":
            gpp_selection[
                "reasons"
            ],

        "tcc_collections":
            tcc_selection[
                "collections"
            ],

        "tcc_reasons":
            tcc_selection[
                "reasons"
            ]
    }


# ============================================================
# ROUTE → PERSISTENT BM25 SHARD PLAN
# ============================================================

def collections_to_shards(
    collections
):
    """
    Resolve selected corpus collections to persistent shard names.
    """

    shard_names = []


    for collection in collections:

        collection_shards = (
            COLLECTION_SHARDS.get(
                collection,
                []
            )
        )


        if not collection_shards:

            raise RuntimeError(
                f"No persistent BM25 shard found for "
                f"collection '{collection}'."
            )


        shard_names.extend(
            collection_shards
        )


    return sorted(
        dedupe_preserve_order(
            shard_names
        )
    )


def build_route_shard_plan(
    query
):
    """
    Build the persistent BM25 shard set for the dynamic route.

    For HYBRID, all selected shards are searched under ONE
    common BM25 statistics space so scores are directly ranked
    together rather than naively comparing separate score scales.
    """

    routing = route_telecom_query(
        query
    )


    route = routing[
        "route"
    ]


    if route == "3gpp":

        selected_shards = list(
            DEDICATED_3GPP_SHARDS
        )

        sources_searched = [
            "3GPP"
        ]


    elif route == "tcc":

        selected_shards = (
            collections_to_shards(
                routing[
                    "tcc_collections"
                ]
            )
        )

        sources_searched = [
            "TCC"
        ]


    elif route == "hybrid":

        tcc_shards = (
            collections_to_shards(
                routing[
                    "tcc_collections"
                ]
            )
        )

        selected_shards = sorted(
            dedupe_preserve_order(
                list(
                    DEDICATED_3GPP_SHARDS
                )
                +
                tcc_shards
            )
        )

        sources_searched = [
            "3GPP",
            "TCC"
        ]


    else:

        raise RuntimeError(
            f"Unsupported dynamic route: {route}"
        )


    if not selected_shards:

        raise RuntimeError(
            f"Dynamic route '{route}' selected no BM25 shards."
        )


    return {

        **routing,

        "sources_searched":
            sources_searched,

        "selected_shards":
            selected_shards,

        "selected_shard_count":
            len(
                selected_shards
            )
    }


# ============================================================
# STATIC ROUTER VALIDATION
# ============================================================

ROUTER_TESTS = [

    {
        "name":
            "3GPP-only",

        "query":
            (
                "Explain the role of the AMF in registration "
                "and mobility management procedures in a "
                "5G Standalone network."
            ),

        "expected_route":
            "3gpp"
    },

    {
        "name":
            "TCC-only",

        "query":
            (
                "Explain the key mechanisms of QUIC as defined "
                "by the IETF, including connection establishment, "
                "stream multiplexing and connection migration."
            ),

        "expected_route":
            "tcc"
    },

    {
        "name":
            "Hybrid",

        "query":
            (
                "Explain how HTTP and TLS support communication "
                "in the 5G Service-Based Architecture and distinguish "
                "3GPP AMF/SMF requirements from IETF mechanisms."
            ),

        "expected_route":
            "hybrid"
    }
]


print("=" * 90)

print(
    "VERSION B — DYNAMIC ROUTER + SHARD PLAN VALIDATION"
)

print("=" * 90)


for test in ROUTER_TESTS:

    plan = build_route_shard_plan(
        test[
            "query"
        ]
    )


    route_ok = (
        plan[
            "route"
        ]
        ==
        test[
            "expected_route"
        ]
    )


    print(
        f"\n{test['name']}"
    )

    print(
        f"  Expected route : "
        f"{test['expected_route'].upper()}"
    )

    print(
        f"  Actual route   : "
        f"{plan['route'].upper()}"
    )

    print(
        f"  3GPP specs     : "
        f"{plan['gpp_specs']}"
    )

    print(
        f"  TCC collections: "
        f"{plan['tcc_collections']}"
    )

    print(
        f"  BM25 shards    : "
        f"{plan['selected_shard_count']}"
    )

    print(
        f"  Sources        : "
        f"{plan['sources_searched']}"
    )

    print(
        f"  Router check   : "
        f"{'PASS' if route_ok else 'FAIL'}"
    )


    if not route_ok:

        raise RuntimeError(
            f"Static router validation failed "
            f"for {test['name']}."
        )


print(
    "\n" + "=" * 90
)

print(
    "✓ Version A routing semantics replicated in Version B."
)

print(
    "✓ 3GPP route uses dedicated 3GPP shards only."
)

print(
    "✓ TCC route selects corpus collections internally."
)

print(
    "✓ Hybrid route combines dedicated 3GPP + non-3GPP TCC shards."
)

print(
    "✓ Hybrid shard set will use one common BM25 statistics space."
)

print(
    "✓ No profile selection will be exposed to the LLM."
)

print("=" * 90)


VERSION B — DYNAMIC ROUTER + SHARD PLAN VALIDATION

3GPP-only
  Expected route : 3GPP
  Actual route   : 3GPP
  3GPP specs     : ['23.501', '23.502']
  TCC collections: ['3GPP-TSG']
  BM25 shards    : 2
  Sources        : ['3GPP']
  Router check   : PASS

TCC-only
  Expected route : TCC
  Actual route   : TCC
  3GPP specs     : []
  TCC collections: ['IETF-RFCs', 'IETF-Drafts']
  BM25 shards    : 2
  Sources        : ['TCC']
  Router check   : PASS

Hybrid
  Expected route : HYBRID
  Actual route   : HYBRID
  3GPP specs     : ['23.501']
  TCC collections: ['IETF-RFCs', 'IETF-Drafts']
  BM25 shards    : 4
  Sources        : ['3GPP', 'TCC']
  Router check   : PASS

✓ Version A routing semantics replicated in Version B.
✓ 3GPP route uses dedicated 3GPP shards only.
✓ TCC route selects corpus collections internally.
✓ Hybrid route combines dedicated 3GPP + non-3GPP TCC shards.
✓ Hybrid shard set will use one common BM25 statistics space.
✓ No profile selection will be exposed to the LLM.


**Observation — Shard Registry + Dynamic Router**

The restored manifest is reconstructed into the deterministic **3GPP / TCC / Hybrid** routing layer aligned with Version A. Internal shard groups are implementation primitives only; the LLM never selects a retrieval profile, source family, collection or shard.

***Key Decision:*** Keep route semantics constant across Version A and Version B so the principal architectural variable remains remote versus persistent BM25 retrieval.


## **Cell 6 — Dynamic Routed BM25 Retrieval Engine**


In [14]:
# ============================================================
# CELL 6 — DYNAMIC ROUTED BM25 RETRIEVAL ENGINE
# ============================================================
#
# Version B retrieval flow:
#
#     query
#       ↓
#     Cell 5 deterministic router
#       ↓
#     selected persistent BM25 shards
#       ↓
#     ONE common BM25 statistics space
#       ↓
#     parallel shard search
#       ↓
#     global ranking
#
# The LLM does not select a profile.
#
# Architectural control:
#
#     Version A → remote raw-corpus retrieval
#     Version B → persistent DuckDB BM25/FTS retrieval
#
# Routing semantics remain aligned across A and B.
# ============================================================


# ============================================================
# QUERY PROCESSING
# ============================================================

QUERY_STOPWORDS = {
    "a", "an", "the", "and", "or",
    "of", "to", "for", "in", "on",
    "with", "by", "from", "as", "at",
    "what", "which", "how", "why",
    "when", "where", "is", "are",
    "was", "were", "be", "been",
    "do", "does", "did",
    "describe", "explain"
}


# Version B originally retained only the Top-N high-IDF query
# terms defined by MAX_CANDIDATE_TERMS. That was efficient, but
# too aggressive for multi-concept and HYBRID questions.
#
# Keep the original configuration as a floor while allowing a
# broader set of discriminative terms for routed retrieval.
ROUTE_MAX_CANDIDATE_TERMS = max(
    int(MAX_CANDIDATE_TERMS),
    8
)


# Hybrid retrieval is intentionally source-aware.
#
# A HYBRID route means that BOTH standards evidence (3GPP)
# and complementary non-3GPP evidence (TCC) are required.
#
# Since the TCC corpus is much larger than the dedicated 3GPP
# corpus, a pure global Top-K can crowd all 3GPP candidates out
# even when valid 3GPP matches exist.
#
# Reserve a minimum number of final evidence slots per source
# family, then fill the remaining slot(s) by global BM25 score.
HYBRID_MIN_RESULTS_PER_SOURCE = 2


# Evidence returned to the LLM should be query-focused rather
# than simply the first N characters of a long specification.
RESULT_TEXT_CHARS = 5000


def telecom_tokenize(
    text
):
    """
    Preserve telecom identifiers such as:

        5G
        N2
        N4
        S-NSSAI
        NG-RAN
        23.501
        GTP-U
    """

    return re.findall(
        r"[A-Za-z0-9]+(?:[.-][A-Za-z0-9]+)*",
        (
            text
            or
            ""
        ).lower()
    )


def process_query(
    query
):
    """
    Tokenize a telecom query and remove lightweight
    natural-language stopwords.
    """

    tokens = telecom_tokenize(
        query
    )

    return [
        token
        for token
        in tokens
        if token
        not in QUERY_STOPWORDS
    ]


# ============================================================
# HYBRID BRANCH-FOCUSED TERMS
# ============================================================

TCC_COLLECTION_SELECTOR_TERMS = {
    "ietf", "rfc", "research", "paper", "study",
    "ieee", "openalex", "patent", "uspto", "epo",
    "wikipedia", "wikidata"
}


def build_hybrid_focus_terms(query, route_plan, candidate_terms, global_df_map):
    """Build source-specific internal evidence intents for HYBRID."""

    normalized_query = normalize_query(query)

    gpp_signal_text = " ".join(
        route_plan.get("gpp_signal_hits", []) or []
    )
    gpp_focus_terms = [
        term for term in telecom_tokenize(gpp_signal_text)
        if global_df_map.get(term, 0) > 0
    ]

    for term in ["service-based", "5g"]:
        if term in process_query(query) and global_df_map.get(term, 0) > 0:
            gpp_focus_terms.append(term)

    if "service based" in normalized_query:
        for term in ["service", "based"]:
            if global_df_map.get(term, 0) > 0:
                gpp_focus_terms.append(term)

    tcc_signal_text = " ".join(
        route_plan.get("tcc_signal_hits", []) or []
    )
    tcc_focus_terms = [
        term for term in telecom_tokenize(tcc_signal_text)
        if (
            term not in TCC_COLLECTION_SELECTOR_TERMS
            and global_df_map.get(term, 0) > 0
        )
    ]

    # Reinforce protocol-mechanism terms present in the original query.
    processed_query = process_query(query)
    for term in ["http", "tls", "quic", "tcp", "udp", "dns"]:
        if term in processed_query and global_df_map.get(term, 0) > 0:
            tcc_focus_terms.append(term)

    if not gpp_focus_terms:
        gpp_focus_terms = list(candidate_terms)
    if not tcc_focus_terms:
        tcc_focus_terms = list(candidate_terms)

    return {
        "3GPP": list(dict.fromkeys(gpp_focus_terms)),
        "TCC": list(dict.fromkeys(tcc_focus_terms)),
    }


# ============================================================
# SELECTED-SHARD BM25 STATISTICS
# ============================================================

def get_selected_shard_stats(
    shard_names,
    query_terms
):
    """
    Calculate BM25 corpus statistics across ALL shards selected
    by the deterministic route.

    This is especially important for HYBRID retrieval.

    Dedicated 3GPP and selected TCC shards are evaluated under
    one common:

        - total document count
        - average document length
        - document-frequency map

    Therefore the resulting BM25 scores are directly comparable
    during global ranking.

    Returns:
        total_docs
        weighted average document length
        global document frequency per query term
    """

    if not shard_names:

        raise ValueError(
            "At least one BM25 shard must be selected."
        )


    total_docs = 0

    weighted_dl = 0.0

    df_map = {
        term: 0
        for term
        in query_terms
    }


    for shard_name in shard_names:

        shard_path = (
            SHARD_DIR
            / f"{shard_name}.duckdb"
        )


        if not shard_path.exists():

            raise FileNotFoundError(
                f"Selected BM25 shard is missing: "
                f"{shard_path}"
            )


        conn = None


        try:

            conn = duckdb.connect(
                str(
                    shard_path
                ),
                read_only=True
            )

            conn.execute(
                "SET threads = 1"
            )

            conn.execute(
                "LOAD fts"
            )


            # ================================================
            # SHARD CORPUS STATISTICS
            # ================================================

            stats_row = (
                conn.execute(
                    f"""
                    SELECT
                        num_docs,
                        avgdl

                    FROM "{FTS_SCHEMA}"."stats"
                    """
                ).fetchone()
            )


            if stats_row is None:

                raise RuntimeError(
                    f"No FTS statistics found "
                    f"for shard {shard_name}."
                )


            num_docs = int(
                stats_row[
                    0
                ]
            )

            shard_avgdl = float(
                stats_row[
                    1
                ]
            )


            total_docs += (
                num_docs
            )

            weighted_dl += (
                num_docs
                *
                shard_avgdl
            )


            # ================================================
            # GLOBAL DOCUMENT FREQUENCY
            # ================================================

            if query_terms:

                placeholders = ",".join(
                    ["?"]
                    *
                    len(
                        query_terms
                    )
                )


                rows = conn.execute(
                    f"""
                    SELECT
                        term,
                        df

                    FROM "{FTS_SCHEMA}"."dict"

                    WHERE term IN (
                        {placeholders}
                    )
                    """,
                    query_terms
                ).fetchall()


                for term, df in rows:

                    df_map[
                        term
                    ] += int(
                        df
                    )


        finally:

            if conn is not None:

                conn.close()


    if total_docs <= 0:

        raise RuntimeError(
            "Selected BM25 shard set contains "
            "no indexed documents."
        )


    selected_avgdl = (
        weighted_dl
        /
        total_docs
    )


    return (
        total_docs,
        selected_avgdl,
        df_map
    )


# ============================================================
# HIGH-IDF TERM SELECTION
# ============================================================

def select_candidate_terms(
    query_terms,
    df_map,
    num_docs,
    max_terms=MAX_CANDIDATE_TERMS
):
    """
    Rank query terms by global IDF across the route-selected
    shard set and retain only the most discriminative terms.
    """

    scored_terms = []


    for term in query_terms:

        df = int(
            df_map.get(
                term,
                0
            )
        )


        if df <= 0:

            continue


        idf = math.log(
            (
                (
                    num_docs
                    -
                    df
                    +
                    0.5
                )
                /
                (
                    df
                    +
                    0.5
                )
            )
            +
            1
        )


        scored_terms.append(
            (
                term,
                df,
                idf
            )
        )


    scored_terms.sort(
        key=lambda item:
            item[
                2
            ],
        reverse=True
    )


    selected_terms = [
        term
        for term, _, _
        in scored_terms[
            :max_terms
        ]
    ]


    return (
        selected_terms,
        scored_terms
    )


# ============================================================
# SEARCH ONE PERSISTENT SHARD
# ============================================================

def search_filtered_shard(
    shard_name,
    query_terms,
    num_docs,
    avgdl,
    df_map,
    top_k=TOP_K_RESULTS,
    allowed_identifiers=None
):
    """
    Search one DuckDB BM25 shard using the SAME global
    statistics calculated across the complete route-selected
    shard set.
    """

    shard_path = (
        SHARD_DIR
        / f"{shard_name}.duckdb"
    )


    active_terms = [
        term
        for term
        in query_terms
        if df_map.get(
            term,
            0
        ) > 0
    ]


    if not active_terms:

        return pd.DataFrame()


    # ========================================================
    # GLOBAL QUERY TERM TABLE
    # ========================================================

    values_sql = ", ".join(
        ["(?, ?)"]
        *
        len(
            active_terms
        )
    )


    parameters = []


    for term in active_terms:

        parameters.extend([
            term,
            int(
                df_map[
                    term
                ]
            )
        ])


    # ========================================================
    # OPTIONAL 3GPP SPECIFICATION CONSTRAINT
    # ========================================================
    #
    # Cell 5 already infers the most relevant 3GPP specification
    # identifiers. For dedicated 3GPP shards, constrain candidate
    # documents to those identifiers before BM25 Top-K selection.
    #
    # This prevents a query such as "AMF registration" from being
    # dominated by unrelated long 3GPP reports that happen to
    # contain the same high-IDF terms.
    # ========================================================

    allowed_identifiers = [
        str(identifier).strip()
        for identifier
        in (
            allowed_identifiers
            or
            []
        )
        if str(identifier).strip()
    ]


    if allowed_identifiers:

        identifier_placeholders = ",".join(
            ["?"]
            *
            len(
                allowed_identifiers
            )
        )

        permitted_docs_cte = f"""
        permitted_docs AS (

            SELECT DISTINCT
                d.docid

            FROM "{FTS_SCHEMA}"."docs" d

            JOIN "{FTS_TABLE}" c
              ON c.doc_id = d.name

            WHERE c.identifier IN (
                {identifier_placeholders}
            )
        ),
        """

        qterms_filter_join = """
            JOIN permitted_docs p
              ON t.docid = p.docid
        """

        parameters.extend(
            allowed_identifiers
        )

    else:

        permitted_docs_cte = ""

        qterms_filter_join = ""


    conn = None


    try:

        conn = duckdb.connect(
            str(
                shard_path
            ),
            read_only=True
        )

        conn.execute(
            "SET threads = 1"
        )

        conn.execute(
            "LOAD fts"
        )


        # ====================================================
        # DIRECT INVERTED-INDEX BM25 SEARCH
        # ====================================================

        sql = f"""
        WITH

        global_query_terms(
            term,
            global_df
        ) AS (

            VALUES {values_sql}
        ),

        qtermids AS (

            SELECT
                d.termid,
                d.term,
                q.global_df

            FROM "{FTS_SCHEMA}"."dict" d

            JOIN global_query_terms q
              ON d.term = q.term
        ),

        {permitted_docs_cte}

        qterms AS (

            SELECT
                t.docid,
                t.termid

            FROM "{FTS_SCHEMA}"."terms" t

            JOIN qtermids q
              ON t.termid = q.termid

            {qterms_filter_join}
        ),

        term_tf AS (

            SELECT
                docid,
                termid,
                COUNT(*) AS tf

            FROM qterms

            GROUP BY
                docid,
                termid
        ),

        subscores AS (

            SELECT
                tf.docid,
                tf.termid,

                LOG(
                    (
                        (
                            (
                                {num_docs}
                                - q.global_df
                                + 0.5
                            )
                            /
                            (
                                q.global_df
                                + 0.5
                            )
                        )
                        + 1
                    )
                )

                *

                (
                    tf.tf
                    *
                    ({BM25_K} + 1)
                )

                /

                (
                    tf.tf

                    +

                    (
                        {BM25_K}
                        *
                        (
                            (1 - {BM25_B})

                            +

                            (
                                {BM25_B}
                                *
                                (
                                    d.len
                                    /
                                    {avgdl}
                                )
                            )
                        )
                    )
                )

                AS bm25_subscore

            FROM term_tf tf

            JOIN "{FTS_SCHEMA}"."docs" d
              ON tf.docid = d.docid

            JOIN qtermids q
              ON tf.termid = q.termid
        ),

        scores AS (

            SELECT
                docid,

                SUM(
                    bm25_subscore
                ) AS bm25_score,

                COUNT(
                    DISTINCT termid
                ) AS matched_terms

            FROM subscores

            GROUP BY
                docid

            ORDER BY
                bm25_score DESC,
                matched_terms DESC

            LIMIT {int(top_k)}
        )

        SELECT
            c.doc_id,
            c.source_family,
            c.collection,
            c.identifier,
            c.title,
            c.release,
            c.document_type,
            c.source_path,

            s.matched_terms,
            s.bm25_score

        FROM scores s

        JOIN "{FTS_SCHEMA}"."docs" d
          ON s.docid = d.docid

        JOIN "{FTS_TABLE}" c
          ON c.doc_id = d.name

        ORDER BY
            s.bm25_score DESC,
            s.matched_terms DESC
        """


        results = conn.execute(
            sql,
            parameters
        ).df()


    finally:

        if conn is not None:

            conn.close()


    if not results.empty:

        results[
            "shard_name"
        ] = shard_name


    return results


# ============================================================
# FETCH TEXT ONLY FOR FINAL RESULTS
# ============================================================

def extract_query_focused_window(
    text,
    query_terms,
    max_chars=RESULT_TEXT_CHARS
):
    """
    Extract a query-focused evidence window from a full document.

    The persistent 3GPP rows can represent long specifications.
    Returning LEFT(text, N) therefore often yields only cover
    pages or tables of contents.

    Strategy:
        1. Find occurrences of the active query terms.
        2. Build candidate windows around those occurrences.
        3. Score each window by unique-term coverage and total
           term occurrences.
        4. Return the highest-density evidence window.

    Falls back to the beginning of the document only when none
    of the query terms are present.
    """

    raw_text = str(
        text
        or
        ""
    )


    if not raw_text:

        return ""


    if len(
        raw_text
    ) <= max_chars:

        return raw_text


    normalized_text = (
        raw_text.lower()
    )


    terms = list(
        dict.fromkeys(
            [
                str(
                    term
                )
                .strip()
                .lower()

                for term
                in (
                    query_terms
                    or
                    []
                )

                if str(
                    term
                ).strip()
            ]
        )
    )


    if not terms:

        return raw_text[
            :max_chars
        ]


    # --------------------------------------------------------
    # COLLECT A BOUNDED SET OF TERM OCCURRENCES
    # --------------------------------------------------------

    anchors = []


    for term in terms:

        search_start = 0

        occurrences = 0


        while occurrences < 12:

            position = (
                normalized_text.find(
                    term,
                    search_start
                )
            )


            if position < 0:

                break


            anchors.append(
                position
            )


            occurrences += 1

            search_start = (
                position
                +
                max(
                    1,
                    len(
                        term
                    )
                )
            )


    if not anchors:

        return raw_text[
            :max_chars
        ]


    # --------------------------------------------------------
    # SCORE CANDIDATE WINDOWS
    # --------------------------------------------------------

    best_window = None

    best_score = None


    for anchor in anchors:

        # Keep approximately one-third of the window before
        # the anchor so section headings/context are retained.
        start_pos = max(
            0,
            anchor
            -
            (
                max_chars
                //
                3
            )
        )


        end_pos = min(
            len(
                raw_text
            ),
            start_pos
            +
            max_chars
        )


        start_pos = max(
            0,
            end_pos
            -
            max_chars
        )


        window_lower = (
            normalized_text[
                start_pos:
                end_pos
            ]
        )


        unique_coverage = sum(
            1
            for term
            in terms
            if term
            in window_lower
        )


        total_occurrences = sum(
            window_lower.count(
                term
            )
            for term
            in terms
        )


        # Unique query-term coverage is more important than
        # repeated occurrence of one high-frequency term.
        score = (
            unique_coverage,
            total_occurrences,
            -start_pos
        )


        if (
            best_score is None
            or
            score
            >
            best_score
        ):

            best_score = score

            best_window = (
                start_pos,
                end_pos
            )


    start_pos, end_pos = (
        best_window
    )


    excerpt = (
        raw_text[
            start_pos:
            end_pos
        ]
    )


    if start_pos > 0:

        excerpt = (
            "... "
            +
            excerpt
        )


    if end_pos < len(
        raw_text
    ):

        excerpt = (
            excerpt
            +
            " ..."
        )


    return excerpt


def fetch_result_text(
    shard_name,
    doc_id,
    query_terms=None,
    max_chars=RESULT_TEXT_CHARS
):
    """
    Fetch the full text for a final Top-K document, then return
    the most query-relevant bounded evidence window.

    Full text is fetched only for final selected evidence items,
    so this does not expand the candidate search workload.
    """

    shard_path = (
        SHARD_DIR
        / f"{shard_name}.duckdb"
    )

    conn = None


    try:

        conn = duckdb.connect(
            str(
                shard_path
            ),
            read_only=True
        )


        row = conn.execute(
            f"""
            SELECT
                text

            FROM "{FTS_TABLE}"

            WHERE doc_id = ?
            """,
            [
                doc_id
            ]
        ).fetchone()


    finally:

        if conn is not None:

            conn.close()


    full_text = (
        row[
            0
        ]
        if row
        else ""
    )


    return extract_query_focused_window(
        text=full_text,
        query_terms=query_terms,
        max_chars=max_chars
    )


# ============================================================
# DYNAMIC ROUTED BM25 SEARCH
# ============================================================

def run_dynamic_bm25_search(
    query,
    top_k=TOP_K_RESULTS
):
    """
    Execute the Version B persistent BM25 retrieval path:

        1. Validate query
        2. Apply Cell 5 deterministic 3GPP/TCC/HYBRID router
        3. Resolve the route to persistent shard names
        4. Telecom-aware query processing
        5. Compute BM25 statistics across ALL selected shards
        6. Select Top-N high-IDF candidate terms
        7. Search selected shards in parallel
        8. Globally rank candidates under one score space
        9. Fetch text only for final Top-K
       10. Return results + complete retrieval trace

    Returns:

        final_results : pandas.DataFrame
        retrieval_trace : dict
    """

    start = time.perf_counter()


    # ========================================================
    # INPUT VALIDATION
    # ========================================================

    if (
        not isinstance(
            query,
            str
        )
        or
        not query.strip()
    ):

        raise ValueError(
            "Search query cannot be empty."
        )


    top_k = max(
        1,
        min(
            int(
                top_k
            ),
            TOP_K_RESULTS
        )
    )


    # ========================================================
    # DETERMINISTIC ROUTING
    # ========================================================

    route_plan = (
        build_route_shard_plan(
            query
        )
    )


    shard_names = list(
        route_plan[
            "selected_shards"
        ]
    )


    if not shard_names:

        raise RuntimeError(
            "Dynamic router selected no persistent BM25 shards."
        )


    # ========================================================
    # QUERY PROCESSING
    # ========================================================

    query_terms = process_query(
        query
    )


    if not query_terms:

        elapsed = (
            time.perf_counter()
            -
            start
        )


        return (
            pd.DataFrame(),
            {
                "architecture":
                    "Version B",

                "retrieval_architecture":
                    "Persistent DuckDB BM25/FTS",

                "query":
                    query,

                "route":
                    route_plan[
                        "route"
                    ],

                "sources_searched":
                    list(
                        route_plan[
                            "sources_searched"
                        ]
                    ),

                "gpp_specs":
                    list(
                        route_plan[
                            "gpp_specs"
                        ]
                    ),

                "tcc_collections":
                    list(
                        route_plan[
                            "tcc_collections"
                        ]
                    ),

                "selected_shards":
                    shard_names,

                "selected_shard_count":
                    len(
                        shard_names
                    ),

                "query_terms":
                    [],

                "candidate_terms":
                    [],

                "candidate_term_limit":
                    int(
                        ROUTE_MAX_CANDIDATE_TERMS
                    ),

                "candidate_term_details":
                    [],

                "global_num_docs":
                    0,

                "global_avgdl":
                    0.0,

                "workers":
                    0,

                "shard_errors":
                    [],

                "result_count":
                    0,

                "retrieval_time_s":
                    elapsed
            }
        )


    # ========================================================
    # COMMON BM25 STATISTICS SPACE
    # ========================================================

    (
        global_docs,
        global_avgdl,
        global_df_map
    ) = get_selected_shard_stats(
        shard_names,
        query_terms
    )


    # ========================================================
    # SELECT TOP HIGH-IDF TERMS
    # ========================================================

    (
        candidate_terms,
        scored_terms
    ) = select_candidate_terms(
        query_terms,
        global_df_map,
        global_docs,
        max_terms=ROUTE_MAX_CANDIDATE_TERMS
    )


    # ========================================================
    # PRESERVE ROUTE-DEFINING TERMS
    # ========================================================
    #
    # High-IDF pruning alone can remove essential low-IDF terms
    # such as HTTP or TLS. Those terms determine the selected
    # source domain and must remain active in routed retrieval.
    # ========================================================

    route_signal_text = " ".join(
        list(
            route_plan.get(
                "gpp_signal_hits",
                []
            )
            or
            []
        )
        +
        list(
            route_plan.get(
                "tcc_signal_hits",
                []
            )
            or
            []
        )
    )


    forced_route_terms = list(
        dict.fromkeys(
            [
                term
                for term
                in telecom_tokenize(
                    route_signal_text
                )
                if global_df_map.get(
                    term,
                    0
                ) > 0
            ]
        )
    )


    candidate_terms = list(
        dict.fromkeys(
            forced_route_terms
            +
            candidate_terms
        )
    )


    candidate_term_details = [
        {
            "term":
                term,

            "document_frequency":
                int(
                    df
                ),

            "idf":
                float(
                    idf
                ),

            "forced_by_router":
                (
                    term
                    in
                    forced_route_terms
                )
        }
        for term, df, idf
        in scored_terms
    ]


    # ========================================================
    # HYBRID SOURCE-SPECIFIC INTERNAL EVIDENCE INTENTS
    # ========================================================

    if route_plan["route"] == "hybrid":
        hybrid_focus_terms = build_hybrid_focus_terms(
            query=query,
            route_plan=route_plan,
            candidate_terms=candidate_terms,
            global_df_map=global_df_map
        )
    else:
        hybrid_focus_terms = {
            "3GPP": list(candidate_terms),
            "TCC": list(candidate_terms),
        }


    if not candidate_terms:

        elapsed = (
            time.perf_counter()
            -
            start
        )


        return (
            pd.DataFrame(),
            {
                "architecture":
                    "Version B",

                "retrieval_architecture":
                    "Persistent DuckDB BM25/FTS",

                "query":
                    query,

                "route":
                    route_plan[
                        "route"
                    ],

                "sources_searched":
                    list(
                        route_plan[
                            "sources_searched"
                        ]
                    ),

                "gpp_specs":
                    list(
                        route_plan[
                            "gpp_specs"
                        ]
                    ),

                "tcc_collections":
                    list(
                        route_plan[
                            "tcc_collections"
                        ]
                    ),

                "selected_shards":
                    shard_names,

                "selected_shard_count":
                    len(
                        shard_names
                    ),

                "query_terms":
                    query_terms,

                "candidate_terms":
                    [],

                "candidate_term_limit":
                    int(
                        ROUTE_MAX_CANDIDATE_TERMS
                    ),

                "candidate_term_details":
                    candidate_term_details,

                "global_num_docs":
                    int(
                        global_docs
                    ),

                "global_avgdl":
                    float(
                        global_avgdl
                    ),

                "workers":
                    0,

                "shard_errors":
                    [],

                "result_count":
                    0,

                "retrieval_time_s":
                    elapsed
            }
        )


    # ========================================================
    # SOURCE-FOCUSED TERMS PER SHARD
    # ========================================================

    def terms_for_shard(shard_name):
        if route_plan["route"] == "hybrid":
            if shard_name in DEDICATED_3GPP_SHARDS:
                return list(hybrid_focus_terms["3GPP"])
            return list(hybrid_focus_terms["TCC"])
        return list(candidate_terms)


    # ========================================================
    # PARALLEL SHARD SEARCH
    # ========================================================

    workers = min(
        MAX_SEARCH_WORKERS,
        len(
            shard_names
        )
    )


    shard_results = []

    shard_errors = []


    with ThreadPoolExecutor(
        max_workers=workers
    ) as executor:

        futures = {

            executor.submit(
                search_filtered_shard,

                shard_name,
                terms_for_shard(
                    shard_name
                ),

                global_docs,
                global_avgdl,
                global_df_map,

                top_k,

                (
                    route_plan[
                        "gpp_specs"
                    ]
                    if shard_name
                    in DEDICATED_3GPP_SHARDS
                    else None
                )
            ):
            shard_name

            for shard_name
            in shard_names
        }


        for future in as_completed(
            futures
        ):

            shard_name = (
                futures[
                    future
                ]
            )


            try:

                result = (
                    future.result()
                )


                if not result.empty:

                    shard_results.append(
                        result
                    )


            except Exception as exc:

                shard_errors.append({
                    "shard_name":
                        shard_name,

                    "error":
                        str(
                            exc
                        )
                })


    # ========================================================
    # STRICT SHARD ERROR CHECK
    # ========================================================

    if shard_errors:

        error_df = pd.DataFrame(
            shard_errors
        )


        print(
            "\nBM25 shard search errors:"
        )

        display(
            error_df
        )


        raise RuntimeError(
            "Dynamic BM25 retrieval failed "
            "on one or more selected shards."
        )


    # ========================================================
    # NO RESULTS
    # ========================================================

    if not shard_results:

        elapsed = (
            time.perf_counter()
            -
            start
        )


        retrieval_trace = {

            "architecture":
                "Version B",

            "retrieval_architecture":
                "Persistent DuckDB BM25/FTS",

            "query":
                query,

            "route":
                route_plan[
                    "route"
                ],

            "sources_searched":
                list(
                    route_plan[
                        "sources_searched"
                    ]
                ),

            "gpp_specs":
                list(
                    route_plan[
                        "gpp_specs"
                    ]
                ),

            "gpp_reasons":
                list(
                    route_plan[
                        "gpp_reasons"
                    ]
                ),

            "tcc_collections":
                list(
                    route_plan[
                        "tcc_collections"
                    ]
                ),

            "tcc_reasons":
                list(
                    route_plan[
                        "tcc_reasons"
                    ]
                ),

            "selected_shards":
                shard_names,

            "selected_shard_count":
                len(
                    shard_names
                ),

            "query_terms":
                query_terms,

            "candidate_terms":
                candidate_terms,

            "forced_route_terms":
                forced_route_terms,

            "hybrid_focus_terms":
                {
                    "3GPP": list(hybrid_focus_terms["3GPP"]),
                    "TCC": list(hybrid_focus_terms["TCC"]),
                },

            "candidate_term_limit":
                int(
                    ROUTE_MAX_CANDIDATE_TERMS
                ),

            "candidate_term_details":
                candidate_term_details,

            "global_num_docs":
                int(
                    global_docs
                ),

            "global_avgdl":
                float(
                    global_avgdl
                ),

            "workers":
                int(
                    workers
                ),

            "shard_errors":
                [],

            "result_count":
                0,

            "source_distribution":
                {},

            "collection_distribution":
                {},

            "retrieval_time_s":
                elapsed
        }


        return (
            pd.DataFrame(),
            retrieval_trace
        )


    # ========================================================
    # GLOBAL MERGE + SOURCE-AWARE TOP-K
    # ========================================================

    candidates = pd.concat(
        shard_results,
        ignore_index=True
    )


    candidates = (

        candidates

        .sort_values(
            [
                "bm25_score",
                "matched_terms"
            ],
            ascending=[
                False,
                False
            ]
        )

        .drop_duplicates(
            subset="doc_id"
        )

        .reset_index(
            drop=True
        )
    )


    hybrid_balance_applied = False

    hybrid_reserved_per_source = 0


    if (
        route_plan[
            "route"
        ]
        ==
        "hybrid"
    ):

        hybrid_balance_applied = True


        hybrid_reserved_per_source = min(
            HYBRID_MIN_RESULTS_PER_SOURCE,
            max(
                1,
                top_k
                //
                2
            )
        )


        gpp_candidates = (
            candidates.loc[
                candidates[
                    "source_family"
                ]
                .fillna("")
                .astype(str)
                .eq(
                    "3GPP"
                )
            ]
            .head(
                hybrid_reserved_per_source
            )
        )


        tcc_candidates = (
            candidates.loc[
                candidates[
                    "source_family"
                ]
                .fillna("")
                .astype(str)
                .eq(
                    "TCC"
                )
            ]
            .head(
                hybrid_reserved_per_source
            )
        )


        # A true Hybrid route requires evidence from both sides.
        if (
            gpp_candidates.empty
            or
            tcc_candidates.empty
        ):

            final_results = (
                candidates
                .head(
                    top_k
                )
                .copy()
            )


        else:

            reserved = pd.concat(
                [
                    gpp_candidates,
                    tcc_candidates
                ],
                ignore_index=True
            )


            reserved_doc_ids = set(
                reserved[
                    "doc_id"
                ]
                .astype(str)
                .tolist()
            )


            remaining = (
                candidates.loc[
                    ~candidates[
                        "doc_id"
                    ]
                    .astype(str)
                    .isin(
                        reserved_doc_ids
                    )
                ]
            )


            slots_remaining = max(
                0,
                top_k
                -
                len(
                    reserved
                )
            )


            final_results = pd.concat(
                [
                    reserved,
                    remaining.head(
                        slots_remaining
                    )
                ],
                ignore_index=True
            )


            # Preserve one common BM25 score space for rank order
            # while maintaining the Hybrid source representation
            # guaranteed above.
            final_results = (
                final_results
                .sort_values(
                    [
                        "bm25_score",
                        "matched_terms"
                    ],
                    ascending=[
                        False,
                        False
                    ]
                )
                .reset_index(
                    drop=True
                )
            )


    else:

        final_results = (
            candidates
            .head(
                top_k
            )
            .copy()
            .reset_index(
                drop=True
            )
        )


    final_results.insert(
        0,
        "rank",
        range(
            1,
            len(
                final_results
            )
            +
            1
        )
    )


    # ========================================================
    # SOURCE-FOCUSED EVIDENCE WINDOWS
    # ========================================================

    def evidence_focus_terms(source_family):
        if route_plan["route"] == "hybrid":
            family = str(source_family or "").strip()
            if family == "3GPP":
                return list(hybrid_focus_terms["3GPP"])
            if family == "TCC":
                return list(hybrid_focus_terms["TCC"])
        return list(candidate_terms)


    # ========================================================
    # FETCH TEXT ONLY AFTER GLOBAL TOP-K IS KNOWN
    # ========================================================

    final_results[
        "text_preview"
    ] = final_results.apply(

        lambda row:
            fetch_result_text(
                shard_name=row[
                    "shard_name"
                ],
                doc_id=row[
                    "doc_id"
                ],
                query_terms=evidence_focus_terms(
                    row[
                        "source_family"
                    ]
                )
            ),

        axis=1
    )


    # ========================================================
    # RESULT DISTRIBUTIONS
    # ========================================================

    source_distribution = (
        final_results[
            "source_family"
        ]
        .fillna("")
        .astype(str)
        .value_counts()
        .to_dict()
    )


    collection_distribution = (
        final_results[
            "collection"
        ]
        .fillna("")
        .astype(str)
        .value_counts()
        .to_dict()
    )


    # ========================================================
    # TOTAL RETRIEVAL LATENCY
    # ========================================================

    elapsed = (
        time.perf_counter()
        -
        start
    )


    # ========================================================
    # COMPLETE RETRIEVAL TRACE
    # ========================================================

    retrieval_trace = {

        "architecture":
            "Version B",

        "retrieval_architecture":
            "Persistent DuckDB BM25/FTS",

        "query":
            query,

        "route":
            route_plan[
                "route"
            ],

        "sources_searched":
            list(
                route_plan[
                    "sources_searched"
                ]
            ),

        "gpp_specs":
            list(
                route_plan[
                    "gpp_specs"
                ]
            ),

        "gpp_reasons":
            list(
                route_plan[
                    "gpp_reasons"
                ]
            ),

        "tcc_collections":
            list(
                route_plan[
                    "tcc_collections"
                ]
            ),

        "tcc_reasons":
            list(
                route_plan[
                    "tcc_reasons"
                ]
            ),

        "selected_shards":
            shard_names,

        "selected_shard_count":
            len(
                shard_names
            ),

        "query_terms":
            query_terms,

        "candidate_terms":
            candidate_terms,

        "hybrid_focus_terms":
            {
                "3GPP": list(hybrid_focus_terms["3GPP"]),
                "TCC": list(hybrid_focus_terms["TCC"]),
            },

        "candidate_term_details":
            candidate_term_details,

        "global_num_docs":
            int(
                global_docs
            ),

        "global_avgdl":
            float(
                global_avgdl
            ),

        "workers":
            int(
                workers
            ),

        "shard_errors":
            [],

        "result_count":
            int(
                len(
                    final_results
                )
            ),

        "source_distribution":
            {
                str(
                    key
                ):
                int(
                    value
                )
                for key, value
                in source_distribution.items()
            },

        "collection_distribution":
            {
                str(
                    key
                ):
                int(
                    value
                )
                for key, value
                in collection_distribution.items()
            },

        "hybrid_balance_applied":
            bool(
                hybrid_balance_applied
            ),

        "hybrid_reserved_per_source":
            int(
                hybrid_reserved_per_source
            ),

        "evidence_window_strategy":
            "query-focused term-density window",

        "retrieval_time_s":
            float(
                elapsed
            )
    }


    return (
        final_results,
        retrieval_trace
    )


# ============================================================
# COMPATIBILITY ALIAS
# ============================================================
#
# Future Version B cells should call run_dynamic_bm25_search().
#
# This alias keeps the notebook easy to read while avoiding
# accidental use of the old LLM-facing profile argument.
# ============================================================

def search_version_b_knowledge(
    query,
    top_k=TOP_K_RESULTS
):
    """
    Canonical Version B retrieval entry point.
    """

    return run_dynamic_bm25_search(
        query=query,
        top_k=top_k
    )


# ============================================================
# RUNTIME RETRIEVAL SUMMARY
# ============================================================

print("=" * 90)

print(
    "VERSION B — DYNAMIC ROUTED BM25 RETRIEVAL ENGINE"
)

print("=" * 90)


print(
    "Query Processing      : "
    "Telecom-aware tokenizer"
)

print(
    "Source Routing        : "
    "Deterministic 3GPP / TCC / HYBRID"
)

print(
    "Shard Selection       : "
    "Cell 5 route-selected persistent shards"
)

print(
    "Hybrid Score Space    : "
    "One common BM25 statistics space"
)

print(
    "Hybrid Top-K Policy   : "
    f"Minimum {HYBRID_MIN_RESULTS_PER_SOURCE} results/source"
)

print(
    f"Candidate Terms       : "
    f"Top-{ROUTE_MAX_CANDIDATE_TERMS} high-IDF"
)

print(
    f"Max Search Workers    : "
    f"{MAX_SEARCH_WORKERS}"
)

print(
    f"Global Top-K          : "
    f"{TOP_K_RESULTS}"
)

print(
    f"Fetched Result Text   : "
    f"{RESULT_TEXT_CHARS:,} chars"
)

print(
    f"BM25 k                : "
    f"{BM25_K}"
)

print(
    f"BM25 b                : "
    f"{BM25_B}"
)


print(
    "\n" + "=" * 90
)


print(
    "✓ Profile argument removed from the canonical retrieval path."
)

print(
    "✓ Cell 5 dynamically selects persistent BM25 shards."
)

print(
    "✓ Dedicated 3GPP candidates are constrained to inferred specifications."
)

print(
    "✓ 3GPP/TCC/HYBRID all use common selected-shard BM25 statistics."
)

print(
    "✓ Hybrid evidence uses one BM25 score space with source-balanced Top-K."
)

print(
    "✓ Router-defining terms are preserved after IDF pruning."
)

print(
    "✓ Hybrid uses source-specific internal evidence intents."
)

print(
    "✓ 3GPP Hybrid branch focuses on 5GS/SBA network-function concepts."
)

print(
    "✓ TCC Hybrid branch focuses on protocol concepts such as HTTP/TLS."
)

print(
    "✓ Final evidence text uses source-focused query windows."
)

print(
    "✓ Complete routing/retrieval trace is returned for evaluation."
)

print(
    "✓ Full result text is fetched only after final Top-K selection."
)

print(
    "✓ No retrieval query executed in this cell."
)

print(
    "✓ Ready for Cell 7 — Three-Route Retrieval Validation."
)

print("=" * 90)


VERSION B — DYNAMIC ROUTED BM25 RETRIEVAL ENGINE
Query Processing      : Telecom-aware tokenizer
Source Routing        : Deterministic 3GPP / TCC / HYBRID
Shard Selection       : Cell 5 route-selected persistent shards
Hybrid Score Space    : One common BM25 statistics space
Hybrid Top-K Policy   : Minimum 2 results/source
Candidate Terms       : Top-8 high-IDF
Max Search Workers    : 6
Global Top-K          : 5
Fetched Result Text   : 5,000 chars
BM25 k                : 1.2
BM25 b                : 0.75

✓ Profile argument removed from the canonical retrieval path.
✓ Cell 5 dynamically selects persistent BM25 shards.
✓ Dedicated 3GPP candidates are constrained to inferred specifications.
✓ 3GPP/TCC/HYBRID all use common selected-shard BM25 statistics.
✓ Hybrid evidence uses one BM25 score space with source-balanced Top-K.
✓ Router-defining terms are preserved after IDF pruning.
✓ Hybrid uses source-specific internal evidence intents.
✓ 3GPP Hybrid branch focuses on 5GS/SBA network-func

**Observation — Dynamic Routed BM25 Retrieval**

The runtime searches only shards selected by the deterministic router, compares candidates in a common BM25 statistics space and applies source-aware evidence selection for Hybrid queries.

***Key Decision:*** Keep candidate/evidence optimization inside retrieval while exposing only the provider-neutral MCP contract to the LLM.


## **Cell 7 — Three-Route Retrieval Validation**


In [15]:
# ============================================================
# CELL 7 — VERSION B THREE-ROUTE RETRIEVAL VALIDATION
# ============================================================
#
# Validates the persistent BM25 retrieval layer BEFORE MCP:
#
#   P1 → 3GPP-only
#   P2 → TCC-only / IETF
#   P3 → Hybrid 3GPP + TCC
#
# No LLM or MCP call is made in this cell.
# ============================================================


VALIDATION_CASES = [
    {
        "id": "P1",
        "name": "3GPP-only",
        "question": (
            "Explain the role of the AMF in registration and mobility "
            "management procedures in a 5G Standalone network."
        ),
        "expected_route": "3gpp",
        "expected_sources": {"3GPP"},
        "expected_tcc_collections": set(),
        "expected_3gpp_identifiers": {"23.501", "23.502"},
        "required_evidence_terms": {"amf"},
        "required_any_evidence_terms": {"registration", "mobility"},
    },
    {
        "id": "P2",
        "name": "TCC-only",
        "question": (
            "Explain the key mechanisms of QUIC as defined by the IETF, "
            "including connection establishment, stream multiplexing and "
            "connection migration."
        ),
        "expected_route": "tcc",
        "expected_sources": {"TCC"},
        "expected_tcc_collections": {"IETF-RFCs", "IETF-Drafts"},
        "expected_3gpp_identifiers": set(),
        "required_evidence_terms": {"quic", "migration"},
        "required_any_evidence_terms": {"multiplex", "stream"},
    },
    {
        "id": "P3",
        "name": "Hybrid 3GPP + TCC",
        "question": (
            "Explain how HTTP and TLS support communication in the 5G "
            "Service-Based Architecture. Distinguish the roles and "
            "requirements defined by 3GPP for network functions such as "
            "the AMF and SMF from the HTTP/TLS transport and security "
            "mechanisms defined by the IETF."
        ),
        "expected_route": "hybrid",
        "expected_sources": {"3GPP", "TCC"},
        "expected_tcc_collections": {"IETF-RFCs", "IETF-Drafts"},
        "expected_3gpp_identifiers": {"23.501", "33.501"},
        "required_evidence_terms": {"http", "tls"},
        "required_any_evidence_terms": {"amf", "smf", "service-based"},
    },
]


REQUIRED_RESULT_COLUMNS = {
    "rank",
    "doc_id",
    "source_family",
    "collection",
    "identifier",
    "title",
    "release",
    "document_type",
    "source_path",
    "shard_name",
    "matched_terms",
    "bm25_score",
    "text_preview",
}


REQUIRED_TRACE_FIELDS = {
    "architecture",
    "retrieval_architecture",
    "query",
    "route",
    "sources_searched",
    "gpp_specs",
    "tcc_collections",
    "selected_shards",
    "selected_shard_count",
    "query_terms",
    "candidate_terms",
    "global_num_docs",
    "global_avgdl",
    "workers",
    "shard_errors",
    "result_count",
    "source_distribution",
    "collection_distribution",
    "retrieval_time_s",
}


def clean_set(values):
    return {
        str(value).strip()
        for value in (values or [])
        if str(value).strip()
    }


def evidence_contains_term(
    evidence_text,
    term
):
    """
    Lightweight retrieval-quality check.

    Terms such as 'multiplex' intentionally allow matches like:
        multiplex
        multiplexing
        multiplexed
    """

    return (
        str(term).lower()
        in
        str(evidence_text).lower()
    )


def validate_case(case, results, trace):

    checks = {}

    # --------------------------------------------------------
    # ROUTE
    # --------------------------------------------------------

    actual_route = str(
        trace.get("route", "")
    ).strip().lower()

    checks["expected_route"] = (
        actual_route == case["expected_route"]
    )

    # --------------------------------------------------------
    # TOP-K / UNIQUENESS
    # --------------------------------------------------------

    checks["results_returned"] = (
        not results.empty
    )

    checks["top_k_returned"] = (
        len(results) == TOP_K_RESULTS
    )

    unique_docs = (
        results["doc_id"].nunique()
        if "doc_id" in results.columns
        else 0
    )

    checks["unique_documents"] = (
        unique_docs == TOP_K_RESULTS
    )

    # --------------------------------------------------------
    # SCHEMA
    # --------------------------------------------------------

    missing_result_columns = (
        REQUIRED_RESULT_COLUMNS
        - set(results.columns)
    )

    missing_trace_fields = (
        REQUIRED_TRACE_FIELDS
        - set(trace.keys())
    )

    checks["result_schema"] = (
        not missing_result_columns
    )

    checks["trace_schema"] = (
        not missing_trace_fields
    )

    # --------------------------------------------------------
    # SHARD PLAN / BM25 STATS
    # --------------------------------------------------------

    selected_shards = list(
        trace.get("selected_shards", []) or []
    )

    checks["selected_shards"] = (
        len(selected_shards) > 0
        and
        len(selected_shards)
        ==
        int(trace.get("selected_shard_count", 0) or 0)
    )

    checks["no_shard_errors"] = (
        len(trace.get("shard_errors", []) or []) == 0
    )

    checks["bm25_statistics"] = (
        int(trace.get("global_num_docs", 0) or 0) > 0
        and
        float(trace.get("global_avgdl", 0.0) or 0.0) > 0
    )

    checks["candidate_terms"] = (
        len(trace.get("candidate_terms", []) or []) > 0
    )

    # --------------------------------------------------------
    # SOURCE FAMILY VALIDATION
    # --------------------------------------------------------

    observed_sources = clean_set(
        results["source_family"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    routed_sources = clean_set(
        trace.get("sources_searched", [])
    )

    checks["expected_source_families"] = (
        case["expected_sources"]
        .issubset(observed_sources)
    )

    checks["sources_searched"] = (
        case["expected_sources"]
        .issubset(routed_sources)
    )

    # --------------------------------------------------------
    # TCC COLLECTION SELECTION
    # --------------------------------------------------------

    routed_tcc_collections = clean_set(
        trace.get("tcc_collections", [])
    )

    if case["expected_tcc_collections"]:

        checks["tcc_collection_selection"] = (
            case["expected_tcc_collections"]
            .issubset(routed_tcc_collections)
        )

    else:

        # In Version A-compatible routing, the router may still
        # carry 3GPP-TSG as TCC metadata for a pure 3GPP query.
        # It must not actually be searched in the 3GPP route.
        checks["tcc_collection_selection"] = True

    # --------------------------------------------------------
    # 3GPP FAIRNESS
    # --------------------------------------------------------

    if case["expected_route"] == "3gpp":

        checks["dedicated_3gpp_only"] = (
            observed_sources == {"3GPP"}
        )

    else:

        checks["dedicated_3gpp_only"] = True

    # --------------------------------------------------------
    # HYBRID FAIRNESS
    # --------------------------------------------------------

    observed_collections = clean_set(
        results["collection"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    if case["expected_route"] == "hybrid":

        checks["hybrid_evidence_mix"] = (
            {"3GPP", "TCC"}
            .issubset(observed_sources)
        )

        checks["hybrid_excludes_tcc_3gpp_tsg"] = (
            "3GPP-TSG"
            not in observed_collections
        )

    else:

        checks["hybrid_evidence_mix"] = True
        checks["hybrid_excludes_tcc_3gpp_tsg"] = True

    # --------------------------------------------------------
    # RETRIEVAL QUALITY GUARDRAILS
    # --------------------------------------------------------
    #
    # Structural route correctness is not enough.
    #
    # Validate that:
    #   - routed 3GPP evidence includes at least one inferred/
    #     expected specification where applicable;
    #   - returned evidence contains the core technical concepts
    #     required by the pilot question.
    #
    # This is a lightweight smoke-test guardrail only.
    # Final relevance/groundedness scoring remains in Notebook 17.
    # --------------------------------------------------------

    result_identifiers = clean_set(
        results["identifier"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    expected_3gpp_identifiers = set(
        case.get(
            "expected_3gpp_identifiers",
            set()
        )
    )

    if expected_3gpp_identifiers:

        observed_3gpp_identifiers = clean_set(
            results.loc[
                results["source_family"]
                .fillna("")
                .astype(str)
                .eq("3GPP"),
                "identifier"
            ]
            .fillna("")
            .astype(str)
            .tolist()
        )

        checks["expected_3gpp_spec_evidence"] = bool(
            expected_3gpp_identifiers
            &
            observed_3gpp_identifiers
        )

    else:

        observed_3gpp_identifiers = set()

        checks["expected_3gpp_spec_evidence"] = True


    combined_evidence_text = "\\n".join(
        results["text_preview"]
        .fillna("")
        .astype(str)
        .tolist()
    ).lower()


    required_terms = set(
        case.get(
            "required_evidence_terms",
            set()
        )
    )

    checks["required_evidence_terms"] = all(
        evidence_contains_term(
            combined_evidence_text,
            term
        )
        for term
        in required_terms
    )


    required_any_terms = set(
        case.get(
            "required_any_evidence_terms",
            set()
        )
    )

    checks["required_any_evidence_term"] = (
        True
        if not required_any_terms
        else any(
            evidence_contains_term(
                combined_evidence_text,
                term
            )
            for term
            in required_any_terms
        )
    )


    # Hybrid must have technically meaningful evidence
    # from BOTH source families, not merely one result from each.
    if case["expected_route"] == "hybrid":

        gpp_evidence_text = "\\n".join(
            results.loc[
                results["source_family"]
                .fillna("")
                .astype(str)
                .eq("3GPP"),
                "text_preview"
            ]
            .fillna("")
            .astype(str)
            .tolist()
        ).lower()

        tcc_evidence_text = "\\n".join(
            results.loc[
                results["source_family"]
                .fillna("")
                .astype(str)
                .eq("TCC"),
                "text_preview"
            ]
            .fillna("")
            .astype(str)
            .tolist()
        ).lower()

        checks["hybrid_3gpp_semantic_evidence"] = any(
            evidence_contains_term(
                gpp_evidence_text,
                term
            )
            for term
            in {
                "amf",
                "smf",
                "service-based",
                "service based"
            }
        )

        checks["hybrid_tcc_http_tls_evidence"] = (
            evidence_contains_term(
                tcc_evidence_text,
                "http"
            )
            and
            evidence_contains_term(
                tcc_evidence_text,
                "tls"
            )
        )

    else:

        checks["hybrid_3gpp_semantic_evidence"] = True
        checks["hybrid_tcc_http_tls_evidence"] = True


    # --------------------------------------------------------
    # EVIDENCE TEXT
    # --------------------------------------------------------

    evidence_with_text = int(
        results["text_preview"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    checks["evidence_text"] = (
        evidence_with_text == TOP_K_RESULTS
    )

    # --------------------------------------------------------
    # RANKING / SCORE
    # --------------------------------------------------------

    checks["ranking"] = (
        results["rank"].astype(int).tolist()
        ==
        list(range(1, TOP_K_RESULTS + 1))
    )

    checks["bm25_scores"] = (
        results["bm25_score"].notna().all()
    )

    checks["trace_result_count"] = (
        int(trace.get("result_count", 0) or 0)
        ==
        len(results)
    )

    checks["retrieval_latency"] = (
        float(trace.get("retrieval_time_s", 0.0) or 0.0)
        > 0
    )

    details = {
        "actual_route": actual_route,
        "observed_sources": sorted(observed_sources),
        "routed_sources": sorted(routed_sources),
        "routed_tcc_collections": sorted(
            routed_tcc_collections
        ),
        "selected_shards": selected_shards,
        "unique_docs": int(unique_docs),
        "evidence_with_text": evidence_with_text,
        "expected_3gpp_identifiers": sorted(
            expected_3gpp_identifiers
        ),
        "observed_3gpp_identifiers": sorted(
            observed_3gpp_identifiers
        ),
        "required_evidence_terms": sorted(
            required_terms
        ),
        "required_any_evidence_terms": sorted(
            required_any_terms
        ),
        "missing_result_columns": sorted(
            missing_result_columns
        ),
        "missing_trace_fields": sorted(
            missing_trace_fields
        ),
    }

    return checks, details


# ============================================================
# RUN VALIDATION
# ============================================================

print("=" * 100)
print("VERSION B — THREE-ROUTE RETRIEVAL VALIDATION")
print("=" * 100)

VALIDATION_RESULTS = []


for case in VALIDATION_CASES:

    print("\n" + "=" * 100)
    print(f"{case['id']} — {case['name']}")
    print("=" * 100)

    print(f"Question       : {case['question']}")
    print(
        f"Expected Route : "
        f"{case['expected_route'].upper()}"
    )

    (
        results,
        trace
    ) = search_version_b_knowledge(
        query=case["question"],
        top_k=TOP_K_RESULTS
    )

    checks, details = validate_case(
        case=case,
        results=results,
        trace=trace
    )

    passed = all(checks.values())

    VALIDATION_RESULTS.append({
        "case": case,
        "results": results,
        "trace": trace,
        "checks": checks,
        "details": details,
        "passed": passed,
    })

    print(
        f"\nActual Route   : "
        f"{details['actual_route'].upper()}"
    )

    print(
        f"Sources        : "
        f"{details['observed_sources']}"
    )

    print(
        f"Router Sources : "
        f"{details['routed_sources']}"
    )

    print(
        f"TCC Collections: "
        f"{details['routed_tcc_collections']}"
    )

    print(
        f"BM25 Shards    : "
        f"{trace['selected_shard_count']}"
    )

    print(
        f"Global Docs    : "
        f"{trace['global_num_docs']:,}"
    )

    print(
        f"Global AvgDL   : "
        f"{trace['global_avgdl']:.2f}"
    )

    print(
        f"Candidate Terms: "
        f"{trace['candidate_terms']}"
    )

    if details[
        "expected_3gpp_identifiers"
    ]:

        print(
            f"Expected Specs : "
            f"{details['expected_3gpp_identifiers']}"
        )

        print(
            f"Observed Specs : "
            f"{details['observed_3gpp_identifiers']}"
        )

    print(
        f"Required Terms : "
        f"{details['required_evidence_terms']}"
    )

    print(
        f"Required Any   : "
        f"{details['required_any_evidence_terms']}"
    )

    print(
        f"Results        : "
        f"{len(results)}"
    )

    print(
        f"Unique Docs    : "
        f"{details['unique_docs']}"
    )

    print(
        f"Evidence Text  : "
        f"{details['evidence_with_text']}"
        f"/{TOP_K_RESULTS}"
    )

    print(
        f"Latency        : "
        f"{trace['retrieval_time_s']:.3f} sec"
    )

    print("\nValidation Checks")
    print("-" * 100)

    for name, ok in checks.items():
        print(
            f"{name:<36} : "
            f"{'PASS' if ok else 'FAIL'}"
        )

    print("\nTop-K Retrieval Results")
    print("-" * 100)

    display(
        results[
            [
                "rank",
                "source_family",
                "collection",
                "identifier",
                "title",
                "shard_name",
                "matched_terms",
                "bm25_score",
            ]
        ].round({
            "bm25_score": 4
        })
    )

    if not results.empty:

        top_result = results.iloc[0]

        evidence_text = str(
            top_result["text_preview"] or ""
        )

        print("\nTop-1 Evidence Sample")
        print("-" * 100)

        print(
            f"Source Family : "
            f"{top_result['source_family']}"
        )

        print(
            f"Collection    : "
            f"{top_result['collection']}"
        )

        print(
            f"Identifier    : "
            f"{top_result['identifier']}"
        )

        print(
            f"Title         : "
            f"{top_result['title']}"
        )

        print(
            f"Shard         : "
            f"{top_result['shard_name']}"
        )

        print(
            f"Evidence Chars: "
            f"{len(evidence_text):,}"
        )

        print("\nEvidence Preview:")
        print(
            evidence_text[:1200]
        )

    print("\n" + "-" * 100)
    print(
        f"{case['id']} STATUS: "
        f"{'PASS' if passed else 'REVIEW'}"
    )
    print("-" * 100)


# ============================================================
# SUMMARY
# ============================================================

summary_rows = []

for item in VALIDATION_RESULTS:

    summary_rows.append({
        "pilot": item["case"]["id"],
        "route": item["trace"]["route"],
        "source_families": ",".join(
            item["details"]["observed_sources"]
        ),
        "selected_shards": int(
            item["trace"]["selected_shard_count"]
        ),
        "results": len(item["results"]),
        "evidence_with_text": int(
            item["details"]["evidence_with_text"]
        ),
        "retrieval_time_s": round(
            float(
                item["trace"]["retrieval_time_s"]
            ),
            3
        ),
        "status": (
            "PASS"
            if item["passed"]
            else "REVIEW"
        ),
    })


RETRIEVAL_VALIDATION_SUMMARY = pd.DataFrame(
    summary_rows
)


print("\n" + "=" * 100)
print(
    "VERSION B — THREE-ROUTE RETRIEVAL "
    "VALIDATION SUMMARY"
)
print("=" * 100)

display(
    RETRIEVAL_VALIDATION_SUMMARY
)


all_passed = all(
    item["passed"]
    for item in VALIDATION_RESULTS
)


print("\n" + "=" * 100)


if all_passed:

    print(
        "✓ VERSION B THREE-ROUTE RETRIEVAL "
        "VALIDATION PASSED"
    )

    print(
        f"✓ 3GPP route returned Top-{TOP_K_RESULTS} "
        "dedicated 3GPP evidence."
    )

    print(
        f"✓ TCC route returned Top-{TOP_K_RESULTS} "
        "TCC evidence."
    )

    print(
        "✓ Hybrid route searched dedicated 3GPP + "
        "non-3GPP TCC shards under one BM25 score space."
    )

    print(
        "✓ Hybrid Top-K contains both 3GPP and TCC evidence."
    )

    print(
        "✓ TCC 3GPP-TSG evidence is excluded from Hybrid."
    )

    print(
        "✓ All final retrieval results contain document text."
    )

    print(
        "✓ Complete route/retrieval traces are available."
    )

    print(
        "✓ Lightweight retrieval-quality guardrails passed."
    )

    print(
        "✓ Expected 3GPP specification evidence is represented."
    )

    print(
        "✓ Hybrid evidence is semantically meaningful on both sides."
    )

    print(
        "✓ Persistent BM25 layer is ready for MCP exposure."
    )

else:

    failed_cases = [
        item["case"]["id"]
        for item in VALIDATION_RESULTS
        if not item["passed"]
    ]

    print(
        "⚠ VERSION B RETRIEVAL VALIDATION "
        "REQUIRES REVIEW"
    )

    print(
        f"Failed cases: {failed_cases}"
    )

    print(
        "Inspect the failed validation checks above "
        "before continuing to MCP exposure."
    )


print("=" * 100)


if not all_passed:

    raise RuntimeError(
        "Version B three-route retrieval validation "
        "did not fully pass."
    )


VERSION B — THREE-ROUTE RETRIEVAL VALIDATION

P1 — 3GPP-only
Question       : Explain the role of the AMF in registration and mobility management procedures in a 5G Standalone network.
Expected Route : 3GPP

Actual Route   : 3GPP
Sources        : ['3GPP']
Router Sources : ['3GPP']
TCC Collections: ['3GPP-TSG']
BM25 Shards    : 2
Global Docs    : 15,052
Global AvgDL   : 24671.85
Candidate Terms: ['5g', 'standalone', 'amf', 'registration', 'mobility', 'management', 'role', 'procedures']
Expected Specs : ['23.501', '23.502']
Observed Specs : ['23.501', '23.502']
Required Terms : ['amf']
Required Any   : ['mobility', 'registration']
Results        : 5
Unique Docs    : 5
Evidence Text  : 5/5
Latency        : 1.824 sec

Validation Checks
----------------------------------------------------------------------------------------------------
expected_route                       : PASS
results_returned                     : PASS
top_k_returned                       : PASS
unique_documents         

,rank,source_family,collection,identifier,title,shard_name,matched_terms,bm25_score
0,1,3GPP,3GPP-Specifications,23.501,3GPP TS 23.501 V18.4.0 (2023-12),3gpp_3gpp_specifications_01,8,8.3903
1,2,3GPP,3GPP-Specifications,23.502,3GPP TS 23.502 V20.0.0 (2025-12),3gpp_3gpp_specifications_00,8,8.3226
2,3,3GPP,3GPP-Specifications,23.501,3GPP TS 23.501 V20.0.0 (2025-12) ---,3gpp_3gpp_specifications_01,8,8.3161
3,4,3GPP,3GPP-Specifications,23.501,3rd Generation Partnership Project; Technical ...,3gpp_3gpp_specifications_01,8,8.2988
4,5,3GPP,3GPP-Specifications,23.502,3GPP TS 23.502 V18.4.0 (2023-12),3gpp_3gpp_specifications_00,8,8.2659



Top-1 Evidence Sample
----------------------------------------------------------------------------------------------------
Source Family : 3GPP
Collection    : 3GPP-Specifications
Identifier    : 23.501
Title         : 3GPP TS 23.501 V18.4.0 (2023-12)
Shard         : 3gpp_3gpp_specifications_01
Evidence Chars: 5,008

Evidence Preview:
... ows".
- [178] IEEE Std 802.1CBdb-2021: "Amendment 2: Extend Stream Identification Functions".
- [179] 3GPP TS 26.522: "5G Real-time Media Transport Protocol Configurations".
- [180] 3GPP TS 23.586: "Architectural Enhancements to support Ranging based services and Sidelink Positioning".
- [181] 3GPP TS 23.542: "Application layer support for Personal IoT Network".
- [182] IETF RFC 8415: "Dynamic Host Configuration Protocol for IPv6 (DHCPv6)".
- [183] 3GPP TS 29.571: "5G System; Common Data Types for Service Based Interfaces; Stage 3".
- [184] 3GPP TS 23.289: "Mission Critical services over 5G System; Stage 2".
- [185] IETF RFC 3550: "RTP: A Transport P

,rank,source_family,collection,identifier,title,shard_name,matched_terms,bm25_score
0,1,TCC,IETF-Drafts,draft-hamilton-quic-transport-protocol-1,None,tcc_ietf_drafts_00,7,17.1154
1,2,TCC,IETF-Drafts,draft-sz-dmsc-iaip-2,None,tcc_ietf_drafts_00,7,16.6437
2,3,TCC,IETF-Drafts,draft-tsvwg-quic-protocol-2,None,tcc_ietf_drafts_00,7,16.1810
3,4,TCC,IETF-Drafts,draft-hamilton-early-deployment-quic,QUIC: A UDP-Based Secure and Reliable Transpor...,tcc_ietf_drafts_00,7,15.6292
4,5,TCC,IETF-Drafts,draft-michel-remote-terminal-http3,Remote terminal over HTTP/3 connections,tcc_ietf_drafts_00,6,15.5080



Top-1 Evidence Sample
----------------------------------------------------------------------------------------------------
Source Family : TCC
Collection    : IETF-Drafts
Identifier    : draft-hamilton-quic-transport-protocol-1
Title         : None
Shard         : tcc_ietf_drafts_00
Evidence Chars: 3,135

Evidence Preview:
### 1 Introduction

QUIC is a multiplexed and secure transport protocol that runs on top of UDP.  QUIC builds on past transport experience and implements mechanisms that make it useful as a modern general-purpose transport protocol.  Using UDP as the substrate, QUIC seeks to be compatible with legacy clients and middleboxes.  QUIC authenticates all of its headers, preventing middleboxes and other third parties from changing them, and encrypts most of its headers, limiting protocol evolution largely to QUIC endpoints only.

This document describes the core QUIC protocol, including the conceptual design, wire format, and mechanisms of the QUIC protocol

Internet-Draft

,rank,source_family,collection,identifier,title,shard_name,matched_terms,bm25_score
0,1,3GPP,3GPP-Specifications,23.501,3GPP TS 23.501 V20.0.0 (2025-12) ---,3gpp_3gpp_specifications_01,5,17.1727
1,2,3GPP,3GPP-Specifications,23.501,3rd Generation Partnership Project; Technical ...,3gpp_3gpp_specifications_01,5,16.8631
2,3,3GPP,3GPP-Specifications,23.501,3GPP TS 23.501 V18.4.0 (2023-12),3gpp_3gpp_specifications_01,5,16.8605
3,4,TCC,IETF-Drafts,draft-friel-tls-atls-5,None,tcc_ietf_drafts_00,2,4.1923
4,5,TCC,IETF-Drafts,draft-friel-tls-atls-5,None,tcc_ietf_drafts_00,2,4.1798



Top-1 Evidence Sample
----------------------------------------------------------------------------------------------------
Source Family : 3GPP
Collection    : 3GPP-Specifications
Identifier    : 23.501
Title         : 3GPP TS 23.501 V20.0.0 (2025-12) ---
Shard         : 3gpp_3gpp_specifications_01
Evidence Chars: 5,008

Evidence Preview:
... unction (NEF).
- Network Repository Function (NRF).
- Network Slice Admission Control Function (NSACF).
- Network Slice-specific and SNPN Authentication and Authorization Function (NSSAAF).
- Network Slice Selection Function (NSSF).
- Policy Control Function (PCF).
- Session Management Function (SMF).
- Unified Data Management (UDM).
- Unified Data Repository (UDR).
- User Plane Function (UPF).
- UE radio Capability Management Function (UCMF).
- Application Function (AF).
- User Equipment (UE).
- (Radio) Access Network ((R)AN).
- 5G-Equipment Identity Register (5G-EIR).
- Network Data Analytics Function (NWDAF).
- CHarging Function (CHF).
- Time 

,pilot,route,source_families,selected_shards,results,evidence_with_text,retrieval_time_s,status
0,P1,3gpp,3GPP,2,5,5,1.824,PASS
1,P2,tcc,TCC,2,5,5,1.764,PASS
2,P3,hybrid,"3GPP,TCC",4,5,5,2.393,PASS



✓ VERSION B THREE-ROUTE RETRIEVAL VALIDATION PASSED
✓ 3GPP route returned Top-5 dedicated 3GPP evidence.
✓ TCC route returned Top-5 TCC evidence.
✓ Hybrid route searched dedicated 3GPP + non-3GPP TCC shards under one BM25 score space.
✓ Hybrid Top-K contains both 3GPP and TCC evidence.
✓ TCC 3GPP-TSG evidence is excluded from Hybrid.
✓ All final retrieval results contain document text.
✓ Complete route/retrieval traces are available.
✓ Lightweight retrieval-quality guardrails passed.
✓ Expected 3GPP specification evidence is represented.
✓ Hybrid evidence is semantically meaningful on both sides.
✓ Persistent BM25 layer is ready for MCP exposure.


**Observation — Three-Route Retrieval Validation**

The persistent BM25 layer passed direct **3GPP-only, TCC/IETF-only and Hybrid 3GPP+TCC** validation before MCP or LLM orchestration.

***Key Decision:*** Prove route/source/evidence behaviour independently so later model/tool-call issues can be separated from the BM25 layer.


# **SECTION 3 — MCP Knowledge Service**

## **Cell 8 — MCP Telecom Knowledge Search Tool**

In [16]:
# ============================================================
# CELL 8 — UNIFIED MCP TELECOM KNOWLEDGE SEARCH TOOL
# ============================================================
#
# Purpose:
#
# Expose the Version B persistent BM25 retrieval architecture
# through ONE provider-neutral MCP tool:
#
#     search_telecom_knowledge(query, top_k=5)
#
# The LLM does NOT select:
#
#     - a profile
#     - a source family
#     - a collection
#     - a shard
#
# Internal flow:
#
#     LLM query
#       ↓
#     MCP tool
#       ↓
#     deterministic 3GPP / TCC / HYBRID router
#       ↓
#     persistent DuckDB BM25 retrieval
#       ↓
#     source-aware Hybrid evidence selection
#       ↓
#     bounded evidence payload
#
# The response schema is intentionally aligned with Version A
# so Notebook 17 can compare architectures directly.
# ============================================================


from fastmcp import FastMCP


# ============================================================
# MCP SERVER
# ============================================================

mcp = FastMCP(
    "Telecom Knowledge Service — Version B"
)


# ============================================================
# SAFE SERIALIZATION HELPERS
# ============================================================

def safe_int(
    value,
    default=0
):
    """
    Convert common numeric values safely to int.
    """

    try:

        if pd.isna(
            value
        ):

            return int(
                default
            )

    except Exception:

        pass


    try:

        return int(
            value
        )

    except Exception:

        return int(
            default
        )


def safe_float(
    value,
    default=0.0
):
    """
    Convert common numeric values safely to float.
    """

    try:

        if pd.isna(
            value
        ):

            return float(
                default
            )

    except Exception:

        pass


    try:

        return float(
            value
        )

    except Exception:

        return float(
            default
        )


def clean_mcp_text(
    value,
    max_chars=None
):
    """
    Normalize MCP text fields without changing their meaning.
    """

    if value is None:

        return ""


    text = str(
        value
    ).strip()


    if (
        max_chars is not None
        and
        len(
            text
        )
        >
        int(
            max_chars
        )
    ):

        return (
            text[
                :int(
                    max_chars
                )
            ]
            .rstrip()
        )


    return text


# ============================================================
# SOURCE-SPECIFIC EVIDENCE TERMS
# ============================================================

def get_mcp_evidence_terms(
    retrieval_trace,
    source_family
):
    """
    Return the retrieval terms used to focus the bounded MCP
    evidence excerpt.

    For HYBRID retrieval, Cell 6 maintains separate internal
    evidence intents:

        3GPP → standards/SBA/network-function concepts
        TCC  → complementary protocol concepts

    For non-Hybrid routes, use the normal candidate-term list.
    """

    source_family = (
        str(
            source_family
            or
            ""
        )
        .strip()
    )


    hybrid_focus_terms = (
        retrieval_trace.get(
            "hybrid_focus_terms",
            {}
        )
        or
        {}
    )


    source_terms = (
        hybrid_focus_terms.get(
            source_family,
            []
        )
        or
        []
    )


    if source_terms:

        return list(
            source_terms
        )


    return list(
        retrieval_trace.get(
            "candidate_terms",
            []
        )
        or
        []
    )


# ============================================================
# CONTROLLED MCP EVIDENCE EXCERPT
# ============================================================

def build_mcp_excerpt(
    text,
    evidence_terms,
    max_chars=MCP_EXCERPT_CHARS
):
    """
    Bound the already query-focused Version B result text to the
    common MCP experiment evidence limit.

    Cell 6 has already extracted a source-aware, query-focused
    window from the persistent document. This function performs
    the final MCP payload bound while preserving that focus.
    """

    text = clean_mcp_text(
        text
    )


    if not text:

        return ""


    if (
        len(
            text
        )
        <=
        int(
            max_chars
        )
    ):

        return text


    # Reuse the query-focused window helper defined in Cell 6.
    return clean_mcp_text(
        extract_query_focused_window(
            text=text,
            query_terms=evidence_terms,
            max_chars=int(
                max_chars
            )
        )
    )


# ============================================================
# FORMAT ONE MCP EVIDENCE ITEM
# ============================================================

def format_version_b_mcp_evidence(
    row,
    retrieval_trace
):
    """
    Convert one ranked BM25 result into the canonical evidence
    schema used for Version A / Version B comparison.
    """

    source_family = clean_mcp_text(
        row.get(
            "source_family",
            ""
        )
    )


    evidence_terms = (
        get_mcp_evidence_terms(
            retrieval_trace=retrieval_trace,
            source_family=source_family
        )
    )


    original_text = clean_mcp_text(
        row.get(
            "text_preview",
            ""
        )
    )


    evidence_text = build_mcp_excerpt(
        text=original_text,
        evidence_terms=evidence_terms,
        max_chars=MCP_EXCERPT_CHARS
    )


    bm25_score = safe_float(
        row.get(
            "bm25_score",
            0.0
        )
    )


    return {

        # ----------------------------------------------------
        # COMMON EVIDENCE IDENTITY
        # ----------------------------------------------------

        "rank":
            safe_int(
                row.get(
                    "rank",
                    0
                )
            ),

        "source_family":
            source_family,

        "collection":
            clean_mcp_text(
                row.get(
                    "collection",
                    ""
                )
            ),

        "identifier":
            clean_mcp_text(
                row.get(
                    "identifier",
                    ""
                )
            ),

        "title":
            clean_mcp_text(
                row.get(
                    "title",
                    ""
                ),
                max_chars=1000
            ),

        "release":
            clean_mcp_text(
                row.get(
                    "release",
                    ""
                )
            ),

        "document_type":
            clean_mcp_text(
                row.get(
                    "document_type",
                    ""
                )
            ),

        "source_path":
            clean_mcp_text(
                row.get(
                    "source_path",
                    ""
                ),
                max_chars=2000
            ),

        "source_shard":
            clean_mcp_text(
                row.get(
                    "shard_name",
                    ""
                )
            ),

        # Version B indexes document-level records and does not
        # persist a separate canonical section-heading field.
        "section_heading":
            "",

        # ----------------------------------------------------
        # COMMON + ARCHITECTURE-SPECIFIC SCORING
        # ----------------------------------------------------

        # Common field used by cross-architecture analysis.
        "relevance_score":
            bm25_score,

        # Explicit architecture-specific field.
        "bm25_score":
            bm25_score,

        "matched_terms":
            safe_int(
                row.get(
                    "matched_terms",
                    0
                )
            ),

        # Version A lexical retrieval exposes these additional
        # scoring components. They are structurally retained here
        # as zero-valued non-applicable fields for schema parity.
        "matched_phrases":
            0,

        "proximity_score":
            0.0,

        "primary_coverage":
            0.0,

        # ----------------------------------------------------
        # EXACT BOUNDED EVIDENCE USED BY THE LLM
        # ----------------------------------------------------

        "evidence":
            evidence_text,

        "evidence_chars":
            len(
                evidence_text
            ),

        "evidence_terms":
            list(
                evidence_terms
            )
    }


# ============================================================
# BUILD MCP TRACE
# ============================================================

def build_version_b_mcp_trace(
    query,
    retrieval_trace,
    tool_elapsed_s
):
    """
    Build the canonical Version B retrieval/MCP trace.

    Preserve the complete Cell 6 retrieval trace rather than
    reducing it to only latency or candidate terms.
    """

    trace = dict(
        retrieval_trace
        or
        {}
    )


    trace.update({

        "architecture":
            "Version B",

        "retrieval_architecture":
            "Persistent DuckDB BM25/FTS",

        "mcp_server":
            "Telecom Knowledge Service — Version B",

        "mcp_tool":
            "search_telecom_knowledge",

        "query":
            query,

        "tool_elapsed_s":
            float(
                tool_elapsed_s
            )
    })


    return trace


# ============================================================
# UNIFIED MCP TELECOM KNOWLEDGE TOOL
# ============================================================

@mcp.tool()
def search_telecom_knowledge(
    query: str,
    top_k: int = TOP_K_RESULTS
) -> dict:
    """
    Search the persistent Version B telecom knowledge base.

    The source route is selected automatically and
    deterministically from the query.

    Possible internal routes:
        3gpp
        tcc
        hybrid

    The caller does not select profiles, collections or shards.

    Returns:
        ranked bounded telecom evidence plus complete routing
        and persistent BM25 retrieval trace metadata.
    """

    tool_start = (
        time.perf_counter()
    )


    # ========================================================
    # INPUT VALIDATION
    # ========================================================

    if (
        not isinstance(
            query,
            str
        )
        or
        not query.strip()
    ):

        raise ValueError(
            "Search query cannot be empty."
        )


    query = (
        query.strip()
    )


    # Common A/B evidence limit.
    effective_top_k = max(
        1,
        min(
            safe_int(
                top_k,
                TOP_K_RESULTS
            ),
            TOP_K_RESULTS,
            MAX_RETRIEVED_SOURCES
        )
    )


    # ========================================================
    # PERSISTENT BM25 RETRIEVAL
    # ========================================================

    (
        results,
        retrieval_trace
    ) = search_version_b_knowledge(
        query=query,
        top_k=effective_top_k
    )


    # ========================================================
    # FORMAT EXACT EVIDENCE PAYLOAD
    # ========================================================

    evidence = []


    if not results.empty:

        for _, row in (
            results.iterrows()
        ):

            evidence.append(
                format_version_b_mcp_evidence(
                    row=row,
                    retrieval_trace=retrieval_trace
                )
            )


    # ========================================================
    # SOURCE DISTRIBUTION
    # ========================================================

    source_distribution = {}


    for item in evidence:

        source_family = (
            item.get(
                "source_family",
                ""
            )
            or
            "UNKNOWN"
        )


        source_distribution[
            source_family
        ] = (
            source_distribution.get(
                source_family,
                0
            )
            +
            1
        )


    # ========================================================
    # COMPLETE MCP TRACE
    # ========================================================

    tool_elapsed_s = (
        time.perf_counter()
        -
        tool_start
    )


    trace = (
        build_version_b_mcp_trace(
            query=query,
            retrieval_trace=retrieval_trace,
            tool_elapsed_s=tool_elapsed_s
        )
    )


    # ========================================================
    # CANONICAL MCP RESPONSE
    # ========================================================
    #
    # This shape mirrors Version A:
    #
    #     query
    #     route
    #     sources_searched
    #     source_distribution
    #     result_count
    #     top_k
    #     evidence
    #     trace
    #
    # ========================================================

    return {

        "query":
            query,

        "route":
            clean_mcp_text(
                retrieval_trace.get(
                    "route",
                    ""
                )
            ),

        "sources_searched":
            list(
                retrieval_trace.get(
                    "sources_searched",
                    []
                )
                or
                []
            ),

        "source_distribution":
            {
                str(
                    key
                ):
                safe_int(
                    value
                )

                for key, value
                in source_distribution.items()
            },

        "result_count":
            len(
                evidence
            ),

        "top_k":
            effective_top_k,

        "evidence":
            evidence,

        "trace":
            trace
    }


# ============================================================
# MCP SERVER / TOOL SUMMARY
# ============================================================

print("=" * 90)

print(
    "VERSION B — UNIFIED MCP TELECOM KNOWLEDGE SERVICE"
)

print("=" * 90)


print(
    "Server               : "
    "Telecom Knowledge Service — Version B"
)

print(
    "Tool                 : "
    "search_telecom_knowledge"
)

print(
    "LLM Tool Signature   : "
    "search_telecom_knowledge(query, top_k=5)"
)

print(
    "Source Selection     : "
    "Internal deterministic 3GPP / TCC / HYBRID router"
)

print(
    "Retrieval            : "
    "Persistent DuckDB BM25/FTS"
)

print(
    "Hybrid Retrieval     : "
    "Source-focused + source-balanced evidence"
)

print(
    f"Max Sources / Call   : "
    f"{MAX_RETRIEVED_SOURCES}"
)

print(
    f"Top-K                : "
    f"{TOP_K_RESULTS}"
)

print(
    f"Evidence Excerpt     : "
    f"{MCP_EXCERPT_CHARS:,} chars/source"
)


print(
    "\n" + "=" * 90
)


print(
    "✓ Old LLM-facing profile argument removed."
)

print(
    "✓ One unified provider-neutral MCP tool exposed."
)

print(
    "✓ Version A routing semantics preserved internally."
)

print(
    "✓ Persistent BM25 retrieval remains hidden behind MCP."
)

print(
    "✓ Hybrid 3GPP/TCC evidence composition preserved."
)

print(
    "✓ Exact bounded evidence text is returned for later grounding analysis."
)

print(
    "✓ Full routing/retrieval trace is preserved in the MCP response."
)

print(
    "✓ Response schema aligned with Version A."
)

print(
    "✓ Ready for Cell 9 — Three-Route MCP Tool Validation."
)

print("=" * 90)


VERSION B — UNIFIED MCP TELECOM KNOWLEDGE SERVICE
Server               : Telecom Knowledge Service — Version B
Tool                 : search_telecom_knowledge
LLM Tool Signature   : search_telecom_knowledge(query, top_k=5)
Source Selection     : Internal deterministic 3GPP / TCC / HYBRID router
Retrieval            : Persistent DuckDB BM25/FTS
Hybrid Retrieval     : Source-focused + source-balanced evidence
Max Sources / Call   : 5
Top-K                : 5
Evidence Excerpt     : 2,500 chars/source

✓ Old LLM-facing profile argument removed.
✓ One unified provider-neutral MCP tool exposed.
✓ Version A routing semantics preserved internally.
✓ Persistent BM25 retrieval remains hidden behind MCP.
✓ Hybrid 3GPP/TCC evidence composition preserved.
✓ Exact bounded evidence text is returned for later grounding analysis.
✓ Full routing/retrieval trace is preserved in the MCP response.
✓ Response schema aligned with Version A.
✓ Ready for Cell 9 — Three-Route MCP Tool Validation.


**Observation — Unified MCP Knowledge Service**

Version B exposes one provider-neutral `search_telecom_knowledge(query, top_k)` tool. Source routing, collection/specification selection, shard selection and Hybrid evidence balancing remain internal.

***Key Decision:*** MCP is the access/orchestration boundary; no LLM-facing `profile` parameter is exposed.


## **Cell 9 — MCP Tool Validation**

In [17]:
# ============================================================
# CELL 9 — THREE-ROUTE MCP TOOL VALIDATION
# ============================================================
#
# Purpose:
#
# Validate the Version B MCP boundary independently of the LLM.
#
# Cell 7 already validated the persistent BM25 retrieval engine.
# This cell validates that the SAME behavior survives:
#
#     FastMCP server
#       ↓
#     real FastMCP Client
#       ↓
#     search_telecom_knowledge(query, top_k=5)
#       ↓
#     structured MCP response
#
# Validation routes:
#
#     P1 → 3GPP-only
#     P2 → TCC-only / IETF
#     P3 → Hybrid 3GPP + TCC
#
# No LLM call is made in this cell.
# ============================================================


from fastmcp import Client


# ============================================================
# MCP VALIDATION CASES
# ============================================================

MCP_VALIDATION_CASES = [

    {
        "id":
            "P1",

        "name":
            "3GPP-only",

        "question":
            (
                "Explain the role of the AMF in registration "
                "and mobility management procedures in a "
                "5G Standalone network."
            ),

        "expected_route":
            "3gpp",

        "expected_source_families":
            {
                "3GPP"
            },

        "expected_tcc_collections":
            set(),

        "expected_3gpp_identifiers":
            {
                "23.501",
                "23.502"
            }
    },

    {
        "id":
            "P2",

        "name":
            "TCC-only",

        "question":
            (
                "Explain the key mechanisms of QUIC as defined "
                "by the IETF, including connection establishment, "
                "stream multiplexing and connection migration."
            ),

        "expected_route":
            "tcc",

        "expected_source_families":
            {
                "TCC"
            },

        "expected_tcc_collections":
            {
                "IETF-RFCs",
                "IETF-Drafts"
            },

        "expected_3gpp_identifiers":
            set()
    },

    {
        "id":
            "P3",

        "name":
            "Hybrid 3GPP + TCC",

        "question":
            (
                "Explain how HTTP and TLS support communication "
                "in the 5G Service-Based Architecture. Distinguish "
                "the roles and requirements defined by 3GPP for "
                "network functions such as the AMF and SMF from "
                "the HTTP/TLS transport and security mechanisms "
                "defined by the IETF."
            ),

        "expected_route":
            "hybrid",

        "expected_source_families":
            {
                "3GPP",
                "TCC"
            },

        "expected_tcc_collections":
            {
                "IETF-RFCs",
                "IETF-Drafts"
            },

        "expected_3gpp_identifiers":
            {
                "23.501",
                "33.501"
            }
    }
]


# ============================================================
# REQUIRED MCP RESPONSE FIELDS
# ============================================================

REQUIRED_MCP_PAYLOAD_FIELDS = {

    "query",
    "route",
    "sources_searched",
    "source_distribution",

    "result_count",
    "top_k",

    "evidence",
    "trace"
}


REQUIRED_MCP_EVIDENCE_FIELDS = {

    "rank",

    "source_family",
    "collection",
    "identifier",
    "title",
    "release",
    "document_type",

    "source_path",
    "source_shard",
    "section_heading",

    "relevance_score",
    "bm25_score",
    "matched_terms",

    "matched_phrases",
    "proximity_score",
    "primary_coverage",

    "evidence",
    "evidence_chars",
    "evidence_terms"
}


REQUIRED_MCP_TRACE_FIELDS = {

    "architecture",
    "retrieval_architecture",

    "mcp_server",
    "mcp_tool",

    "query",
    "route",
    "sources_searched",

    "gpp_specs",
    "tcc_collections",

    "selected_shards",
    "selected_shard_count",

    "query_terms",
    "candidate_terms",

    "global_num_docs",
    "global_avgdl",

    "workers",
    "shard_errors",

    "result_count",

    "source_distribution",
    "collection_distribution",

    "retrieval_time_s",
    "tool_elapsed_s"
}


# ============================================================
# HELPERS
# ============================================================

def clean_string_set(
    values
):
    """
    Normalize iterable values to a clean set of strings.
    """

    return {
        str(
            value
        ).strip()

        for value
        in (
            values
            or
            []
        )

        if str(
            value
        ).strip()
    }


def normalize_source_family(
    value
):
    """
    Normalize the source-family labels used in A/B evaluation.
    """

    value = (
        str(
            value
            or
            ""
        )
        .strip()
        .upper()
    )


    if value == "3GPP":

        return "3GPP"


    if value == "TCC":

        return "TCC"


    return value


def normalized_source_set(
    values
):
    """
    Normalize multiple source-family values.
    """

    return {
        normalize_source_family(
            value
        )

        for value
        in (
            values
            or
            []
        )

        if normalize_source_family(
            value
        )
    }


# ============================================================
# VALIDATE ONE MCP PAYLOAD
# ============================================================

def validate_mcp_payload(
    case,
    payload
):
    """
    Validate one structured MCP response.

    This validates the MCP serialization boundary, not just
    the underlying retrieval engine.
    """

    checks = {}


    # --------------------------------------------------------
    # PAYLOAD TYPE / SCHEMA
    # --------------------------------------------------------

    checks[
        "payload_is_dict"
    ] = isinstance(
        payload,
        dict
    )


    if not checks[
        "payload_is_dict"
    ]:

        return (
            checks,
            {
                "error":
                    "Payload is not a dictionary."
            }
        )


    missing_payload_fields = (
        REQUIRED_MCP_PAYLOAD_FIELDS
        -
        set(
            payload.keys()
        )
    )


    checks[
        "payload_schema"
    ] = (
        len(
            missing_payload_fields
        )
        ==
        0
    )


    # --------------------------------------------------------
    # TOOL CONTRACT
    # --------------------------------------------------------

    checks[
        "no_profile_field"
    ] = (
        "profile"
        not in
        payload
    )


    checks[
        "query_roundtrip"
    ] = (
        str(
            payload.get(
                "query",
                ""
            )
        )
        .strip()
        ==
        case[
            "question"
        ]
        .strip()
    )


    # --------------------------------------------------------
    # ROUTE
    # --------------------------------------------------------

    actual_route = (
        str(
            payload.get(
                "route",
                ""
            )
        )
        .strip()
        .lower()
    )


    checks[
        "expected_route"
    ] = (
        actual_route
        ==
        case[
            "expected_route"
        ]
    )


    # --------------------------------------------------------
    # EVIDENCE COUNT
    # --------------------------------------------------------

    evidence = (
        payload.get(
            "evidence",
            []
        )
        or
        []
    )


    result_count = safe_int(
        payload.get(
            "result_count",
            0
        )
    )


    effective_top_k = safe_int(
        payload.get(
            "top_k",
            0
        )
    )


    checks[
        "result_count"
    ] = (
        result_count
        ==
        len(
            evidence
        )
        ==
        TOP_K_RESULTS
    )


    checks[
        "top_k_contract"
    ] = (
        effective_top_k
        ==
        TOP_K_RESULTS
    )


    checks[
        "max_source_limit"
    ] = (
        result_count
        <=
        MAX_RETRIEVED_SOURCES
    )


    # --------------------------------------------------------
    # EVIDENCE SCHEMA
    # --------------------------------------------------------

    evidence_schema_ok = True

    empty_evidence_items = 0

    oversized_evidence_items = 0

    bad_evidence_char_counts = 0

    bad_ranks = 0

    missing_evidence_fields = []


    expected_ranks = list(
        range(
            1,
            len(
                evidence
            )
            +
            1
        )
    )


    actual_ranks = []


    for index, item in enumerate(
        evidence,
        start=1
    ):

        if not isinstance(
            item,
            dict
        ):

            evidence_schema_ok = False

            missing_evidence_fields.append(
                {
                    "rank":
                        index,

                    "missing":
                        [
                            "evidence_item_not_dict"
                        ]
                }
            )

            continue


        missing_fields = (
            REQUIRED_MCP_EVIDENCE_FIELDS
            -
            set(
                item.keys()
            )
        )


        if missing_fields:

            evidence_schema_ok = False

            missing_evidence_fields.append(
                {
                    "rank":
                        index,

                    "missing":
                        sorted(
                            missing_fields
                        )
                }
            )


        evidence_text = (
            str(
                item.get(
                    "evidence",
                    ""
                )
                or
                ""
            )
            .strip()
        )


        evidence_chars = safe_int(
            item.get(
                "evidence_chars",
                0
            )
        )


        if not evidence_text:

            empty_evidence_items += 1


        if (
            evidence_chars
            !=
            len(
                evidence_text
            )
        ):

            bad_evidence_char_counts += 1


        # extract_query_focused_window() may add short
        # leading/trailing ellipses around the bounded window.
        if (
            evidence_chars
            >
            MCP_EXCERPT_CHARS
            +
            10
        ):

            oversized_evidence_items += 1


        actual_rank = safe_int(
            item.get(
                "rank",
                0
            )
        )


        actual_ranks.append(
            actual_rank
        )


        if actual_rank != index:

            bad_ranks += 1


    checks[
        "evidence_schema"
    ] = (
        evidence_schema_ok
    )


    checks[
        "evidence_non_empty"
    ] = (
        empty_evidence_items
        ==
        0
    )


    checks[
        "evidence_char_counts"
    ] = (
        bad_evidence_char_counts
        ==
        0
    )


    checks[
        "evidence_bound"
    ] = (
        oversized_evidence_items
        ==
        0
    )


    checks[
        "evidence_ranks"
    ] = (
        bad_ranks
        ==
        0

        and

        actual_ranks
        ==
        expected_ranks
    )


    # --------------------------------------------------------
    # SOURCE DISTRIBUTION
    # --------------------------------------------------------

    observed_sources = normalized_source_set(
        [
            item.get(
                "source_family",
                ""
            )
            for item
            in evidence
        ]
    )


    routed_sources = normalized_source_set(
        payload.get(
            "sources_searched",
            []
        )
    )


    expected_sources = set(
        case[
            "expected_source_families"
        ]
    )


    checks[
        "expected_source_families"
    ] = (
        expected_sources
        .issubset(
            observed_sources
        )
    )


    checks[
        "sources_searched"
    ] = (
        expected_sources
        .issubset(
            routed_sources
        )
    )


    payload_source_distribution = {
        normalize_source_family(
            key
        ):
        safe_int(
            value
        )

        for key, value
        in (
            payload.get(
                "source_distribution",
                {}
            )
            or
            {}
        ).items()
    }


    actual_source_distribution = {}


    for item in evidence:

        source_family = (
            normalize_source_family(
                item.get(
                    "source_family",
                    ""
                )
            )
            or
            "UNKNOWN"
        )


        actual_source_distribution[
            source_family
        ] = (
            actual_source_distribution.get(
                source_family,
                0
            )
            +
            1
        )


    checks[
        "source_distribution"
    ] = (
        payload_source_distribution
        ==
        actual_source_distribution
    )


    # --------------------------------------------------------
    # SOURCE-SPECIFIC CONTRACT
    # --------------------------------------------------------

    if case[
        "expected_route"
    ] == "3gpp":

        checks[
            "dedicated_3gpp_only"
        ] = (
            observed_sources
            ==
            {
                "3GPP"
            }

            and

            all(
                str(
                    item.get(
                        "collection",
                        ""
                    )
                )
                ==
                "3GPP-Specifications"

                for item
                in evidence
            )
        )

    else:

        checks[
            "dedicated_3gpp_only"
        ] = True


    if case[
        "expected_route"
    ] == "hybrid":

        checks[
            "hybrid_evidence_mix"
        ] = (
            {
                "3GPP",
                "TCC"
            }
            .issubset(
                observed_sources
            )
        )


        checks[
            "hybrid_excludes_tcc_3gpp_tsg"
        ] = (
            all(
                str(
                    item.get(
                        "collection",
                        ""
                    )
                )
                !=
                "3GPP-TSG"

                for item
                in evidence
            )
        )

    else:

        checks[
            "hybrid_evidence_mix"
        ] = True

        checks[
            "hybrid_excludes_tcc_3gpp_tsg"
        ] = True


    # --------------------------------------------------------
    # EXPECTED 3GPP SPEC REPRESENTATION
    # --------------------------------------------------------

    expected_3gpp_identifiers = set(
        case.get(
            "expected_3gpp_identifiers",
            set()
        )
    )


    observed_3gpp_identifiers = {
        str(
            item.get(
                "identifier",
                ""
            )
        ).strip()

        for item
        in evidence

        if (
            normalize_source_family(
                item.get(
                    "source_family",
                    ""
                )
            )
            ==
            "3GPP"
        )
    }


    if expected_3gpp_identifiers:

        checks[
            "expected_3gpp_spec_evidence"
        ] = bool(
            expected_3gpp_identifiers
            &
            observed_3gpp_identifiers
        )

    else:

        checks[
            "expected_3gpp_spec_evidence"
        ] = True


    # --------------------------------------------------------
    # TCC COLLECTION ROUTING
    # --------------------------------------------------------

    trace = (
        payload.get(
            "trace",
            {}
        )
        or
        {}
    )


    routed_tcc_collections = clean_string_set(
        trace.get(
            "tcc_collections",
            []
        )
    )


    expected_tcc_collections = set(
        case[
            "expected_tcc_collections"
        ]
    )


    if expected_tcc_collections:

        checks[
            "tcc_collection_selection"
        ] = (
            expected_tcc_collections
            .issubset(
                routed_tcc_collections
            )
        )

    else:

        checks[
            "tcc_collection_selection"
        ] = True


    # --------------------------------------------------------
    # TRACE SCHEMA / CONSISTENCY
    # --------------------------------------------------------

    missing_trace_fields = (
        REQUIRED_MCP_TRACE_FIELDS
        -
        set(
            trace.keys()
        )
    )


    checks[
        "trace_schema"
    ] = (
        len(
            missing_trace_fields
        )
        ==
        0
    )


    checks[
        "trace_architecture"
    ] = (
        str(
            trace.get(
                "architecture",
                ""
            )
        )
        ==
        "Version B"

        and

        str(
            trace.get(
                "retrieval_architecture",
                ""
            )
        )
        ==
        "Persistent DuckDB BM25/FTS"
    )


    checks[
        "trace_mcp_identity"
    ] = (
        str(
            trace.get(
                "mcp_server",
                ""
            )
        )
        ==
        "Telecom Knowledge Service — Version B"

        and

        str(
            trace.get(
                "mcp_tool",
                ""
            )
        )
        ==
        "search_telecom_knowledge"
    )


    checks[
        "trace_query"
    ] = (
        str(
            trace.get(
                "query",
                ""
            )
        )
        .strip()
        ==
        case[
            "question"
        ]
        .strip()
    )


    checks[
        "trace_route"
    ] = (
        str(
            trace.get(
                "route",
                ""
            )
        )
        .strip()
        .lower()
        ==
        actual_route
    )


    checks[
        "trace_result_count"
    ] = (
        safe_int(
            trace.get(
                "result_count",
                0
            )
        )
        ==
        result_count
    )


    selected_shards = list(
        trace.get(
            "selected_shards",
            []
        )
        or
        []
    )


    checks[
        "trace_shards"
    ] = (
        len(
            selected_shards
        )
        >
        0

        and

        len(
            selected_shards
        )
        ==
        safe_int(
            trace.get(
                "selected_shard_count",
                0
            )
        )
    )


    checks[
        "trace_candidate_terms"
    ] = (
        len(
            trace.get(
                "candidate_terms",
                []
            )
            or
            []
        )
        >
        0
    )


    checks[
        "trace_no_shard_errors"
    ] = (
        len(
            trace.get(
                "shard_errors",
                []
            )
            or
            []
        )
        ==
        0
    )


    checks[
        "trace_retrieval_latency"
    ] = (
        safe_float(
            trace.get(
                "retrieval_time_s",
                0.0
            )
        )
        >
        0
    )


    checks[
        "trace_tool_latency"
    ] = (
        safe_float(
            trace.get(
                "tool_elapsed_s",
                0.0
            )
        )
        >=
        safe_float(
            trace.get(
                "retrieval_time_s",
                0.0
            )
        )
    )


    # --------------------------------------------------------
    # DETAILS
    # --------------------------------------------------------

    details = {

        "actual_route":
            actual_route,

        "observed_sources":
            sorted(
                observed_sources
            ),

        "routed_sources":
            sorted(
                routed_sources
            ),

        "source_distribution":
            actual_source_distribution,

        "observed_3gpp_identifiers":
            sorted(
                observed_3gpp_identifiers
            ),

        "routed_tcc_collections":
            sorted(
                routed_tcc_collections
            ),

        "selected_shard_count":
            safe_int(
                trace.get(
                    "selected_shard_count",
                    0
                )
            ),

        "candidate_terms":
            list(
                trace.get(
                    "candidate_terms",
                    []
                )
                or
                []
            ),

        "result_count":
            result_count,

        "retrieval_time_s":
            safe_float(
                trace.get(
                    "retrieval_time_s",
                    0.0
                )
            ),

        "tool_elapsed_s":
            safe_float(
                trace.get(
                    "tool_elapsed_s",
                    0.0
                )
            ),

        "missing_payload_fields":
            sorted(
                missing_payload_fields
            ),

        "missing_trace_fields":
            sorted(
                missing_trace_fields
            ),

        "missing_evidence_fields":
            missing_evidence_fields,

        "empty_evidence_items":
            empty_evidence_items,

        "oversized_evidence_items":
            oversized_evidence_items,

        "bad_evidence_char_counts":
            bad_evidence_char_counts
    }


    return (
        checks,
        details
    )


# ============================================================
# RUN REAL FASTMCP CLIENT VALIDATION
# ============================================================

async def validate_version_b_mcp_tool():
    """
    Invoke the real in-memory FastMCP server/client boundary
    across 3GPP, TCC and Hybrid routes.
    """

    print("=" * 100)

    print(
        "VERSION B — THREE-ROUTE MCP TOOL VALIDATION"
    )

    print("=" * 100)


    print(
        "MCP Server : "
        "Telecom Knowledge Service — Version B"
    )

    print(
        "MCP Tool   : "
        "search_telecom_knowledge(query, top_k=5)"
    )

    print(
        "Profile    : "
        "NOT EXPOSED"
    )


    mcp_client = Client(
        mcp
    )


    validation_results = []


    async with mcp_client:


        for case in MCP_VALIDATION_CASES:

            print(
                "\n" + "=" * 100
            )

            print(
                f"{case['id']} — "
                f"{case['name']}"
            )

            print("=" * 100)


            print(
                f"Question       : "
                f"{case['question']}"
            )

            print(
                f"Expected Route : "
                f"{case['expected_route'].upper()}"
            )


            # ------------------------------------------------
            # REAL MCP TOOL CALL
            # ------------------------------------------------

            call_start = (
                time.perf_counter()
            )


            result = await mcp_client.call_tool(
                "search_telecom_knowledge",
                {
                    "query":
                        case[
                            "question"
                        ],

                    "top_k":
                        TOP_K_RESULTS
                }
            )


            roundtrip_s = (
                time.perf_counter()
                -
                call_start
            )


            # ------------------------------------------------
            # STRUCTURED RESULT
            # ------------------------------------------------

            if result.data is None:

                raise RuntimeError(
                    f"{case['id']} MCP tool call did not "
                    "return a structured data payload."
                )


            payload = (
                result.data
            )


            # ------------------------------------------------
            # VALIDATE SERIALIZED PAYLOAD
            # ------------------------------------------------

            (
                checks,
                details
            ) = validate_mcp_payload(
                case=case,
                payload=payload
            )


            passed = all(
                checks.values()
            )


            retrieval_time_s = (
                details[
                    "retrieval_time_s"
                ]
            )


            tool_elapsed_s = (
                details[
                    "tool_elapsed_s"
                ]
            )


            mcp_overhead_s = max(
                0.0,
                roundtrip_s
                -
                retrieval_time_s
            )


            validation_results.append({

                "case":
                    case,

                "payload":
                    payload,

                "checks":
                    checks,

                "details":
                    details,

                "mcp_roundtrip_s":
                    float(
                        roundtrip_s
                    ),

                "mcp_overhead_s":
                    float(
                        mcp_overhead_s
                    ),

                "passed":
                    passed
            })


            # ------------------------------------------------
            # SUMMARY
            # ------------------------------------------------

            print(
                f"\nActual Route   : "
                f"{details['actual_route'].upper()}"
            )

            print(
                f"Sources        : "
                f"{details['observed_sources']}"
            )

            print(
                f"Distribution   : "
                f"{details['source_distribution']}"
            )

            print(
                f"TCC Collections: "
                f"{details['routed_tcc_collections']}"
            )

            if details[
                "observed_3gpp_identifiers"
            ]:

                print(
                    f"3GPP Specs     : "
                    f"{details['observed_3gpp_identifiers']}"
                )


            print(
                f"BM25 Shards    : "
                f"{details['selected_shard_count']}"
            )

            print(
                f"Candidate Terms: "
                f"{details['candidate_terms']}"
            )

            print(
                f"Evidence Items : "
                f"{details['result_count']}"
            )

            print(
                f"Retrieval      : "
                f"{retrieval_time_s:.3f} sec"
            )

            print(
                f"Tool Elapsed   : "
                f"{tool_elapsed_s:.3f} sec"
            )

            print(
                f"MCP Roundtrip  : "
                f"{roundtrip_s:.3f} sec"
            )

            print(
                f"MCP Overhead   : "
                f"{mcp_overhead_s:.3f} sec"
            )


            # ------------------------------------------------
            # CHECK TABLE
            # ------------------------------------------------

            print(
                "\nMCP Validation Checks"
            )

            print(
                "-" * 100
            )


            for check_name, ok in (
                checks.items()
            ):

                print(
                    f"{check_name:<36} : "
                    f"{'PASS' if ok else 'FAIL'}"
                )


            # ------------------------------------------------
            # EVIDENCE TABLE
            # ------------------------------------------------

            evidence = (
                payload.get(
                    "evidence",
                    []
                )
                or
                []
            )


            evidence_df = pd.DataFrame([
                {
                    "rank":
                        item[
                            "rank"
                        ],

                    "source_family":
                        item[
                            "source_family"
                        ],

                    "collection":
                        item[
                            "collection"
                        ],

                    "identifier":
                        item[
                            "identifier"
                        ],

                    "title":
                        item[
                            "title"
                        ],

                    "source_shard":
                        item[
                            "source_shard"
                        ],

                    "matched_terms":
                        item[
                            "matched_terms"
                        ],

                    "bm25_score":
                        item[
                            "bm25_score"
                        ],

                    "evidence_chars":
                        item[
                            "evidence_chars"
                        ]
                }

                for item
                in evidence
            ])


            print(
                "\nMCP Evidence Payload"
            )

            print(
                "-" * 100
            )


            display(
                evidence_df.round({
                    "bm25_score":
                        4
                })
            )


            # ------------------------------------------------
            # TOP EVIDENCE PREVIEW
            # ------------------------------------------------

            if evidence:

                top_item = (
                    evidence[
                        0
                    ]
                )


                print(
                    "\nTop-1 MCP Evidence"
                )

                print(
                    "-" * 100
                )


                print(
                    f"Source Family : "
                    f"{top_item['source_family']}"
                )

                print(
                    f"Collection    : "
                    f"{top_item['collection']}"
                )

                print(
                    f"Identifier    : "
                    f"{top_item['identifier']}"
                )

                print(
                    f"Title         : "
                    f"{top_item['title']}"
                )

                print(
                    f"Evidence Chars: "
                    f"{top_item['evidence_chars']:,}"
                )

                print(
                    "\nEvidence Preview:"
                )

                print(
                    str(
                        top_item[
                            "evidence"
                        ]
                    )[
                        :1200
                    ]
                )


            print(
                "\n" + "-" * 100
            )

            print(
                f"{case['id']} STATUS: "
                f"{'PASS' if passed else 'REVIEW'}"
            )

            print(
                "-" * 100
            )


    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    summary_rows = []


    for item in validation_results:

        summary_rows.append({

            "pilot":
                item[
                    "case"
                ][
                    "id"
                ],

            "route":
                item[
                    "details"
                ][
                    "actual_route"
                ],

            "source_families":
                ",".join(
                    item[
                        "details"
                    ][
                        "observed_sources"
                    ]
                ),

            "results":
                item[
                    "details"
                ][
                    "result_count"
                ],

            "retrieval_time_s":
                round(
                    item[
                        "details"
                    ][
                        "retrieval_time_s"
                    ],
                    3
                ),

            "mcp_roundtrip_s":
                round(
                    item[
                        "mcp_roundtrip_s"
                    ],
                    3
                ),

            "mcp_overhead_s":
                round(
                    item[
                        "mcp_overhead_s"
                    ],
                    3
                ),

            "status":
                (
                    "PASS"
                    if item[
                        "passed"
                    ]
                    else
                    "REVIEW"
                )
        })


    summary_df = pd.DataFrame(
        summary_rows
    )


    print(
        "\n" + "=" * 100
    )

    print(
        "VERSION B — THREE-ROUTE MCP VALIDATION SUMMARY"
    )

    print("=" * 100)


    display(
        summary_df
    )


    all_passed = all(
        item[
            "passed"
        ]
        for item
        in validation_results
    )


    print(
        "\n" + "=" * 100
    )


    if all_passed:

        print(
            "✓ VERSION B THREE-ROUTE MCP TOOL VALIDATION PASSED"
        )

        print(
            "✓ Real FastMCP Client invoked the unified tool."
        )

        print(
            "✓ No LLM-facing profile argument is required."
        )

        print(
            "✓ 3GPP, TCC and Hybrid routes survived MCP serialization."
        )

        print(
            "✓ Version B Top-K and source limits are enforced."
        )

        print(
            f"✓ Evidence excerpts respect the "
            f"{MCP_EXCERPT_CHARS:,}-character control."
        )

        print(
            "✓ Complete bounded evidence is preserved."
        )

        print(
            "✓ Complete persistent BM25 retrieval traces are preserved."
        )

        print(
            "✓ Hybrid evidence retains both 3GPP and TCC source families."
        )

        print(
            "✓ MCP boundary is ready for LLM orchestration."
        )

    else:

        failed_cases = [
            item[
                "case"
            ][
                "id"
            ]

            for item
            in validation_results

            if not item[
                "passed"
            ]
        ]


        print(
            "⚠ VERSION B MCP TOOL VALIDATION REQUIRES REVIEW"
        )

        print(
            f"Failed cases: {failed_cases}"
        )

        print(
            "Inspect the failed MCP-boundary checks before "
            "continuing to LLM orchestration."
        )


    print("=" * 100)


    if not all_passed:

        raise RuntimeError(
            "Version B three-route MCP tool validation "
            "did not fully pass."
        )


    return (
        validation_results,
        summary_df
    )


# ============================================================
# EXECUTE MCP VALIDATION
# ============================================================
#
# The validation client is intentionally scoped inside the
# async context above. Cell 11 will create its own live FastMCP
# client for end-to-end LLM orchestration.
# ============================================================

(
    MCP_VALIDATION_RESULTS,
    MCP_VALIDATION_SUMMARY
) = await validate_version_b_mcp_tool()


VERSION B — THREE-ROUTE MCP TOOL VALIDATION
MCP Server : Telecom Knowledge Service — Version B
MCP Tool   : search_telecom_knowledge(query, top_k=5)
Profile    : NOT EXPOSED

P1 — 3GPP-only
Question       : Explain the role of the AMF in registration and mobility management procedures in a 5G Standalone network.
Expected Route : 3GPP


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Actual Route   : 3GPP
Sources        : ['3GPP']
Distribution   : {'3GPP': 5}
TCC Collections: ['3GPP-TSG']
3GPP Specs     : ['23.501', '23.502']
BM25 Shards    : 2
Candidate Terms: ['5g', 'standalone', 'amf', 'registration', 'mobility', 'management', 'role', 'procedures']
Evidence Items : 5
Retrieval      : 2.881 sec
Tool Elapsed   : 2.884 sec
MCP Roundtrip  : 3.993 sec
MCP Overhead   : 1.112 sec

MCP Validation Checks
----------------------------------------------------------------------------------------------------
payload_is_dict                      : PASS
payload_schema                       : PASS
no_profile_field                     : PASS
query_roundtrip                      : PASS
expected_route                       : PASS
result_count                         : PASS
top_k_contract                       : PASS
max_source_limit                     : PASS
evidence_schema                      : PASS
evidence_non_empty                   : PASS
evidence_char_counts               

,rank,source_family,collection,identifier,title,source_shard,matched_terms,bm25_score,evidence_chars
0,1,3GPP,3GPP-Specifications,23.501,3GPP TS 23.501 V18.4.0 (2023-12),3gpp_3gpp_specifications_01,8,8.3903,2508
1,2,3GPP,3GPP-Specifications,23.502,3GPP TS 23.502 V20.0.0 (2025-12),3gpp_3gpp_specifications_00,8,8.3226,2508
2,3,3GPP,3GPP-Specifications,23.501,3GPP TS 23.501 V20.0.0 (2025-12) ---,3gpp_3gpp_specifications_01,8,8.3161,2508
3,4,3GPP,3GPP-Specifications,23.501,3rd Generation Partnership Project; Technical ...,3gpp_3gpp_specifications_01,8,8.2988,2508
4,5,3GPP,3GPP-Specifications,23.502,3GPP TS 23.502 V18.4.0 (2023-12),3gpp_3gpp_specifications_00,8,8.2659,2508


...


Top-1 MCP Evidence
----------------------------------------------------------------------------------------------------
Source Family : TCC
Collection    : IETF-Drafts
Identifier    : draft-hamilton-quic-transport-protocol-1
Title         : 
Evidence Chars: 2,504

Evidence Preview:
### 1 Introduction

QUIC is a multiplexed and secure transport protocol that runs on top of UDP.  QUIC builds on past transport experience and implements mechanisms that make it useful as a modern general-purpose transport protocol.  Using UDP as the substrate, QUIC seeks to be compatible with legacy clients and middleboxes.  QUIC authenticates all of its headers, preventing middleboxes and other third parties from changing them, and encrypts most of its headers, limiting protocol evolution largely to QUIC endpoints only.

This document describes the core QUIC protocol, including the conceptual design, wire format, and mechanisms of the QUIC protocol

Internet-Draft                    QUIC                  

,rank,source_family,collection,identifier,title,source_shard,matched_terms,bm25_score,evidence_chars
0,1,3GPP,3GPP-Specifications,23.501,3GPP TS 23.501 V20.0.0 (2025-12) ---,3gpp_3gpp_specifications_01,5,17.1727,2508
1,2,3GPP,3GPP-Specifications,23.501,3rd Generation Partnership Project; Technical ...,3gpp_3gpp_specifications_01,5,16.8631,2508
2,3,3GPP,3GPP-Specifications,23.501,3GPP TS 23.501 V18.4.0 (2023-12),3gpp_3gpp_specifications_01,5,16.8605,2508
3,4,TCC,IETF-Drafts,draft-friel-tls-atls-5,,tcc_ietf_drafts_00,2,4.1923,2504
4,5,TCC,IETF-Drafts,draft-friel-tls-atls-5,,tcc_ietf_drafts_00,2,4.1798,2508



Top-1 MCP Evidence
----------------------------------------------------------------------------------------------------
Source Family : 3GPP
Collection    : 3GPP-Specifications
Identifier    : 23.501
Title         : 3GPP TS 23.501 V20.0.0 (2025-12) ---
Evidence Chars: 2,508

Evidence Preview:
... 

The functional descriptions of these Network Functions and entities are specified in clause 6.

- Non-3GPP InterWorking Function (N3IWF).
- Trusted Non-3GPP Gateway Function (TNGF).
- Wireline Access Gateway Function (W-AGF).
- Trusted WLAN Interworking Function (TWIF).
- Energy Information Function (EIF).

### 4.2.3 Non-roaming reference architecture

Figure 4.2.3-1 depicts the non-roaming reference architecture. Service-based interfaces are used within the Control Plane.

![Figure 4.2.3-1: Non-Roaming 5G System Architecture diagram showing the Control Plane and User Plane components and their interfaces.](78d5774278a3f4a614f8c0ae485ce8d9_img.jpg)

The diagram illustrates the Non-Roaming 5

,pilot,route,source_families,results,retrieval_time_s,mcp_roundtrip_s,mcp_overhead_s,status
0,P1,3gpp,3GPP,5,2.881,3.993,1.112,PASS
1,P2,tcc,TCC,5,1.855,1.861,0.006,PASS
2,P3,hybrid,"3GPP,TCC",5,2.413,2.419,0.006,PASS



✓ VERSION B THREE-ROUTE MCP TOOL VALIDATION PASSED
✓ Real FastMCP Client invoked the unified tool.
✓ No LLM-facing profile argument is required.
✓ 3GPP, TCC and Hybrid routes survived MCP serialization.
✓ Version B Top-K and source limits are enforced.
✓ Evidence excerpts respect the 2,500-character control.
✓ Complete bounded evidence is preserved.
✓ Complete persistent BM25 retrieval traces are preserved.
✓ Hybrid evidence retains both 3GPP and TCC source families.
✓ MCP boundary is ready for Claude orchestration.


**Observation — Three-Route MCP Validation**

A real FastMCP client successfully exercised the same **3GPP, TCC and Hybrid** paths through the serialization boundary while preserving bounded evidence and full retrieval traces.

***Key Decision:*** Validate the MCP boundary independently before connecting any model.


# **SECTION 4 — DeepSeek V4 + MCP Orchestration**


## **Cell 10 — DeepSeek V4 Setup + Frozen Shared System Prompt**


In [23]:
# ============================================================
# CELL 10 — DeepSeek V4 Flash 0731 + OPENROUTER SETUP
#           + FROZEN SHARED SYSTEM PROMPT
# ============================================================
#
# Purpose:
#
# Configure DeepSeek V4 Flash 0731 through OpenRouter for the
# Version B end-to-end MCP runtime.
#
# Cells 5–9 remain identical across Version B models.
#
# Experimental controls are intentionally aligned with the
# frozen Version A / Version B Benchmark v2 experiments:
#
#     Model               : deepseek/deepseek-v4-flash-0731
#     Provider            : OpenRouter
#     Max output tokens   : 1800
#     Temperature         : API default / not explicitly set
#     Max MCP searches    : 3
#     Top-K retrieval     : 5
#     Max sources/search  : 5
#     Evidence excerpt    : 2500 characters/source
#
# IMPORTANT:
#
# The system prompt below is byte-identical to the prompt used
# for Version A Benchmark v2 and Version B Claude Benchmark v2.
#
# Do not modify its wording during this evaluation.
# ============================================================


import hashlib
import os

from openai import OpenAI


# ============================================================
# OPENROUTER API KEY
# ============================================================

# Prefer an existing environment variable.
OPENROUTER_API_KEY = os.environ.get(
    "OPENROUTER_API_KEY"
)


# Google Colab fallback.
if not OPENROUTER_API_KEY:

    try:

        from google.colab import userdata

        OPENROUTER_API_KEY = userdata.get(
            "OPENROUTER_API_KEY"
        )

    except Exception:

        OPENROUTER_API_KEY = None


if not OPENROUTER_API_KEY:

    raise ValueError(
        "OPENROUTER_API_KEY was not found. "
        "Add it to Google Colab Secrets or the "
        "OPENROUTER_API_KEY environment variable."
    )


os.environ[
    "OPENROUTER_API_KEY"
] = OPENROUTER_API_KEY


# ============================================================
# OPENROUTER / MODEL CONFIGURATION
# ============================================================

OPENROUTER_BASE_URL = (
    "https://openrouter.ai/api/v1"
)


LLM_PROVIDER = (
    "OpenRouter"
)


LLM_MODEL = (
    "deepseek/deepseek-v4-flash-0731"
)


LLM_DISPLAY_NAME = (
    "DeepSeek V4 Flash 0731"
)


LLM_MAX_TOKENS = 1800


# Temperature intentionally remains unset.
#
# Cell 11 must omit the temperature parameter when this is None.
LLM_TEMPERATURE = None


# ============================================================
# EXPERIMENTAL CONTROL VALIDATION
# ============================================================

if MAX_MCP_SEARCHES != 3:

    raise RuntimeError(
        "Version B experimental control violation: "
        "MAX_MCP_SEARCHES must remain 3."
    )


if MAX_RETRIEVED_SOURCES != 5:

    raise RuntimeError(
        "Version B experimental control violation: "
        "MAX_RETRIEVED_SOURCES must remain 5."
    )


if TOP_K_RESULTS != 5:

    raise RuntimeError(
        "Version B experimental control violation: "
        "TOP_K_RESULTS must remain 5."
    )


if MCP_EXCERPT_CHARS != 2500:

    raise RuntimeError(
        "Version B experimental control violation: "
        "MCP_EXCERPT_CHARS must remain 2500."
    )


if LLM_MAX_TOKENS != 1800:

    raise RuntimeError(
        "Version B experimental control violation: "
        "LLM_MAX_TOKENS must remain 1800."
    )


# ============================================================
# OPENAI-COMPATIBLE OPENROUTER CLIENT
# ============================================================

openrouter = OpenAI(

    base_url=
        OPENROUTER_BASE_URL,

    api_key=
        OPENROUTER_API_KEY
)


# Provider-neutral aliases used by the Version B OpenRouter
# orchestration cell.
llm_client = openrouter


# ============================================================
# FROZEN SHARED VERSION A/B SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are an expert telecommunications network engineer with access to
a telecom knowledge search tool through MCP.

Use retrieved telecom documentation as the authoritative basis for
technical answers.

Instructions:

1. Use the MCP knowledge tool when technical documentation is required.
   Use ONE search by default, a SECOND only if important evidence is
   missing, and a THIRD only as an exceptional fallback. Never exceed
   3 searches.

2. Reason across the retrieved evidence. You may connect facts, compare
   functions, explain relationships, infer technical implications, and
   synthesize information from multiple retrieved documents.

3. Do not invent facts. Any reasoning or inference must remain consistent
   with and supported by the retrieved evidence. If the evidence does not
   support a required technical point, do not fill the gap with assumptions
   or unsupported knowledge.

4. Prefer authoritative standards and technical documentation when
   evaluating conflicting or overlapping evidence. Use no more than
   5 retrieved sources for the final answer.

5. Answer every technical element requested by the user. Keep the response
   precise, practical, and suitable for a telecommunications engineer.

6. Target approximately 300–500 words unless the question clearly requires
   more detail. Prefer concise sections, bullets, or numbered steps where
   they improve clarity.

7. Do not discuss sources, citations, retrieval mechanics, BM25, DuckDB,
   shards, candidate terms, search profiles, or other internal implementation
   details unless the user explicitly asks.

8. If the retrieved documentation is insufficient to answer the question,
   state:
   "The requested details are not available in the documentation."
""".strip()


# Backward-compatible alias.
VERSION_B_SYSTEM_PROMPT = SYSTEM_PROMPT


# ============================================================
# FROZEN PROMPT FINGERPRINT
# ============================================================

EXPECTED_SYSTEM_PROMPT_SHA256 = (
    "854374b0ea09e63dc8ddde34d722788231c2be860f4b8ac1a709e907966baf09"
)


SYSTEM_PROMPT_SHA256 = (
    hashlib
    .sha256(
        SYSTEM_PROMPT.encode(
            "utf-8"
        )
    )
    .hexdigest()
)


if (
    SYSTEM_PROMPT_SHA256
    !=
    EXPECTED_SYSTEM_PROMPT_SHA256
):

    raise RuntimeError(
        "Frozen Version A/B system prompt hash mismatch. "
        f"Expected {EXPECTED_SYSTEM_PROMPT_SHA256}, "
        f"found {SYSTEM_PROMPT_SHA256}."
    )


# ============================================================
# SYSTEM PROMPT SAFETY CHECKS
# ============================================================

_REQUIRED_PROMPT_FRAGMENTS = [

    "Never exceed\n   3 searches.",

    "Use no more than\n   5 retrieved sources",

    "Target approximately 300–500 words",

    (
        "The requested details are not "
        "available in the documentation."
    )
]


for fragment in _REQUIRED_PROMPT_FRAGMENTS:

    if fragment not in SYSTEM_PROMPT:

        raise RuntimeError(
            "Frozen shared system prompt validation "
            f"failed for fragment: {fragment}"
        )


# ============================================================
# PROVIDER / MODEL SAFETY CHECKS
# ============================================================

if LLM_PROVIDER != "OpenRouter":

    raise RuntimeError(
        "DeepSeek V4 Flash 0731 experiment requires OpenRouter."
    )


if (
    LLM_MODEL
    !=
    "deepseek/deepseek-v4-flash-0731"
):

    raise RuntimeError(
        "Unexpected model configured for "
        "the DeepSeek V4 Flash experiment."
    )


# ============================================================
# PROMPT OBSERVABILITY
# ============================================================

SYSTEM_PROMPT_WORDS = len(
    SYSTEM_PROMPT.split()
)


SYSTEM_PROMPT_CHARS = len(
    SYSTEM_PROMPT
)


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 90)

print(
    "VERSION B — DEEPSEEK V4 FLASH 0731 / OPENROUTER "
    "RUNTIME CONFIGURATION"
)

print("=" * 90)


print(
    f"LLM Provider          : "
    f"{LLM_PROVIDER}"
)

print(
    f"LLM Model             : "
    f"{LLM_MODEL}"
)

print(
    f"Model Display Name    : "
    f"{LLM_DISPLAY_NAME}"
)

print(
    f"OpenRouter Base URL   : "
    f"{OPENROUTER_BASE_URL}"
)

print(
    f"Max Output Tokens     : "
    f"{LLM_MAX_TOKENS}"
)

print(
    "Temperature           : "
    "API DEFAULT"
)

print(
    f"Max MCP Searches      : "
    f"{MAX_MCP_SEARCHES}"
)

print(
    f"Top-K Retrieval       : "
    f"{TOP_K_RESULTS}"
)

print(
    f"Max Sources / Search  : "
    f"{MAX_RETRIEVED_SOURCES}"
)

print(
    f"Evidence Excerpt      : "
    f"{MCP_EXCERPT_CHARS:,} chars/source"
)

print(
    "MCP Tool              : "
    "search_telecom_knowledge(query, top_k=5)"
)

print(
    "LLM Profile Control   : "
    "NONE"
)

print(
    f"System Prompt SHA256  : "
    f"{SYSTEM_PROMPT_SHA256}"
)

print(
    f"System Prompt Words   : "
    f"{SYSTEM_PROMPT_WORDS:,}"
)

print(
    f"System Prompt Chars   : "
    f"{SYSTEM_PROMPT_CHARS:,}"
)


print(
    "\n" + "=" * 90
)

print(
    "✓ OpenRouter API key loaded."
)

print(
    "✓ OpenAI-compatible OpenRouter client initialized."
)

print(
    "✓ DeepSeek V4 Flash 0731 configured."
)

print(
    "✓ Shared Version A/B system prompt loaded byte-identically."
)

print(
    "✓ System prompt SHA-256 matches Benchmark v2."
)

print(
    "✓ Maximum MCP search budget fixed at 3."
)

print(
    "✓ Top-K and evidence controls aligned with Claude Version B."
)

print(
    "✓ No LLM-facing retrieval profile exists."
)

print(
    "✓ No OpenRouter API request executed in this cell."
)

print(
    "✓ Ready for Cell 11 — OpenRouter End-to-End MCP Orchestration."
)

print("=" * 90)


VERSION B — DEEPSEEK V4 FLASH 0731 / OPENROUTER RUNTIME CONFIGURATION
LLM Provider          : OpenRouter
LLM Model             : deepseek/deepseek-v4-flash-0731
Model Display Name    : DeepSeek V4 Flash 0731
OpenRouter Base URL   : https://openrouter.ai/api/v1
Max Output Tokens     : 1800
Temperature           : API DEFAULT
Max MCP Searches      : 3
Top-K Retrieval       : 5
Max Sources / Search  : 5
Evidence Excerpt      : 2,500 chars/source
MCP Tool              : search_telecom_knowledge(query, top_k=5)
LLM Profile Control   : NONE
System Prompt SHA256  : 854374b0ea09e63dc8ddde34d722788231c2be860f4b8ac1a709e907966baf09
System Prompt Words   : 241
System Prompt Chars   : 1,752

✓ OpenRouter API key loaded.
✓ OpenAI-compatible OpenRouter client initialized.
✓ DeepSeek V4 Flash 0731 configured.
✓ Shared Version A/B system prompt loaded byte-identically.
✓ System prompt SHA-256 matches Benchmark v2.
✓ Maximum MCP search budget fixed at 3.
✓ Top-K and evidence controls aligned with Claud

**Observation — DeepSeek V4 Runtime Setup**

**DeepSeek V4 Flash 0731 (`deepseek/deepseek-v4-flash-0731`)** is configured through OpenRouter with the frozen Benchmark v2 prompt, API-default temperature, 1,800-token cap, maximum three MCP searches, Top-K 5 and bounded evidence excerpts.

***Key Decision:*** Keep this configuration frozen as the untuned baseline; any later reasoning/thinking or token/search configuration optimization must be evaluated separately.


## **Cell 11 — End-to-End MCP Orchestration**

In [24]:
# ============================================================
# CELL 11 — END-TO-END OPENROUTER LLM + MCP ORCHESTRATION
# ============================================================
#
# Purpose:
#
# Connect the configured OpenRouter LLM (Gemma 4 or DeepSeek)
# to the validated Version B MCP knowledge service.
#
# Architecture:
#
#     User Question
#          ↓
#     OpenRouter LLM
#          ↓
#      tool_calls
#          ↓
#   Input Validation
#          ↓
#       FastMCP
#          ↓
# search_telecom_knowledge(query, top_k)
#          ↓
# Persistent Version B BM25 Retrieval
#          ↓
#      role="tool"
#          ↓
#     OpenRouter LLM
#          ↓
#      Final Answer
#
# Experimental controls:
#
#     - ONE MCP knowledge tool
#     - NO LLM-facing profile argument
#     - Maximum 3 EXECUTED MCP searches
#     - Top-K maximum 5
#     - Frozen shared Version A/B system prompt
#     - No explicit temperature
#     - Full token / latency / routing / evidence trace
#
# This cell is SHARED by:
#
#     - Gemma 4 26B A4B IT
#     - DeepSeek V4 Flash 0731
#
# Cell 10 supplies the provider/model-specific configuration.
# ============================================================


import json
import re
import time

from fastmcp import Client


VERSION_B_ARCHITECTURE = (
    "Version B"
)


VERSION_B_RETRIEVAL_ARCHITECTURE = (
    "Persistent DuckDB BM25/FTS"
)


# ============================================================
# GENERIC NORMALIZATION HELPERS
# ============================================================

def to_plain_dict(value):
    """
    Convert common dictionary-like objects into a plain dict.

    Handles:
        - dict
        - Pydantic v2 model_dump()
        - Pydantic v1 dict()
        - JSON strings
    """

    if value is None:
        return {}

    if isinstance(value, dict):
        return dict(value)

    model_dump = getattr(
        value,
        "model_dump",
        None
    )

    if callable(model_dump):

        try:

            dumped = model_dump()

            if isinstance(dumped, dict):
                return dict(dumped)

        except Exception:
            pass

    dict_method = getattr(
        value,
        "dict",
        None
    )

    if callable(dict_method):

        try:

            dumped = dict_method()

            if isinstance(dumped, dict):
                return dict(dumped)

        except Exception:
            pass

    if isinstance(value, str):

        try:

            parsed = json.loads(value)

            if isinstance(parsed, dict):
                return parsed

        except Exception:
            pass

    return {}


def safe_int_value(
    value,
    default=0
):
    """
    Safely convert a scalar to int.
    """

    if isinstance(value, bool):
        return int(default)

    if value is None:
        return int(default)

    try:
        return int(value)
    except Exception:
        pass

    try:
        return int(
            float(
                str(value).strip()
            )
        )
    except Exception:
        return int(default)


def safe_float_value(
    value,
    default=0.0
):
    """
    Safely convert a scalar to float.
    """

    try:

        if value is None:
            return float(default)

        return float(value)

    except Exception:

        return float(default)


def safe_list_value(value):
    """
    Safely normalize a list-like object.
    """

    if value is None:
        return []

    if isinstance(
        value,
        list
    ):
        return list(value)

    if isinstance(
        value,
        tuple
    ):
        return list(value)

    if isinstance(
        value,
        set
    ):
        return list(value)

    return [value]


# ============================================================
# FASTMCP INPUT SCHEMA NORMALIZATION
# ============================================================

def normalize_mcp_input_schema(schema):
    """
    Convert FastMCP input_schema to a plain JSON Schema
    suitable for OpenAI-compatible function calling.
    """

    schema_dict = to_plain_dict(
        schema
    )

    if not schema_dict:

        raise RuntimeError(
            "FastMCP tool input_schema could not "
            "be converted to a dictionary."
        )

    schema_type = schema_dict.get(
        "type"
    )

    if schema_type is None:

        schema_dict[
            "type"
        ] = "object"

    elif schema_type != "object":

        raise RuntimeError(
            "Expected MCP tool input_schema type "
            f"'object', received: {schema_type}"
        )

    properties = schema_dict.get(
        "properties"
    )

    if properties is None:

        schema_dict[
            "properties"
        ] = {}

    elif not isinstance(
        properties,
        dict
    ):

        raise RuntimeError(
            "MCP input_schema.properties "
            "must be a dictionary."
        )

    required = schema_dict.get(
        "required"
    )

    if (
        required is not None
        and
        not isinstance(
            required,
            list
        )
    ):

        schema_dict[
            "required"
        ] = list(
            required
        )

    return schema_dict


# ============================================================
# FASTMCP → OPENAI-COMPATIBLE TOOL CONVERSION
# ============================================================

def mcp_tool_to_openrouter(tool):
    """
    Convert a FastMCP tool into the OpenAI-compatible
    function-tool schema expected by OpenRouter.
    """

    tool_name = getattr(
        tool,
        "name",
        None
    )

    tool_description = (
        getattr(
            tool,
            "description",
            ""
        )
        or ""
    )

    input_schema = getattr(
        tool,
        "input_schema",
        None
    )

    if not tool_name:

        raise RuntimeError(
            "MCP tool is missing a name."
        )

    if input_schema is None:

        raise RuntimeError(
            f"MCP tool '{tool_name}' "
            "is missing input_schema."
        )

    normalized_schema = (
        normalize_mcp_input_schema(
            input_schema
        )
    )

    return {

        "type":
            "function",

        "function": {

            "name":
                str(
                    tool_name
                ),

            "description":
                str(
                    tool_description
                ),

            "parameters":
                normalized_schema
        }
    }


# ============================================================
# DISCOVER OPENROUTER-ACCESSIBLE MCP TOOL
# ============================================================

async with Client(
    mcp
) as _tool_discovery_client:

    _available_mcp_tools = (
        await _tool_discovery_client.list_tools()
    )


_version_a_search_tools = [

    tool

    for tool
    in _available_mcp_tools

    if (
        getattr(
            tool,
            "name",
            ""
        )
        ==
        "search_telecom_knowledge"
    )
]


if len(
    _version_a_search_tools
) != 1:

    raise RuntimeError(
        "Expected exactly one "
        "search_telecom_knowledge MCP tool. "
        f"Found {len(_version_a_search_tools)}."
    )


OPENROUTER_MCP_TOOLS = [

    mcp_tool_to_openrouter(
        _version_a_search_tools[
            0
        ]
    )
]


# ============================================================
# TOOL INPUT NORMALIZATION
# ============================================================

def normalize_telecom_tool_input(
    raw_input
):
    """
    Validate and normalize LLM-generated arguments before
    forwarding them to FastMCP.

    Allowed arguments:

        query
        top_k
    """

    tool_input = to_plain_dict(
        raw_input
    )

    # --------------------------------------------------------
    # QUERY
    # --------------------------------------------------------

    raw_query = tool_input.get(
        "query"
    )

    if raw_query is None:

        return (
            None,
            "Tool input is missing required field 'query'."
        )

    if not isinstance(
        raw_query,
        str
    ):

        try:

            raw_query = str(
                raw_query
            )

        except Exception:

            return (
                None,
                "Tool input 'query' could not be converted "
                "to text."
            )

    query = re.sub(
        r"\s+",
        " ",
        raw_query
    ).strip()

    if not query:

        return (
            None,
            "Tool input 'query' must contain non-empty text."
        )

    # --------------------------------------------------------
    # TOP-K
    # --------------------------------------------------------

    raw_top_k = tool_input.get(
        "top_k",
        TOP_K_RESULTS
    )

    top_k = safe_int_value(
        raw_top_k,
        default=
            TOP_K_RESULTS
    )

    top_k = max(
        1,
        min(
            top_k,
            TOP_K_RESULTS,
            MAX_RETRIEVED_SOURCES
        )
    )

    # --------------------------------------------------------
    # RETURN ONLY SUPPORTED ARGUMENTS
    # --------------------------------------------------------

    normalized_input = {

        "query":
            query,

        "top_k":
            top_k
    }

    return (
        normalized_input,
        None
    )


# ============================================================
# OPENROUTER RESPONSE HELPERS
# ============================================================

def get_openrouter_message(
    response
):
    """
    Safely extract the first assistant message.
    """

    choices = getattr(
        response,
        "choices",
        []
    ) or []

    if not choices:

        raise RuntimeError(
            "OpenRouter returned no completion choices."
        )

    message = getattr(
        choices[0],
        "message",
        None
    )

    if message is None:

        raise RuntimeError(
            "OpenRouter response contained no assistant message."
        )

    return message


def extract_openrouter_text(
    response
):
    """
    Extract assistant text safely.
    """

    message = get_openrouter_message(
        response
    )

    content = getattr(
        message,
        "content",
        ""
    )

    if content is None:
        return ""

    if isinstance(
        content,
        str
    ):
        return content.strip()

    # Defensive support for structured content.
    if isinstance(
        content,
        list
    ):

        text_parts = []

        for item in content:

            item_dict = to_plain_dict(
                item
            )

            text_value = (
                item_dict.get(
                    "text"
                )
                or
                getattr(
                    item,
                    "text",
                    ""
                )
            )

            if text_value:

                text_parts.append(
                    str(
                        text_value
                    )
                )

        return "\n".join(
            text_parts
        ).strip()

    return str(
        content
    ).strip()


def extract_openrouter_tool_calls(
    response
):
    """
    Extract OpenAI-compatible tool_calls.
    """

    message = get_openrouter_message(
        response
    )

    tool_calls = getattr(
        message,
        "tool_calls",
        None
    )

    if not tool_calls:
        return []

    return list(
        tool_calls
    )


def extract_usage(
    response
):
    """
    Extract OpenAI/OpenRouter token usage.

    Mapped to the same input/output labels used by
    the previous model experiments.
    """

    usage = getattr(
        response,
        "usage",
        None
    )

    if usage is None:

        return {

            "input_tokens":
                0,

            "output_tokens":
                0
        }

    return {

        "input_tokens":
            safe_int_value(
                getattr(
                    usage,
                    "prompt_tokens",
                    0
                ),
                0
            ),

        "output_tokens":
            safe_int_value(
                getattr(
                    usage,
                    "completion_tokens",
                    0
                ),
                0
            )
    }


def get_raw_finish_reason(
    response
):
    """
    Return OpenAI-compatible finish_reason.
    """

    choices = getattr(
        response,
        "choices",
        []
    ) or []

    if not choices:
        return None

    return getattr(
        choices[0],
        "finish_reason",
        None
    )


def normalize_finish_reason(
    finish_reason
):
    """
    Normalize OpenAI/OpenRouter finish reasons into the
    labels used by the existing Version A/B evaluation.

        stop       → end_turn
        length     → max_tokens
        tool_calls → tool_use

    This preserves comparison compatibility.
    """

    mapping = {

        "stop":
            "end_turn",

        "length":
            "max_tokens",

        "tool_calls":
            "tool_use"
    }

    if finish_reason is None:
        return None

    return mapping.get(
        str(
            finish_reason
        ),
        str(
            finish_reason
        )
    )


def serialize_openrouter_assistant_message(
    response
):
    """
    Preserve an OpenRouter assistant response in conversation
    history, including all tool calls.
    """

    message = get_openrouter_message(
        response
    )

    serialized = {

        "role":
            "assistant"
    }

    content = getattr(
        message,
        "content",
        None
    )

    if content is not None:

        serialized[
            "content"
        ] = content

    tool_calls = getattr(
        message,
        "tool_calls",
        None
    )

    if tool_calls:

        serialized[
            "tool_calls"
        ] = []

        for tool_call in tool_calls:

            function = getattr(
                tool_call,
                "function",
                None
            )

            serialized[
                "tool_calls"
            ].append({

                "id":
                    getattr(
                        tool_call,
                        "id",
                        None
                    ),

                "type":
                    "function",

                "function": {

                    "name":
                        getattr(
                            function,
                            "name",
                            ""
                        ),

                    "arguments":
                        getattr(
                            function,
                            "arguments",
                            "{}"
                        )
                }
            })

    return serialized


# ============================================================
# MCP RESPONSE EXTRACTION
# ============================================================

def extract_mcp_payload(
    tool_result
):
    """
    Extract structured dictionary from FastMCP CallToolResult.
    """

    payload = getattr(
        tool_result,
        "data",
        None
    )

    if payload is None:

        payload = getattr(
            tool_result,
            "structured_content",
            None
        )

    payload = to_plain_dict(
        payload
    )

    if not payload:

        raise RuntimeError(
            "MCP telecom knowledge tool did not return "
            "a usable structured response."
        )

    return payload


# ============================================================
# EVIDENCE TRACE EXTRACTION
# ============================================================

def extract_evidence_trace(
    payload
):
    """
    Preserve the complete bounded Version B MCP evidence
    returned to the LLM.

    The retained evidence supports downstream evaluation of:

        - retrieval relevance
        - source authority
        - technical correctness
        - answer groundedness
        - claim-to-evidence support
        - unsupported-claim / hallucination risk

    Only the bounded MCP excerpts are stored.
    Full source documents are never copied into the artifact.
    """

    evidence_items = (
        payload.get(
            "evidence",
            []
        )
    )


    if not isinstance(
        evidence_items,
        list
    ):

        evidence_items = []


    evidence_trace = []


    for raw_item in evidence_items:

        item = to_plain_dict(
            raw_item
        )


        if not item:

            continue


        evidence_text = str(
            item.get(
                "evidence",
                ""
            )
            or
            ""
        )


        evidence_trace.append({

            # ------------------------------------------------
            # RANKING / SCORES
            # ------------------------------------------------

            "rank":
                item.get(
                    "rank"
                ),

            "relevance_score":
                item.get(
                    "relevance_score"
                ),

            "bm25_score":
                item.get(
                    "bm25_score"
                ),

            "matched_terms":
                item.get(
                    "matched_terms"
                ),

            "matched_phrases":
                item.get(
                    "matched_phrases"
                ),

            "proximity_score":
                item.get(
                    "proximity_score"
                ),

            "primary_coverage":
                item.get(
                    "primary_coverage"
                ),

            # ------------------------------------------------
            # SOURCE PROVENANCE
            # ------------------------------------------------

            "source_family":
                item.get(
                    "source_family"
                ),

            "collection":
                item.get(
                    "collection"
                ),

            "identifier":
                item.get(
                    "identifier"
                ),

            "release":
                item.get(
                    "release"
                ),

            "document_type":
                item.get(
                    "document_type"
                ),

            "title":
                item.get(
                    "title"
                ),

            "section_heading":
                item.get(
                    "section_heading"
                ),

            "source_path":
                item.get(
                    "source_path"
                ),

            "source_shard":
                item.get(
                    "source_shard"
                ),

            # ------------------------------------------------
            # RETRIEVAL INTENT / GROUNDEDNESS EVIDENCE
            # ------------------------------------------------

            "evidence_terms":
                safe_list_value(
                    item.get(
                        "evidence_terms",
                        []
                    )
                ),

            "evidence":
                evidence_text,

            "evidence_chars":
                len(
                    evidence_text
                )
        })


    return evidence_trace


# ============================================================
# SINGLE OPENROUTER API CALL
# ============================================================

def call_openrouter(
    messages,
    tools=None
):
    """
    Execute one OpenRouter Chat Completions request.

    Temperature intentionally remains unset.

    The frozen SYSTEM_PROMPT is carried as the first system
    message in the conversation.
    """

    if not isinstance(
        messages,
        list
    ):

        raise TypeError(
            "OpenRouter messages must be supplied as a list."
        )

    if not messages:

        raise ValueError(
            "OpenRouter messages must not be empty."
        )

    request_args = {

        "model":
            LLM_MODEL,

        "max_tokens":
            LLM_MAX_TOKENS,

        "messages":
            messages
    }

    if tools:

        request_args[
            "tools"
        ] = tools

        request_args[
            "tool_choice"
        ] = "auto"

    # LLM_TEMPERATURE intentionally remains None.
    #
    # Therefore no temperature argument is sent.

    return (
        openrouter
        .chat
        .completions
        .create(
            **request_args
        )
    )


# ============================================================
# END-TO-END OPENROUTER LLM + MCP ORCHESTRATION
# ============================================================

async def run_version_b_e2e(
    mcp_client,
    question,
    verbose=True
):
    """
    Execute one complete Version B question through:

        OpenRouter LLM
              ↓
             MCP
              ↓
        Persistent BM25 Retrieval
              ↓
        OpenRouter LLM

    Maximum EXECUTED MCP searches:
        MAX_MCP_SEARCHES = 3

    Invalid tool arguments do not execute retrieval and therefore
    do not count as an MCP search.

    Returns a structured record suitable for:
        - pilot validation
        - 8-question evaluation
        - Version A/B analysis
        - cross-LLM analysis
    """

    # ========================================================
    # QUESTION VALIDATION
    # ========================================================

    if question is None:

        raise ValueError(
            "Question must not be None."
        )

    if not isinstance(
        question,
        str
    ):

        question = str(
            question
        )

    question = re.sub(
        r"\s+",
        " ",
        question
    ).strip()

    if not question:

        raise ValueError(
            "Question must not be empty."
        )

    # ========================================================
    # EXECUTION STATE
    # ========================================================

    e2e_start = (
        time.perf_counter()
    )

    messages = [

        {
            "role":
                "system",

            "content":
                SYSTEM_PROMPT
        },

        {
            "role":
                "user",

            "content":
                question
        }
    ]

    # Successfully dispatched MCP searches.
    tool_call_count = 0

    # Every model tool request, including malformed ones.
    tool_request_count = 0

    llm_call_count = 0

    tool_queries = []
    tool_routes = []
    tool_sources = []
    tool_source_distributions = []

    retrieval_times = []
    tool_elapsed_times = []
    mcp_roundtrip_times = []
    mcp_overheads = []

    evidence_trace = []
    error_log = []
    turn_usage = []

    total_input_tokens = 0
    total_output_tokens = 0

    final_answer = ""
    final_answer_tokens = 0

    final_stop_reason = None
    final_raw_finish_reason = None

    completion_mode = None

    # ========================================================
    # OPTIONAL HEADER
    # ========================================================

    if verbose:

        print(
            "=" * 90
        )

        print(
            "VERSION B — OPENROUTER LLM + MCP "
            "END-TO-END EXECUTION"
        )

        print(
            "=" * 90
        )

        print(
            f"Question        : "
            f"{question}"
        )

        print(
            f"LLM Provider    : "
            f"{LLM_PROVIDER}"
        )

        print(
            f"LLM Model       : "
            f"{LLM_MODEL}"
        )

        print(
            f"Max MCP Searches: "
            f"{MAX_MCP_SEARCHES}"
        )

    # ========================================================
    # LLM / MCP LOOP
    # ========================================================

    while True:

        # ----------------------------------------------------
        # LLM TURN
        # ----------------------------------------------------

        llm_start = (
            time.perf_counter()
        )

        try:

            response = call_openrouter(

                messages=
                    messages,

                tools=
                    OPENROUTER_MCP_TOOLS
            )

        except Exception as exc:

            error_log.append({

                "stage":
                    "openrouter_api",

                "error_type":
                    type(
                        exc
                    ).__name__,

                "error":
                    str(
                        exc
                    )
            })

            raise

        llm_elapsed_s = (
            time.perf_counter()
            -
            llm_start
        )

        llm_call_count += 1

        usage = extract_usage(
            response
        )

        total_input_tokens += (
            usage[
                "input_tokens"
            ]
        )

        total_output_tokens += (
            usage[
                "output_tokens"
            ]
        )

        raw_finish_reason = (
            get_raw_finish_reason(
                response
            )
        )

        tool_calls = (
            extract_openrouter_tool_calls(
                response
            )
        )

        turn_usage.append({

            "turn":
                llm_call_count,

            "input_tokens":
                usage[
                    "input_tokens"
                ],

            "output_tokens":
                usage[
                    "output_tokens"
                ],

            "elapsed_s":
                round(
                    llm_elapsed_s,
                    6
                ),

            "finish_reason":
                raw_finish_reason,

            "tool_calls":
                len(
                    tool_calls
                )
        })

        # ----------------------------------------------------
        # PRESERVE COMPLETE ASSISTANT MESSAGE
        # ----------------------------------------------------

        messages.append(
            serialize_openrouter_assistant_message(
                response
            )
        )

        # ====================================================
        # NO TOOL REQUEST → FINAL ANSWER
        # ====================================================

        if not tool_calls:

            final_answer = (
                extract_openrouter_text(
                    response
                )
            )

            final_answer_tokens = (
                usage[
                    "output_tokens"
                ]
            )

            final_raw_finish_reason = (
                raw_finish_reason
            )

            final_stop_reason = (
                normalize_finish_reason(
                    raw_finish_reason
                )
            )

            completion_mode = (
                "normal"
            )

            if verbose:

                print(
                    "\nLLM completed without "
                    "another MCP request."
                )

                print(
                    f"LLM Turn Time: "
                    f"{llm_elapsed_s:.3f} sec"
                )

            break

        # ====================================================
        # PROCESS TOOL REQUESTS
        # ====================================================

        tool_messages = []

        for tool_call in tool_calls:

            tool_request_count += 1

            tool_call_id = getattr(
                tool_call,
                "id",
                None
            )

            function = getattr(
                tool_call,
                "function",
                None
            )

            tool_name = getattr(
                function,
                "name",
                ""
            )

            raw_arguments = getattr(
                function,
                "arguments",
                "{}"
            )

            # ------------------------------------------------
            # TOOL CALL ID VALIDATION
            # ------------------------------------------------

            if not tool_call_id:

                raise RuntimeError(
                    "LLM returned a tool call "
                    "without an id."
                )

            # ------------------------------------------------
            # UNKNOWN TOOL
            # ------------------------------------------------

            if (
                tool_name
                !=
                "search_telecom_knowledge"
            ):

                error_message = (
                    f"Unsupported tool requested: "
                    f"{tool_name or 'UNKNOWN'}"
                )

                error_log.append({

                    "stage":
                        "tool_selection",

                    "tool":
                        tool_name,

                    "error":
                        error_message
                })

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        error_message
                })

                continue

            # ------------------------------------------------
            # NORMALIZE / VALIDATE TOOL INPUT
            # ------------------------------------------------

            (
                normalized_input,
                input_error
            ) = normalize_telecom_tool_input(
                raw_arguments
            )

            if input_error is not None:

                error_log.append({

                    "stage":
                        "tool_input_validation",

                    "tool":
                        tool_name,

                    "error":
                        input_error
                })

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        (
                            "Invalid tool input: "
                            f"{input_error}"
                        )
                })

                if verbose:

                    print(
                        "\nMCP TOOL INPUT ERROR"
                    )

                    print(
                        input_error
                    )

                continue

            query = (
                normalized_input[
                    "query"
                ]
            )

            # ------------------------------------------------
            # MCP SEARCH CAP
            # ------------------------------------------------

            if (
                tool_call_count
                >=
                MAX_MCP_SEARCHES
            ):

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        (
                            "MCP search was not executed because "
                            "the maximum search budget has already "
                            "been reached."
                        )
                })

                continue

            # =================================================
            # EXECUTE VALID MCP SEARCH
            # =================================================

            tool_call_count += 1

            tool_queries.append(
                query
            )

            if verbose:

                print(
                    f"\nMCP SEARCH "
                    f"{tool_call_count}/"
                    f"{MAX_MCP_SEARCHES}"
                )

                print(
                    f"Query : "
                    f"{query}"
                )

                print(
                    f"Top-K : "
                    f"{normalized_input['top_k']}"
                )

            mcp_start = (
                time.perf_counter()
            )

            try:

                tool_result = (
                    await mcp_client.call_tool(

                        tool_name,

                        normalized_input
                    )
                )

                mcp_roundtrip_s = (
                    time.perf_counter()
                    -
                    mcp_start
                )

                payload = (
                    extract_mcp_payload(
                        tool_result
                    )
                )

                trace = to_plain_dict(
                    payload.get(
                        "trace",
                        {}
                    )
                )

                retrieval_time_s = (
                    safe_float_value(
                        trace.get(
                            "retrieval_time_s",
                            0
                        )
                    )
                )

                tool_elapsed_s = (
                    safe_float_value(
                        trace.get(
                            "tool_elapsed_s",
                            0
                        )
                    )
                )

                mcp_overhead_s = max(

                    0.0,

                    (
                        mcp_roundtrip_s
                        -
                        tool_elapsed_s
                    )
                )

                route = str(
                    payload.get(
                        "route",
                        ""
                    )
                    or ""
                )

                sources = [

                    str(
                        source
                    )

                    for source
                    in safe_list_value(
                        payload.get(
                            "sources_searched",
                            []
                        )
                    )
                ]

                source_distribution = (
                    to_plain_dict(
                        payload.get(
                            "source_distribution",
                            {}
                        )
                    )
                )

                current_evidence = (
                    extract_evidence_trace(
                        payload
                    )
                )

                # --------------------------------------------
                # TRACE
                # --------------------------------------------

                tool_routes.append(
                    route
                )

                tool_sources.append(
                    sources
                )

                tool_source_distributions.append(
                    source_distribution
                )

                retrieval_times.append(
                    retrieval_time_s
                )

                tool_elapsed_times.append(
                    tool_elapsed_s
                )

                mcp_roundtrip_times.append(
                    mcp_roundtrip_s
                )

                mcp_overheads.append(
                    mcp_overhead_s
                )


                # ------------------------------------------------------------
                # COMPLETE SEARCH TRACE FOR EVALUATION
                # ------------------------------------------------------------

                retrieval_trace = to_plain_dict(
                    payload.get(
                        "trace",
                        {}
                    )
                )


                evidence_trace.append({

                    # --------------------------------------------------------
                    # SEARCH IDENTITY
                    # --------------------------------------------------------

                    "search_number":
                        tool_call_count,

                    "query":
                        query,

                    "top_k":
                        payload.get(
                            "top_k"
                        ),

                    # --------------------------------------------------------
                    # ROUTING
                    # --------------------------------------------------------

                    "route":
                        route,

                    "sources_searched":
                        sources,

                    "source_distribution":
                        source_distribution,

                    "retrieval_trace":
                        retrieval_trace,

                    # --------------------------------------------------------
                    # RESULTS
                    # --------------------------------------------------------

                    "result_count":
                        payload.get(
                            "result_count",
                            len(
                                current_evidence
                            )
                        ),

                    "results":
                        current_evidence,

                    # --------------------------------------------------------
                    # SEARCH-LEVEL TIMING
                    # --------------------------------------------------------

                    "retrieval_time_s":
                        retrieval_time_s,

                    "tool_elapsed_s":
                        tool_elapsed_s,

                    "mcp_roundtrip_s":
                        mcp_roundtrip_s,

                    "mcp_overhead_s":
                        mcp_overhead_s
                })


                # --------------------------------------------
                # MCP PAYLOAD → OPENAI TOOL MESSAGE
                # --------------------------------------------

                tool_result_text = json.dumps(

                    payload,

                    ensure_ascii=False,

                    default=str
                )

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        tool_result_text
                })

                if verbose:

                    print(
                        f"Route     : "
                        f"{route.upper()}"
                    )

                    print(
                        f"Sources   : "
                        f"{sources}"
                    )

                    print(
                        f"Results   : "
                        f"{payload.get('result_count', 0)}"
                    )

                    print(
                        f"Retrieval : "
                        f"{retrieval_time_s:.3f} sec"
                    )

                    print(
                        f"MCP Round : "
                        f"{mcp_roundtrip_s:.3f} sec"
                    )

            # =================================================
            # MCP EXECUTION ERROR
            # =================================================

            except Exception as exc:

                mcp_roundtrip_s = (
                    time.perf_counter()
                    -
                    mcp_start
                )

                mcp_roundtrip_times.append(
                    mcp_roundtrip_s
                )

                error_entry = {

                    "stage":
                        "mcp_tool_call",

                    "search_number":
                        tool_call_count,

                    "tool":
                        tool_name,

                    "query":
                        query,

                    "error_type":
                        type(
                            exc
                        ).__name__,

                    "error":
                        str(
                            exc
                        )
                }

                error_log.append(
                    error_entry
                )

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        (
                            "Tool execution failed: "
                            f"{type(exc).__name__}: "
                            f"{exc}"
                        )
                })

                if verbose:

                    print(
                        f"MCP ERROR: "
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    )

        # ====================================================
        # OPENAI / OPENROUTER TOOL MESSAGE ORDERING
        # ====================================================
        #
        # One role="tool" message is returned for every tool_call,
        # each carrying the corresponding tool_call_id.
        # ====================================================

        messages.extend(
            tool_messages
        )

        search_cap_reached = (
            tool_call_count
            >=
            MAX_MCP_SEARCHES
        )

        # ====================================================
        # SEARCH CAP → FINAL LLM TURN WITHOUT TOOLS
        # ====================================================

        if search_cap_reached:

            messages.append({

                "role":
                    "user",

                "content":
                    (
                        "The external MCP search budget has now "
                        "been reached. Using only the telecom "
                        "evidence already retrieved, provide the "
                        "final answer. Do not request another tool."
                    )
            })

            if verbose:

                print(
                    "\nMaximum MCP search budget reached."
                )

                print(
                    "Requesting final answer using "
                    "retrieved evidence only..."
                )

            final_start = (
                time.perf_counter()
            )

            try:

                final_response = call_openrouter(

                    messages=
                        messages,

                    tools=
                        None
                )

            except Exception as exc:

                error_log.append({

                    "stage":
                        "openrouter_final_answer",

                    "error_type":
                        type(
                            exc
                        ).__name__,

                    "error":
                        str(
                            exc
                        )
                })

                raise

            final_llm_elapsed_s = (
                time.perf_counter()
                -
                final_start
            )

            llm_call_count += 1

            final_usage = (
                extract_usage(
                    final_response
                )
            )

            total_input_tokens += (
                final_usage[
                    "input_tokens"
                ]
            )

            total_output_tokens += (
                final_usage[
                    "output_tokens"
                ]
            )

            final_raw_finish_reason = (
                get_raw_finish_reason(
                    final_response
                )
            )

            final_stop_reason = (
                normalize_finish_reason(
                    final_raw_finish_reason
                )
            )

            final_answer = (
                extract_openrouter_text(
                    final_response
                )
            )

            final_answer_tokens = (
                final_usage[
                    "output_tokens"
                ]
            )

            turn_usage.append({

                "turn":
                    llm_call_count,

                "input_tokens":
                    final_usage[
                        "input_tokens"
                    ],

                "output_tokens":
                    final_usage[
                        "output_tokens"
                    ],

                "elapsed_s":
                    round(
                        final_llm_elapsed_s,
                        6
                    ),

                "finish_reason":
                    final_raw_finish_reason,

                "tool_calls":
                    0
            })

            completion_mode = (
                "forced_after_search_cap"
            )

            if verbose:

                print(
                    f"Final LLM Turn: "
                    f"{final_llm_elapsed_s:.3f} sec"
                )

            break

    # ========================================================
    # FINAL EXECUTION METRICS
    # ========================================================

    e2e_time_s = (
        time.perf_counter()
        -
        e2e_start
    )

    total_retrieval_time_s = sum(
        retrieval_times
    )

    total_tool_elapsed_s = sum(
        tool_elapsed_times
    )

    total_mcp_roundtrip_s = sum(
        mcp_roundtrip_times
    )

    total_mcp_overhead_s = sum(
        mcp_overheads
    )

    final_word_count = len(
        final_answer.split()
    )

    # ========================================================
    # FINAL RESULT OBJECT
    # ========================================================

    result = {

        "architecture":
            VERSION_B_ARCHITECTURE,

        "retrieval_architecture":
            VERSION_B_RETRIEVAL_ARCHITECTURE,

        "provider":
            LLM_PROVIDER,

        "model":
            LLM_MODEL,

        "model_display_name":
            LLM_DISPLAY_NAME,

        "question":
            question,

        "completed":
            bool(
                final_answer.strip()
            ),

        "completion_mode":
            completion_mode,

        # Normalized to existing Version A/B labels.
        "stop_reason":
            final_stop_reason,

        # Provider-native value retained as well.
        "raw_finish_reason":
            final_raw_finish_reason,

        "final_answer":
            final_answer,

        # Compatibility alias used by the shared Benchmark cell.
        "answer":
            final_answer,

        "word_count":
            final_word_count,

        "final_answer_words":
            final_word_count,

        # ----------------------------------------------------
        # LLM
        # ----------------------------------------------------

        "llm_calls":
            llm_call_count,

        # Compatibility alias for existing evaluation code.
        # This can be removed once Cell 12 becomes provider-neutral.
        "claude_calls":
            llm_call_count,

        "input_tokens":
            total_input_tokens,

        "output_tokens":
            total_output_tokens,

        "total_tokens":
            (
                total_input_tokens
                +
                total_output_tokens
            ),

        "final_answer_tokens":
            final_answer_tokens,

        "turn_usage":
            turn_usage,

        # ----------------------------------------------------
        # MCP
        # ----------------------------------------------------

        "mcp_tool_requests":
            tool_request_count,

        "mcp_searches":
            tool_call_count,

        "tool_queries":
            tool_queries,

        "tool_routes":
            tool_routes,

        "tool_sources":
            tool_sources,

        "tool_source_distributions":
            tool_source_distributions,

        # ----------------------------------------------------
        # EVIDENCE
        # ----------------------------------------------------

        "evidence_trace":
            evidence_trace,

        # Canonical Version B search-level trace alias.
        "tool_trace":
            evidence_trace,

        # ----------------------------------------------------
        # TIMING
        # ----------------------------------------------------

        "retrieval_times_s":
            retrieval_times,

        "tool_elapsed_times_s":
            tool_elapsed_times,

        "mcp_roundtrip_times_s":
            mcp_roundtrip_times,

        "mcp_overheads_s":
            mcp_overheads,

        "total_retrieval_time_s":
            total_retrieval_time_s,

        "total_tool_elapsed_s":
            total_tool_elapsed_s,

        "total_mcp_roundtrip_s":
            total_mcp_roundtrip_s,

        "total_mcp_overhead_s":
            total_mcp_overhead_s,

        "e2e_time_s":
            e2e_time_s,

        # Compatibility alias used by prior Version B code.
        "total_latency_sec":
            e2e_time_s,

        # ----------------------------------------------------
        # ERRORS
        # ----------------------------------------------------

        "errors":
            error_log
    }

    # ========================================================
    # OPTIONAL EXECUTION SUMMARY
    # ========================================================

    if verbose:

        print(
            "\n" + "=" * 90
        )

        print(
            "VERSION B — OPENROUTER LLM E2E EXECUTION SUMMARY"
        )

        print(
            "=" * 90
        )

        print(
            f"Completed          : "
            f"{result['completed']}"
        )

        print(
            f"Completion Mode    : "
            f"{completion_mode}"
        )

        print(
            f"Stop Reason        : "
            f"{final_stop_reason}"
        )

        print(
            f"Raw Finish Reason  : "
            f"{final_raw_finish_reason}"
        )

        print(
            f"LLM Calls          : "
            f"{llm_call_count}"
        )

        print(
            f"MCP Tool Requests  : "
            f"{tool_request_count}"
        )

        print(
            f"MCP Searches       : "
            f"{tool_call_count}"
        )

        print(
            f"Tool Routes        : "
            f"{tool_routes}"
        )

        print(
            f"Retrieval Time     : "
            f"{total_retrieval_time_s:.3f} sec"
        )

        print(
            f"MCP Round Trip     : "
            f"{total_mcp_roundtrip_s:.3f} sec"
        )

        print(
            f"E2E Time           : "
            f"{e2e_time_s:.3f} sec"
        )

        print(
            f"Input Tokens       : "
            f"{total_input_tokens:,}"
        )

        print(
            f"Output Tokens      : "
            f"{total_output_tokens:,}"
        )

        print(
            f"Total Tokens       : "
            f"{result['total_tokens']:,}"
        )

        print(
            f"Final Answer Tokens: "
            f"{final_answer_tokens:,}"
        )

        print(
            f"Final Word Count   : "
            f"{final_word_count:,}"
        )

        print(
            f"Errors             : "
            f"{len(error_log)}"
        )

        print(
            "\n" + "=" * 90
        )

        print(
            "FINAL ANSWER"
        )

        print(
            "=" * 90
        )

        print(
            final_answer
        )

        print(
            "=" * 90
        )

    return result


# ============================================================
# STATIC ORCHESTRATION VALIDATION
# ============================================================

if len(
    OPENROUTER_MCP_TOOLS
) != 1:

    raise RuntimeError(
        "Configured OpenRouter LLM must receive exactly one "
        "MCP knowledge-search tool."
    )


if (
    OPENROUTER_MCP_TOOLS[
        0
    ][
        "type"
    ]
    !=
    "function"
):

    raise RuntimeError(
        "OpenRouter MCP tool is not "
        "declared as a function."
    )


if (
    OPENROUTER_MCP_TOOLS[
        0
    ][
        "function"
    ][
        "name"
    ]
    !=
    "search_telecom_knowledge"
):

    raise RuntimeError(
        "Unexpected OpenRouter MCP tool."
    )


if (
    OPENROUTER_MCP_TOOLS[
        0
    ][
        "function"
    ][
        "parameters"
    ].get(
        "type"
    )
    !=
    "object"
):

    raise RuntimeError(
        "OpenRouter function parameters are not "
        "a valid object schema."
    )


tool_parameters = (
    OPENROUTER_MCP_TOOLS[
        0
    ][
        "function"
    ][
        "parameters"
    ]
)


tool_properties = (
    tool_parameters.get(
        "properties",
        {}
    )
    or
    {}
)


if "profile" in tool_properties:

    raise RuntimeError(
        "Version B MCP contract violation: "
        "profile is still exposed to the LLM."
    )


if "query" not in tool_properties:

    raise RuntimeError(
        "Version B MCP contract violation: "
        "query is missing from the tool schema."
    )


unexpected_schema_fields = (
    set(
        tool_properties.keys()
    )
    -
    {
        "query",
        "top_k"
    }
)


if unexpected_schema_fields:

    raise RuntimeError(
        "Unexpected Version B MCP tool fields exposed: "
        f"{sorted(unexpected_schema_fields)}"
    )


if MCP_EXCERPT_CHARS != 2500:

    raise RuntimeError(
        "MCP_EXCERPT_CHARS must remain 2500."
    )


if MAX_MCP_SEARCHES != 3:

    raise RuntimeError(
        "Maximum MCP search budget "
        "must remain 3."
    )


if TOP_K_RESULTS != 5:

    raise RuntimeError(
        "TOP_K_RESULTS must remain 5."
    )


if MAX_RETRIEVED_SOURCES != 5:

    raise RuntimeError(
        "MAX_RETRIEVED_SOURCES must remain 5."
    )


if LLM_MAX_TOKENS != 1800:

    raise RuntimeError(
        "LLM_MAX_TOKENS must remain 1800."
    )


# ============================================================
# TOOL INPUT NORMALIZATION SELF-TEST
# ============================================================

_test_valid_input, _test_valid_error = (
    normalize_telecom_tool_input({

        "query":
            "  Explain   AMF registration  ",

        "top_k":
            "5",

        # Legacy Version B field — must never be forwarded.
        "profile":
            "3gpp",

        "unexpected_parameter":
            "ignored"
    })
)


if _test_valid_error is not None:

    raise RuntimeError(
        "Tool input normalization self-test failed."
    )


if (
    _test_valid_input
    !=
    {
        "query":
            "Explain AMF registration",

        "top_k":
            5
    }
):

    raise RuntimeError(
        "Tool input normalization produced "
        "unexpected output."
    )


_test_missing_query, _test_missing_error = (
    normalize_telecom_tool_input({
        "top_k":
            5
    })
)


if (
    _test_missing_query is not None
    or
    _test_missing_error is None
):

    raise RuntimeError(
        "Missing-query validation self-test failed."
    )


_test_clamped_input, _ = (
    normalize_telecom_tool_input({

        "query":
            "Explain PFCP",

        "top_k":
            999
    })
)


if (
    _test_clamped_input[
        "top_k"
    ]
    !=
    5
):

    raise RuntimeError(
        "Top-K clamping self-test failed."
    )


# ============================================================
# FINISH-REASON NORMALIZATION SELF-TEST
# ============================================================

if (
    normalize_finish_reason(
        "stop"
    )
    !=
    "end_turn"
):

    raise RuntimeError(
        "Finish-reason normalization failed for 'stop'."
    )


if (
    normalize_finish_reason(
        "length"
    )
    !=
    "max_tokens"
):

    raise RuntimeError(
        "Finish-reason normalization failed for 'length'."
    )


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print(
    "=" * 90
)

print(
    "VERSION B — OPENROUTER LLM + MCP ORCHESTRATION"
)

print(
    "=" * 90
)


print(
    f"LLM Provider        : "
    f"{LLM_PROVIDER}"
)

print(
    f"LLM Model           : "
    f"{LLM_MODEL}"
)

print(
    f"OpenRouter Tools    : "
    f"{len(OPENROUTER_MCP_TOOLS)}"
)

print(
    f"Exposed Tool        : "
    f"{OPENROUTER_MCP_TOOLS[0]['function']['name']}"
)

print(
    "Tool Arguments      : "
    f"{sorted(tool_properties.keys())}"
)

print(
    "Profile Argument    : "
    "NOT EXPOSED"
)

print(
    "Tool Schema         : "
    "OpenAI-compatible function"
)

print(
    f"Max MCP Searches    : "
    f"{MAX_MCP_SEARCHES}"
)

print(
    f"Maximum Top-K       : "
    f"{TOP_K_RESULTS}"
)

print(
    "Temperature         : "
    "API DEFAULT"
)


print(
    "\n" + "=" * 90
)


print(
    "✓ Current FastMCP input_schema API used."
)

print(
    "✓ FastMCP schema converted to OpenAI function format."
)

print(
    "✓ OpenRouter tool input normalization enabled."
)

print(
    "✓ Missing/empty query protection enabled."
)

print(
    "✓ Unexpected tool arguments are not forwarded."
)

print(
    "✓ Top-K coercion and 1–5 clamping enabled."
)

print(
    "✓ Exactly one MCP knowledge tool exposed."
)

print(
    "✓ Dynamic 3GPP / TCC / HYBRID routing remains internal."
)

print(
    "✓ LLM-facing profile argument removed."
)

print(
    "✓ OpenAI-compatible tool_call / tool message loop configured."
)

print(
    "✓ Every tool call receives a matching tool_call_id result."
)

print(
    "✓ Maximum executed MCP search budget fixed at 3."
)

print(
    "✓ Final answer generated without tools after search cap."
)

print(
    "✓ OpenRouter token accounting enabled."
)

print(
    "✓ Retrieval, tool and MCP timing traces enabled."
)

print(
    "✓ Exact bounded Version B evidence + full BM25 trace preserved."
)

print(
    "✓ Version A/B-compatible result fields preserved."
)

print(
    "✓ Provider-native finish reasons normalized for A/B comparison."
)

print(
    "✓ Tool-input and finish-reason self-tests passed."
)

print(
    "✓ No end-to-end question executed in this cell."
)

print(
    "✓ Ready for Cell 12 — Three-Route End-to-End Pilot."
)

print(
    "=" * 90
)


VERSION B — OPENROUTER LLM + MCP ORCHESTRATION
LLM Provider        : OpenRouter
LLM Model           : deepseek/deepseek-v4-flash-0731
OpenRouter Tools    : 1
Exposed Tool        : search_telecom_knowledge
Tool Arguments      : ['query', 'top_k']
Profile Argument    : NOT EXPOSED
Tool Schema         : OpenAI-compatible function
Max MCP Searches    : 3
Maximum Top-K       : 5
Temperature         : API DEFAULT

✓ Current FastMCP input_schema API used.
✓ FastMCP schema converted to OpenAI function format.
✓ OpenRouter tool input normalization enabled.
✓ Missing/empty query protection enabled.
✓ Unexpected tool arguments are not forwarded.
✓ Top-K coercion and 1–5 clamping enabled.
✓ Exactly one MCP knowledge tool exposed.
✓ Dynamic 3GPP / TCC / HYBRID routing remains internal.
✓ LLM-facing profile argument removed.
✓ OpenAI-compatible tool_call / tool message loop configured.
✓ Every tool call receives a matching tool_call_id result.
✓ Maximum executed MCP search budget fixed at 3.
✓ Final

**Observation — End-to-End MCP Orchestration**

DeepSeek V4 Flash 0731 can request evidence through the single MCP knowledge tool, receive bounded Version B evidence and continue to a final answer while the runtime records tool calls, routes, source distributions, retrieval traces, tokens and timing.

***Key Decision:*** Preserve autonomous tool use under a hard three-search ceiling while keeping provider-specific orchestration separate from the frozen retrieval service.


## **Cell 12 — Three-Route End-to-End Pilot**


In [25]:
# ============================================================
# CELL 12 — THREE-ROUTE OPENROUTER END-TO-END PILOT
# ============================================================
#
# Purpose:
#
# Validate the COMPLETE Version B runtime before launching the
# eight-question Benchmark v2:
#
#     OpenRouter LLM
#       ↓
#     unified MCP tool
#       ↓
#     deterministic 3GPP / TCC / HYBRID router
#       ↓
#     persistent BM25 retrieval
#       ↓
#     bounded evidence
#       ↓
#     OpenRouter LLM final answer
#
# Pilot routes:
#
#     P1 → 3GPP-only
#     P2 → TCC-only / IETF
#     P3 → Hybrid 3GPP + TCC
#
# This cell performs REAL OpenRouter API calls.
#
# It is shared by:
#     - Gemma 4 26B A4B IT
#     - DeepSeek V4 Flash 0731
# ============================================================


# ============================================================
# OPENROUTER PILOT CONFIGURATION VALIDATION
# ============================================================

if LLM_PROVIDER != "OpenRouter":

    raise RuntimeError(
        "Cell 12 requires the OpenRouter runtime from Cell 10."
    )


SUPPORTED_VERSION_B_OPENROUTER_MODELS = {

    "google/gemma-4-26b-a4b-it",

    "deepseek/deepseek-v4-flash-0731"
}


if LLM_MODEL not in SUPPORTED_VERSION_B_OPENROUTER_MODELS:

    raise RuntimeError(
        "Unsupported Version B OpenRouter pilot model: "
        f"{LLM_MODEL}"
    )


# ============================================================
# PILOT CASES
# ============================================================

VERSION_B_PILOT_CASES = [

    {
        "id":
            "P1",

        "name":
            "3GPP-only",

        "question":
            (
                "Explain the role of the AMF in registration "
                "and mobility management procedures in a "
                "5G Standalone network."
            ),

        "expected_route":
            "3gpp",

        "expected_source_families":
            {
                "3GPP"
            }
    },

    {
        "id":
            "P2",

        "name":
            "TCC-only",

        "question":
            (
                "Explain the key mechanisms of QUIC as defined "
                "by the IETF, including connection establishment, "
                "stream multiplexing and connection migration."
            ),

        "expected_route":
            "tcc",

        "expected_source_families":
            {
                "TCC"
            }
    },

    {
        "id":
            "P3",

        "name":
            "Hybrid 3GPP + TCC",

        "question":
            (
                "Explain how HTTP and TLS support communication "
                "in the 5G Service-Based Architecture. Distinguish "
                "the roles and requirements defined by 3GPP for "
                "network functions such as the AMF and SMF from "
                "the HTTP/TLS transport and security mechanisms "
                "defined by the IETF."
            ),

        "expected_route":
            "hybrid",

        "expected_source_families":
            {
                "3GPP",
                "TCC"
            }
    }
]


# ============================================================
# PILOT VALIDATION HELPERS
# ============================================================

def pilot_source_families(
    result
):
    """
    Collect source families from the exact evidence returned by
    all executed MCP searches.
    """

    observed = set()


    for search in (
        result.get(
            "tool_trace",
            []
        )
        or
        []
    ):

        for evidence_item in (
            search.get(
                "results",
                []
            )
            or
            []
        ):

            source_family = (
                str(
                    evidence_item.get(
                        "source_family",
                        ""
                    )
                    or
                    ""
                )
                .strip()
            )


            if source_family:

                observed.add(
                    source_family
                )


    return observed


def pilot_evidence_count(
    result
):
    """
    Count exact bounded evidence items captured across all
    executed MCP searches.
    """

    return sum(

        len(
            search.get(
                "results",
                []
            )
            or
            []
        )

        for search
        in (
            result.get(
                "tool_trace",
                []
            )
            or
            []
        )
    )


def pilot_evidence_with_text_count(
    result
):
    """
    Count evidence items containing non-empty bounded evidence.
    """

    count = 0


    for search in (
        result.get(
            "tool_trace",
            []
        )
        or
        []
    ):

        for evidence_item in (
            search.get(
                "results",
                []
            )
            or
            []
        ):

            evidence_text = (
                str(
                    evidence_item.get(
                        "evidence",
                        ""
                    )
                    or
                    ""
                )
                .strip()
            )


            if evidence_text:

                count += 1


    return count


def validate_version_b_pilot_result(
    case,
    result
):
    """
    Validate one complete OpenRouter LLM + Version B MCP pilot result.

    This is an execution/routing/evidence validation only.
    Formal answer-quality and groundedness evaluation remains
    in the final analysis notebook.
    """

    checks = {}


    # --------------------------------------------------------
    # COMPLETION
    # --------------------------------------------------------

    final_answer = (
        str(
            result.get(
                "final_answer",
                ""
            )
            or
            ""
        )
        .strip()
    )


    checks[
        "completed"
    ] = bool(
        result.get(
            "completed",
            False
        )
    )


    checks[
        "final_answer_non_empty"
    ] = bool(
        final_answer
    )


    # --------------------------------------------------------
    # MODEL / ARCHITECTURE
    # --------------------------------------------------------

    checks[
        "architecture"
    ] = (
        result.get(
            "architecture"
        )
        ==
        "Version B"
    )


    checks[
        "retrieval_architecture"
    ] = (
        result.get(
            "retrieval_architecture"
        )
        ==
        "Persistent DuckDB BM25/FTS"
    )


    checks[
        "model"
    ] = (
        result.get(
            "model"
        )
        ==
        LLM_MODEL
    )


    # --------------------------------------------------------
    # MCP SEARCH BUDGET
    # --------------------------------------------------------

    mcp_searches = safe_int_value(
        result.get(
            "mcp_searches",
            0
        )
    )


    checks[
        "at_least_one_mcp_search"
    ] = (
        mcp_searches
        >=
        1
    )


    checks[
        "search_budget"
    ] = (
        1
        <=
        mcp_searches
        <=
        MAX_MCP_SEARCHES
    )


    # --------------------------------------------------------
    # SEARCH TRACE COUNT
    # --------------------------------------------------------

    tool_trace = (
        result.get(
            "tool_trace",
            []
        )
        or
        []
    )


    checks[
        "tool_trace_count"
    ] = (
        len(
            tool_trace
        )
        ==
        mcp_searches
    )


    # --------------------------------------------------------
    # ROUTING
    # --------------------------------------------------------

    routes_used = [

        str(
            route
            or
            ""
        )
        .strip()
        .lower()

        for route
        in (
            result.get(
                "tool_routes",
                []
            )
            or
            []
        )

        if str(
            route
            or
            ""
        ).strip()
    ]


    # --------------------------------------------------------
    # ROUTE EXPECTATION
    # --------------------------------------------------------
    #
    # Keep pilot semantics aligned with the frozen Benchmark v2.
    #
    # A Hybrid task is successful when the LLM either:
    #
    #   1. issues an explicit HYBRID search, or
    #   2. decomposes the task into complementary 3GPP + TCC
    #      searches whose combined evidence covers both domains.
    #
    # This avoids incorrectly failing legitimate multi-search
    # orchestration such as:
    #
    #       TCC → 3GPP → 3GPP
    #
    # while still recording the literal routes for Notebook 17.
    # --------------------------------------------------------

    expected_route = (
        case[
            "expected_route"
        ]
    )


    if expected_route == "hybrid":

        hybrid_route_detected = (
            "hybrid"
            in
            routes_used
        )


        hybrid_decomposition_detected = (
            "3gpp"
            in
            routes_used

            and

            "tcc"
            in
            routes_used
        )


        checks[
            "expected_route_detected"
        ] = (
            hybrid_route_detected
            or
            hybrid_decomposition_detected
        )


        checks[
            "route_adherence"
        ] = (
            bool(
                routes_used
            )

            and

            all(
                route
                in
                {
                    "hybrid",
                    "3gpp",
                    "tcc"
                }

                for route
                in routes_used
            )
        )


    else:

        checks[
            "expected_route_detected"
        ] = (
            expected_route
            in
            routes_used
        )


        checks[
            "route_adherence"
        ] = (
            bool(
                routes_used
            )

            and

            all(
                route
                ==
                expected_route

                for route
                in routes_used
            )
        )


    # --------------------------------------------------------
    # SOURCE-FAMILY COVERAGE
    # --------------------------------------------------------

    observed_source_families = (
        pilot_source_families(
            result
        )
    )


    checks[
        "expected_source_families"
    ] = (
        case[
            "expected_source_families"
        ]
        .issubset(
            observed_source_families
        )
    )


    if (
        case[
            "expected_route"
        ]
        ==
        "3gpp"
    ):

        checks[
            "3gpp_source_purity"
        ] = (
            observed_source_families
            ==
            {
                "3GPP"
            }
        )

    else:

        checks[
            "3gpp_source_purity"
        ] = True


    if (
        case[
            "expected_route"
        ]
        ==
        "tcc"
    ):

        checks[
            "tcc_source_purity"
        ] = (
            observed_source_families
            ==
            {
                "TCC"
            }
        )

    else:

        checks[
            "tcc_source_purity"
        ] = True


    if (
        case[
            "expected_route"
        ]
        ==
        "hybrid"
    ):

        checks[
            "hybrid_source_mix"
        ] = (
            {
                "3GPP",
                "TCC"
            }
            .issubset(
                observed_source_families
            )
        )

    else:

        checks[
            "hybrid_source_mix"
        ] = True


    # --------------------------------------------------------
    # EVIDENCE CAPTURE
    # --------------------------------------------------------

    evidence_items = (
        pilot_evidence_count(
            result
        )
    )


    evidence_items_with_text = (
        pilot_evidence_with_text_count(
            result
        )
    )


    checks[
        "evidence_items_present"
    ] = (
        evidence_items
        >
        0
    )


    checks[
        "evidence_text_complete"
    ] = (
        evidence_items
        >
        0
        and
        evidence_items_with_text
        ==
        evidence_items
    )


    # Every successful search should return at least one and no
    # more than the controlled Top-K evidence items.
    checks[
        "per_search_evidence_bounds"
    ] = all(

        1
        <=
        len(
            search.get(
                "results",
                []
            )
            or
            []
        )
        <=
        TOP_K_RESULTS

        for search
        in tool_trace
    )


    # --------------------------------------------------------
    # RETRIEVAL TRACE
    # --------------------------------------------------------

    checks[
        "retrieval_trace_present"
    ] = all(

        isinstance(
            search.get(
                "retrieval_trace"
            ),
            dict
        )
        and
        bool(
            search.get(
                "retrieval_trace"
            )
        )

        for search
        in tool_trace
    )


    checks[
        "retrieval_latency_present"
    ] = all(

        safe_float_value(
            search.get(
                "retrieval_time_s",
                0.0
            )
        )
        >
        0

        for search
        in tool_trace
    )


    checks[
        "mcp_roundtrip_present"
    ] = all(

        safe_float_value(
            search.get(
                "mcp_roundtrip_s",
                0.0
            )
        )
        >
        0

        for search
        in tool_trace
    )


    # --------------------------------------------------------
    # TOKEN / TIMING ACCOUNTING
    # --------------------------------------------------------

    checks[
        "token_accounting"
    ] = (
        safe_int_value(
            result.get(
                "total_tokens",
                0
            )
        )
        ==
        (
            safe_int_value(
                result.get(
                    "input_tokens",
                    0
                )
            )
            +
            safe_int_value(
                result.get(
                    "output_tokens",
                    0
                )
            )
        )
        and
        safe_int_value(
            result.get(
                "total_tokens",
                0
            )
        )
        >
        0
    )


    checks[
        "e2e_latency"
    ] = (
        safe_float_value(
            result.get(
                "e2e_time_s",
                0.0
            )
        )
        >
        0
    )


    checks[
        "turn_usage"
    ] = (
        len(
            result.get(
                "turn_usage",
                []
            )
            or
            []
        )
        ==
        safe_int_value(
            result.get(
                "llm_turns",
                result.get(
                    "claude_calls",
                    0
                )
            )
        )
    )


    # --------------------------------------------------------
    # ERRORS
    # --------------------------------------------------------

    errors = (
        result.get(
            "errors",
            []
        )
        or
        []
    )


    checks[
        "no_execution_errors"
    ] = (
        len(
            errors
        )
        ==
        0
    )


    details = {

        "routes_used":
            routes_used,

        "observed_source_families":
            sorted(
                observed_source_families
            ),

        "mcp_searches":
            mcp_searches,

        "mcp_tool_requests":
            safe_int_value(
                result.get(
                    "mcp_tool_requests",
                    0
                )
            ),

        "llm_calls":
            safe_int_value(
                result.get(
                    "llm_turns",
                    result.get(
                        "claude_calls",
                        0
                    )
                )
            ),

        "evidence_items":
            evidence_items,

        "evidence_items_with_text":
            evidence_items_with_text,

        "retrieval_time_s":
            safe_float_value(
                result.get(
                    "total_retrieval_time_s",
                    0.0
                )
            ),

        "mcp_roundtrip_s":
            safe_float_value(
                result.get(
                    "total_mcp_roundtrip_s",
                    0.0
                )
            ),

        "mcp_overhead_s":
            safe_float_value(
                result.get(
                    "total_mcp_overhead_s",
                    0.0
                )
            ),

        "e2e_time_s":
            safe_float_value(
                result.get(
                    "e2e_time_s",
                    0.0
                )
            ),

        "input_tokens":
            safe_int_value(
                result.get(
                    "input_tokens",
                    0
                )
            ),

        "output_tokens":
            safe_int_value(
                result.get(
                    "output_tokens",
                    0
                )
            ),

        "total_tokens":
            safe_int_value(
                result.get(
                    "total_tokens",
                    0
                )
            ),

        "final_answer_tokens":
            safe_int_value(
                result.get(
                    "final_answer_tokens",
                    0
                )
            ),

        "word_count":
            safe_int_value(
                result.get(
                    "word_count",
                    0
                )
            ),

        "stop_reason":
            result.get(
                "stop_reason"
            ),

        "completion_mode":
            result.get(
                "completion_mode"
            ),

        "errors":
            errors
    }


    return (
        checks,
        details
    )


# ============================================================
# RUN COMPLETE THREE-ROUTE PILOT
# ============================================================

async def run_version_b_three_route_pilot():
    """
    Run all three Version B pilot questions through a fresh,
    live FastMCP client.
    """

    print("=" * 100)

    print(
        "VERSION B — THREE-ROUTE OPENROUTER LLM + MCP PILOT"
    )

    print("=" * 100)


    print(
        f"LLM Model : "
        f"{LLM_MODEL}"
    )

    print(
        "MCP Tool     : "
        "search_telecom_knowledge(query, top_k=5)"
    )

    print(
        f"Search Cap   : "
        f"{MAX_MCP_SEARCHES}"
    )

    print(
        f"Top-K        : "
        f"{TOP_K_RESULTS}"
    )


    pilot_results = []


    async with Client(
        mcp
    ) as mcp_client:


        for case in (
            VERSION_B_PILOT_CASES
        ):

            print(
                "\n" + "=" * 100
            )

            print(
                f"{case['id']} — "
                f"{case['name']}"
            )

            print("=" * 100)


            result = (
                await run_version_b_e2e(
                    mcp_client=
                        mcp_client,

                    question=
                        case[
                            "question"
                        ],

                    verbose=
                        True
                )
            )


            (
                checks,
                details
            ) = validate_version_b_pilot_result(
                case=
                    case,

                result=
                    result
            )


            passed = all(
                checks.values()
            )


            pilot_results.append({

                "case":
                    case,

                "result":
                    result,

                "checks":
                    checks,

                "details":
                    details,

                "passed":
                    passed
            })


            # ------------------------------------------------
            # PILOT CHECKS
            # ------------------------------------------------

            print(
                "\nPILOT VALIDATION CHECKS"
            )

            print(
                "-" * 100
            )


            for check_name, ok in (
                checks.items()
            ):

                print(
                    f"{check_name:<36} : "
                    f"{'PASS' if ok else 'FAIL'}"
                )


            # ------------------------------------------------
            # PILOT SUMMARY
            # ------------------------------------------------

            print(
                "\nPILOT EXECUTION SUMMARY"
            )

            print(
                "-" * 100
            )


            print(
                f"Routes Used      : "
                f"{details['routes_used']}"
            )

            print(
                f"Source Families  : "
                f"{details['observed_source_families']}"
            )

            print(
                f"MCP Searches     : "
                f"{details['mcp_searches']}"
            )

            print(
                f"MCP Tool Requests: "
                f"{details['mcp_tool_requests']}"
            )

            print(
                f"LLM Calls     : "
                f"{details['llm_calls']}"
            )

            print(
                f"Evidence Items   : "
                f"{details['evidence_items']}"
            )

            print(
                f"Evidence w/Text  : "
                f"{details['evidence_items_with_text']}"
            )

            print(
                f"Retrieval Time   : "
                f"{details['retrieval_time_s']:.3f} sec"
            )

            print(
                f"MCP Roundtrip    : "
                f"{details['mcp_roundtrip_s']:.3f} sec"
            )

            print(
                f"MCP Overhead     : "
                f"{details['mcp_overhead_s']:.3f} sec"
            )

            print(
                f"E2E Time         : "
                f"{details['e2e_time_s']:.3f} sec"
            )

            print(
                f"Total Tokens     : "
                f"{details['total_tokens']:,}"
            )

            print(
                f"Answer Words     : "
                f"{details['word_count']:,}"
            )

            print(
                f"Stop Reason      : "
                f"{details['stop_reason']}"
            )

            print(
                f"Completion Mode  : "
                f"{details['completion_mode']}"
            )


            print(
                "\n" + "-" * 100
            )

            print(
                f"{case['id']} STATUS: "
                f"{'PASS' if passed else 'REVIEW'}"
            )

            print(
                "-" * 100
            )


    # ========================================================
    # FINAL PILOT SUMMARY
    # ========================================================

    summary_rows = []


    for item in pilot_results:

        details = (
            item[
                "details"
            ]
        )


        summary_rows.append({

            "pilot":
                item[
                    "case"
                ][
                    "id"
                ],

            "expected_route":
                item[
                    "case"
                ][
                    "expected_route"
                ],

            "routes_used":
                ",".join(
                    details[
                        "routes_used"
                    ]
                ),

            "source_families":
                ",".join(
                    details[
                        "observed_source_families"
                    ]
                ),

            "mcp_searches":
                details[
                    "mcp_searches"
                ],

            "llm_calls":
                details[
                    "llm_calls"
                ],

            "evidence_items":
                details[
                    "evidence_items"
                ],

            "retrieval_time_s":
                round(
                    details[
                        "retrieval_time_s"
                    ],
                    3
                ),

            "e2e_time_s":
                round(
                    details[
                        "e2e_time_s"
                    ],
                    3
                ),

            "total_tokens":
                details[
                    "total_tokens"
                ],

            "answer_words":
                details[
                    "word_count"
                ],

            "status":
                (
                    "PASS"
                    if item[
                        "passed"
                    ]
                    else
                    "REVIEW"
                )
        })


    summary_df = pd.DataFrame(
        summary_rows
    )


    print(
        "\n" + "=" * 100
    )

    print(
        "VERSION B — THREE-ROUTE END-TO-END PILOT SUMMARY"
    )

    print("=" * 100)


    display(
        summary_df
    )


    all_passed = all(
        item[
            "passed"
        ]
        for item
        in pilot_results
    )


    print(
        "\n" + "=" * 100
    )


    if all_passed:

        print(
            "✓ VERSION B THREE-ROUTE END-TO-END PILOT PASSED"
        )

        print(
            "✓ The OpenRouter LLM successfully used the unified MCP tool."
        )

        print(
            "✓ 3GPP, TCC and Hybrid route behavior passed."
        )

        print(
            "✓ Single-domain searches stayed on-route; Hybrid allowed "
            "explicit or complementary 3GPP/TCC decomposition."
        )

        print(
            "✓ Required source-family evidence reached the LLM."
        )

        print(
            "✓ Exact bounded evidence text was captured."
        )

        print(
            "✓ Full persistent BM25 retrieval traces were captured."
        )

        print(
            "✓ Per-turn token and timing accounting is complete."
        )

        print(
            "✓ No execution errors were recorded."
        )

        print(
            "✓ OpenRouter runtime is ready for Benchmark v2."
        )


    else:

        failed_cases = [

            item[
                "case"
            ][
                "id"
            ]

            for item
            in pilot_results

            if not item[
                "passed"
            ]
        ]


        print(
            "⚠ VERSION B END-TO-END PILOT REQUIRES REVIEW"
        )

        print(
            f"Failed cases: "
            f"{failed_cases}"
        )

        print(
            "Review the failed route/evidence/execution checks "
            "before launching Benchmark v2."
        )


    print("=" * 100)


    if not all_passed:

        raise RuntimeError(
            "Version B three-route end-to-end pilot "
            "did not fully pass."
        )


    return (
        pilot_results,
        summary_df
    )


# ============================================================
# EXECUTE THREE-ROUTE PILOT
# ============================================================

(
    VERSION_B_PILOT_RESULTS,
    VERSION_B_PILOT_SUMMARY
) = await run_version_b_three_route_pilot()


VERSION B — THREE-ROUTE OPENROUTER LLM + MCP PILOT
LLM Model : deepseek/deepseek-v4-flash-0731
MCP Tool     : search_telecom_knowledge(query, top_k=5)
Search Cap   : 3
Top-K        : 5

P1 — 3GPP-only
VERSION B — OPENROUTER LLM + MCP END-TO-END EXECUTION
Question        : Explain the role of the AMF in registration and mobility management procedures in a 5G Standalone network.
LLM Provider    : OpenRouter
LLM Model       : deepseek/deepseek-v4-flash-0731
Max MCP Searches: 3

MCP SEARCH 1/3
Query : AMF role in registration and mobility management procedures 5G standalone
Top-K : 5
Route     : 3GPP
Sources   : ['3GPP']
Results   : 5
Retrieval : 2.188 sec
MCP Round : 2.198 sec

MCP SEARCH 2/3
Query : AMF functions registration area management reachability mobility NAS termination N2 N1
Top-K : 5
Route     : 3GPP
Sources   : ['3GPP']
Results   : 5
Retrieval : 1.738 sec
MCP Round : 1.744 sec

MCP SEARCH 3/3
Query : AMF supports NAS authentication UE context AN release CM-IDLE handover N2 mo

,pilot,expected_route,routes_used,source_families,mcp_searches,llm_calls,evidence_items,retrieval_time_s,e2e_time_s,total_tokens,answer_words,status
0,P1,3gpp,"3gpp,3gpp,3gpp",3GPP,3,4,15,5.709,147.749,37963,396,PASS
1,P2,tcc,tcc,TCC,1,2,5,2.158,82.786,7679,486,PASS
2,P3,hybrid,"tcc,hybrid,3gpp","3GPP,TCC",3,3,15,5.913,147.669,30976,134,PASS



✓ VERSION B THREE-ROUTE END-TO-END PILOT PASSED
✓ The OpenRouter LLM successfully used the unified MCP tool.
✓ 3GPP, TCC and Hybrid route behavior passed.
✓ Single-domain searches stayed on-route; Hybrid allowed explicit or complementary 3GPP/TCC decomposition.
✓ Required source-family evidence reached Claude.
✓ Exact bounded evidence text was captured.
✓ Full persistent BM25 retrieval traces were captured.
✓ Per-turn token and timing accounting is complete.
✓ No execution errors were recorded.
✓ OpenRouter runtime is ready for Benchmark v2.


**Observation — Three-Route End-to-End Pilot**

The executed pilot passed **3GPP-only, TCC/IETF-only and Hybrid** pre-flight cases. Hybrid is valid either through one explicit Hybrid search or complementary 3GPP+TCC searches whose accumulated evidence covers both source families.

***Key Decision:*** Treat the pilot only as a runtime pre-flight check; do not tune retrieval/routing after formal Benchmark v2 begins.


# **SECTION 5 — Version B Evaluation**

## **Cell 13 — Run 8-Question Version B Evaluation**

In [26]:
# ============================================================
# CELL 13 — MODULE 3 EVALUATION BENCHMARK V2
# VERSION B — OPENROUTER (GEMMA 4 / DEEPSEEK V4 FLASH)
# ============================================================

from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import time
import pandas as pd


# ============================================================
# BENCHMARK V2
# 5 × 3GPP | 2 × TCC | 1 × HYBRID
# ============================================================

BENCHMARK_VERSION = "module3_eval_v2"

VERSION_B_EVAL_QUESTIONS = [

    {
        "id": "Q1",
        "domain": "5G Core",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["23.501", "23.502"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "AMF",
            "registration",
            "mobility management",
            "NAS",
            "N1",
            "N2"
        ],
        "question": (
            "Explain the role of the AMF in registration and mobility "
            "management procedures in a 5G Standalone network."
        )
    },

    {
        "id": "Q2",
        "domain": "Mobility",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["23.502", "38.300", "38.423"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "source gNB",
            "target gNB",
            "AMF",
            "handover",
            "Xn",
            "N2"
        ],
        "question": (
            "Explain how inter-gNB handover operates in a 5G network, "
            "including the roles of the source gNB, target gNB and AMF."
        )
    },

    {
        "id": "Q3",
        "domain": "RAN",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["38.331"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "radio link failure",
            "RLF",
            "RRC",
            "T310",
            "N310",
            "RRC re-establishment"
        ],
        "question": (
            "Explain how radio link failure is detected and handled "
            "in a 5G NR network."
        )
    },

    {
        "id": "Q4",
        "domain": "QoS",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["23.501", "23.503"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "5QI",
            "QoS Flow",
            "QFI",
            "SMF",
            "UPF",
            "QoS rules"
        ],
        "question": (
            "Explain how 5QI and QoS flows are handled in a 5G network, "
            "including the roles of the SMF and UPF."
        )
    },

    {
        "id": "Q5",
        "domain": "Security",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["33.501"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "AMF",
            "AUSF",
            "UDM",
            "5G-AKA",
            "SEAF",
            "authentication"
        ],
        "question": (
            "Explain the 5G authentication procedure and the roles "
            "of the AMF, AUSF and UDM."
        )
    },

    {
        "id": "Q6",
        "domain": "Internet Transport",
        "knowledge_scope": "TCC",
        "expected_route": "tcc",
        "expected_source_families": ["TCC"],
        "expected_3gpp_specs": [],
        "expected_tcc_collections": ["IETF-RFCs", "IETF-Drafts"],
        "expected_elements": [
            "QUIC",
            "TLS 1.3",
            "stream multiplexing",
            "Connection ID",
            "connection migration"
        ],
        "question": (
            "Explain the key mechanisms of QUIC as defined by the IETF, "
            "including connection establishment, stream multiplexing "
            "and connection migration."
        )
    },

    {
        "id": "Q7",
        "domain": "Telecom AI Research",
        "knowledge_scope": "TCC",
        "expected_route": "tcc",
        "expected_source_families": ["TCC"],
        "expected_3gpp_specs": [],
        "expected_tcc_collections": ["IEEE-Access", "OpenAlex"],
        "expected_elements": [
            "traffic prediction",
            "machine learning",
            "forecasting",
            "network management",
            "benefits",
            "challenges"
        ],
        "question": (
            "According to IEEE and research literature, what are the main "
            "benefits and technical challenges of using AI/ML-based traffic "
            "prediction for mobile network management?"
        )
    },

    {
        "id": "Q8",
        "domain": "5G SBA + Internet Protocols",
        "knowledge_scope": "Hybrid",
        "expected_route": "hybrid",
        "expected_source_families": ["3GPP", "TCC"],
        "expected_3gpp_specs": ["23.501", "33.501"],
        "expected_tcc_collections": ["IETF-RFCs", "IETF-Drafts"],
        "expected_elements": [
            "Service-Based Architecture",
            "AMF",
            "SMF",
            "HTTP",
            "TLS",
            "3GPP",
            "IETF"
        ],
        "question": (
            "Explain how HTTP and TLS support communication in the 5G "
            "Service-Based Architecture. Distinguish the roles and "
            "requirements defined by 3GPP for network functions such as "
            "the AMF and SMF from the HTTP/TLS transport and security "
            "mechanisms defined by the IETF."
        )
    }
]


# ============================================================
# BENCHMARK VALIDATION
# ============================================================

assert len(VERSION_B_EVAL_QUESTIONS) == 8

assert [item["id"] for item in VERSION_B_EVAL_QUESTIONS] == [
    "Q1", "Q2", "Q3", "Q4",
    "Q5", "Q6", "Q7", "Q8"
]

assert [item["expected_route"] for item in VERSION_B_EVAL_QUESTIONS].count("3gpp") == 5
assert [item["expected_route"] for item in VERSION_B_EVAL_QUESTIONS].count("tcc") == 2
assert [item["expected_route"] for item in VERSION_B_EVAL_QUESTIONS].count("hybrid") == 1

assert MAX_MCP_SEARCHES == 3
assert TOP_K_RESULTS == 5
assert MAX_RETRIEVED_SOURCES == 5
assert MCP_EXCERPT_CHARS == 2500


# ============================================================
# STATIC ROUTER PRE-CHECK
# ============================================================

for item in VERSION_B_EVAL_QUESTIONS:

    routing = route_telecom_query(
        item["question"]
    )

    actual_route = str(
        routing.get(
            "route",
            ""
        )
        or ""
    ).lower()

    expected_route = item[
        "expected_route"
    ]

    if actual_route != expected_route:

        raise RuntimeError(
            f"{item['id']} router mismatch: "
            f"expected={expected_route}, actual={actual_route}"
        )


# ============================================================
# OUTPUT LOCATION
# ============================================================

EVAL_DIR = Path(
    "/content/version_b_evaluation"
)

EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MODEL_FILE_LABELS = {

    "google/gemma-4-26b-a4b-it":
        "gemma_4_26b_a4b_it",

    "deepseek/deepseek-v4-flash-0731":
        "deepseek_v4_flash_0731"
}


if LLM_MODEL not in MODEL_FILE_LABELS:

    raise RuntimeError(
        "Unsupported OpenRouter model for "
        "Version B Benchmark v2: "
        f"{LLM_MODEL}"
    )


MODEL_FILE_LABEL = (
    MODEL_FILE_LABELS[
        LLM_MODEL
    ]
)


VERSION_B_JSON_PATH = (
    EVAL_DIR
    / f"version_b_{MODEL_FILE_LABEL}_evaluation_v2.json"
)

VERSION_B_CSV_PATH = (
    EVAL_DIR
    / f"version_b_{MODEL_FILE_LABEL}_summary_v2.csv"
)


# ============================================================
# OPENROUTER BENCHMARK CONFIGURATION VALIDATION
# ============================================================

if LLM_PROVIDER != "OpenRouter":

    raise RuntimeError(
        "Cell 13 requires the OpenRouter runtime from Cell 10."
    )


SUPPORTED_VERSION_B_OPENROUTER_MODELS = {

    "google/gemma-4-26b-a4b-it",

    "deepseek/deepseek-v4-flash-0731"
}


if LLM_MODEL not in SUPPORTED_VERSION_B_OPENROUTER_MODELS:

    raise RuntimeError(
        "Unsupported Version B OpenRouter benchmark model: "
        f"{LLM_MODEL}"
    )


# ============================================================
# FINGERPRINTS
# ============================================================

SYSTEM_PROMPT_HASH = hashlib.sha256(
    SYSTEM_PROMPT.encode("utf-8")
).hexdigest()

SYSTEM_PROMPT_WORDS = len(
    SYSTEM_PROMPT.split()
)

SYSTEM_PROMPT_CHARS = len(
    SYSTEM_PROMPT
)

benchmark_payload = json.dumps(
    VERSION_B_EVAL_QUESTIONS,
    sort_keys=True,
    ensure_ascii=False,
    separators=(",", ":")
)

BENCHMARK_SHA256 = hashlib.sha256(
    benchmark_payload.encode("utf-8")
).hexdigest()


# ============================================================
# EXPERIMENT METADATA
# ============================================================

evaluation_timestamp = datetime.now(
    timezone.utc
).isoformat()

experiment_metadata = {

    "experiment":
        "Telecom AI MCP",

    "module":
        "Module 3",

    "architecture_version":
        "Version B",

    "evaluation_type":
        "8-question end-to-end MCP evaluation",

    "evaluation_version":
        BENCHMARK_VERSION,

    "evaluation_timestamp_utc":
        evaluation_timestamp,

    "question_count":
        len(
            VERSION_B_EVAL_QUESTIONS
        ),

    "benchmark": {

        "version":
            BENCHMARK_VERSION,

        "sha256":
            BENCHMARK_SHA256,

        "composition": {
            "3gpp": 5,
            "tcc": 2,
            "hybrid": 1
        },

        # Preserve the exact benchmark definition in the artifact.
        "questions":
            VERSION_B_EVAL_QUESTIONS
    },

    "llm": {

        "provider":
            LLM_PROVIDER,

        "model":
            LLM_MODEL,

        "max_output_tokens":
            LLM_MAX_TOKENS,

        "temperature":
            None,

        "target_answer_words":
            "300-500"
    },

    "prompt": {

        "system_prompt":
            SYSTEM_PROMPT,

        "system_prompt_sha256":
            SYSTEM_PROMPT_HASH,

        "system_prompt_words":
            SYSTEM_PROMPT_WORDS,

        "system_prompt_chars":
            SYSTEM_PROMPT_CHARS
    },

    "knowledge_base": {

        "storage":
            "Persistent DuckDB BM25/FTS shards",

        "persistent_index":
            True,

        "manifest_rows":
            int(
                len(
                    RUNTIME_MANIFEST
                )
            ),

        "persistent_shards":
            int(
                len(
                    ALL_SHARD_NAMES
                )
            ),

        "sources": [
            "GSMA/Telco-Common-Corpus",
            "GSMA/3GPP"
        ],

        "3gpp_shards":
            int(
                len(
                    DEDICATED_3GPP_SHARDS
                )
            ),

        # TCC-only collections.
        # Dedicated GSMA/3GPP specifications are intentionally
        # excluded from this metadata field.
        "tcc_collections":
            sorted(
                RUNTIME_MANIFEST.loc[
                    RUNTIME_MANIFEST[
                        "source_family"
                    ] == "TCC",
                    "collection"
                ]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            )
    },

    "retrieval": {

        "engine":
            VERSION_B_RETRIEVAL_ARCHITECTURE,

        "strategy": (
            "deterministic 3GPP / TCC / hybrid routing + "
            "persistent DuckDB BM25/FTS retrieval + "
            "3GPP specification constraints + "
            "source-focused hybrid retrieval + "
            "source-balanced hybrid Top-K"
        ),

        "bm25_k":
            BM25_K,

        "bm25_b":
            BM25_B,

        "max_candidate_terms":
            ROUTE_MAX_CANDIDATE_TERMS,

        "hybrid_min_results_per_source":
            HYBRID_MIN_RESULTS_PER_SOURCE,

        "top_k":
            TOP_K_RESULTS
    },

    "mcp": {

        "server":
            "Telecom Knowledge Service — Version B",

        "tool":
            "search_telecom_knowledge",

        "max_searches_per_question":
            MAX_MCP_SEARCHES,

        "max_sources_per_search":
            MAX_RETRIEVED_SOURCES,

        "excerpt_chars":
            MCP_EXCERPT_CHARS
    },

    "evidence_capture": {

        "enabled":
            True,

        "location":
            "results[].tool_trace[].evidence",

        "full_source_documents_saved":
            False,

        "bounded_evidence_excerpts_saved":
            True,

        "purpose": [
            "retrieval relevance",
            "source authority",
            "claim-level groundedness",
            "unsupported-claim analysis",
            "technical-quality evaluation"
        ]
    }
}


# ============================================================
# HELPERS
# ============================================================

def _safe_list(value):

    if value is None:
        return []

    if isinstance(
        value,
        list
    ):
        return value

    if isinstance(
        value,
        (tuple, set)
    ):
        return list(
            value
        )

    return [
        value
    ]


def build_version_b_tool_trace(
    result
):
    """
    Build a complete per-search trace.

    Unlike Benchmark v1, the bounded evidence excerpts are
    intentionally retained so Notebook 17 can reconstruct
    retrieval relevance and claim-level groundedness.
    """

    raw_trace = _safe_list(
        result.get(
            "tool_trace",
            result.get(
                "evidence_trace",
                []
            )
        )
    )

    normalized_trace = []

    for index, raw_item in enumerate(
        raw_trace,
        start=1
    ):

        item = (
            raw_item
            if isinstance(
                raw_item,
                dict
            )
            else {}
        )

        evidence = _safe_list(
            item.get(
                "results",
                item.get(
                    "evidence",
                    []
                )
            )
        )

        normalized_trace.append({

            "search_number":
                int(
                    item.get(
                        "search_number",
                        index
                    )
                    or index
                ),

            "query":
                item.get(
                    "query"
                ),

            "top_k":
                item.get(
                    "top_k"
                ),

            "route":
                str(
                    item.get(
                        "route",
                        ""
                    )
                    or ""
                ).lower(),

            "sources_searched":
                _safe_list(
                    item.get(
                        "sources_searched",
                        []
                    )
                ),

            "source_distribution":
                item.get(
                    "source_distribution",
                    {}
                )
                or {},

            "retrieval_trace":
                item.get(
                    "retrieval_trace",
                    {}
                )
                or {},

            "result_count":
                int(
                    item.get(
                        "result_count",
                        len(
                            evidence
                        )
                    )
                    or 0
                ),

            "retrieval_time_s":
                float(
                    item.get(
                        "retrieval_time_s",
                        0.0
                    )
                    or 0.0
                ),

            "tool_elapsed_s":
                float(
                    item.get(
                        "tool_elapsed_s",
                        0.0
                    )
                    or 0.0
                ),

            "mcp_roundtrip_s":
                float(
                    item.get(
                        "mcp_roundtrip_s",
                        0.0
                    )
                    or 0.0
                ),

            "mcp_overhead_s":
                float(
                    item.get(
                        "mcp_overhead_s",
                        0.0
                    )
                    or 0.0
                ),

            # CRITICAL: preserve complete bounded evidence.
            "evidence":
                evidence
        })

    return normalized_trace


def extract_observed_provenance(
    tool_trace
):
    """
    Derive route/source/collection/spec observations from the
    complete Version B search trace without discarding the raw
    bounded evidence.

    Version B stores routing diagnostics directly under:
        retrieval_trace["tcc_collections"]
        retrieval_trace["gpp_specs"]
    """

    routes = []

    source_families = set()

    tcc_collections = set()

    gpp_specs = set()

    evidence_items = 0

    evidence_items_with_text = 0


    for search in tool_trace:

        route = str(
            search.get(
                "route",
                ""
            )
            or ""
        ).lower()


        if route:

            routes.append(
                route
            )


        retrieval_trace = (
            search.get(
                "retrieval_trace",
                {}
            )
            or {}
        )


        for collection in _safe_list(
            retrieval_trace.get(
                "tcc_collections",
                []
            )
        ):

            if collection:

                tcc_collections.add(
                    str(
                        collection
                    )
                )


        for spec in _safe_list(
            retrieval_trace.get(
                "gpp_specs",
                []
            )
        ):

            if spec:

                gpp_specs.add(
                    str(
                        spec
                    )
                )


        for evidence in _safe_list(
            search.get(
                "evidence",
                []
            )
        ):

            if not isinstance(
                evidence,
                dict
            ):

                continue


            evidence_items += 1


            source_family = str(
                evidence.get(
                    "source_family",
                    ""
                )
                or ""
            )


            if source_family:

                source_families.add(
                    source_family
                )


            if (
                source_family
                ==
                "3GPP"
            ):

                identifier = str(
                    evidence.get(
                        "identifier",
                        ""
                    )
                    or ""
                ).strip()


                if re.fullmatch(
                    r"\d{2}\.\d{3}",
                    identifier
                ):

                    gpp_specs.add(
                        identifier
                    )


            evidence_text = str(
                evidence.get(
                    "evidence",
                    ""
                )
                or ""
            ).strip()


            if evidence_text:

                evidence_items_with_text += 1


    return {

        "routes":
            routes,

        "source_families":
            sorted(
                source_families
            ),

        "tcc_collections":
            sorted(
                tcc_collections
            ),

        "gpp_specs":
            sorted(
                gpp_specs
            ),

        "evidence_items":
            evidence_items,

        "evidence_items_with_text":
            evidence_items_with_text
    }


def route_expectation_met(
    expected_route,
    observed_routes,
    observed_source_families
):
    """
    Hybrid is accepted either as one explicit HYBRID search or
    as separate 3GPP + TCC searches that jointly cover both
    source families.
    """

    expected_route = str(
        expected_route
    ).lower()

    observed_routes = [
        str(
            route
        ).lower()
        for route
        in observed_routes
    ]

    if expected_route == "hybrid":

        return (
            "hybrid"
            in observed_routes
        ) or (
            {"3GPP", "TCC"}
            .issubset(
                set(
                    observed_source_families
                )
            )
        )

    return (
        expected_route
        in observed_routes
    )



def strict_route_adherence_met(
    expected_route,
    observed_routes,
    observed_source_families
):
    """
    Supplemental diagnostic only.

    The frozen route_match metric remains unchanged.

    Single-domain:
        every executed search must stay on the expected route.

    Hybrid:
        every route must be one of hybrid/3gpp/tcc AND the
        accumulated evidence must contain both source families.
    """

    expected_route = str(
        expected_route
    ).lower()


    observed_routes = [
        str(
            route
        ).lower()

        for route
        in observed_routes

        if str(
            route
        ).strip()
    ]


    if not observed_routes:

        return False


    if expected_route == "hybrid":

        return (
            all(
                route
                in
                {
                    "hybrid",
                    "3gpp",
                    "tcc"
                }

                for route
                in observed_routes
            )
            and
            {"3GPP", "TCC"}
            .issubset(
                set(
                    observed_source_families
                )
            )
        )


    return all(
        route
        ==
        expected_route

        for route
        in observed_routes
    )


def save_version_b_evaluation_checkpoint(
    results,
    summary_rows,
    aggregate_metrics=None
):

    artifact = {

        "metadata":
            experiment_metadata,

        "aggregate_metrics":
            aggregate_metrics or {},

        "results":
            results
    }

    with open(
        VERSION_B_JSON_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            artifact,
            f,
            indent=2,
            ensure_ascii=False,
            default=str
        )

    pd.DataFrame(
        summary_rows
    ).to_csv(
        VERSION_B_CSV_PATH,
        index=False
    )


# ============================================================
# RESULT CONTAINERS
# ============================================================

evaluation_results = []
summary_rows = []


# ============================================================
# EVALUATION HEADER
# ============================================================

print("=" * 100)
print("VERSION B — MODULE 3 EVALUATION BENCHMARK V2")
print("=" * 100)

print(f"Benchmark             : {BENCHMARK_VERSION}")
print(f"Composition           : 5 × 3GPP | 2 × TCC | 1 × HYBRID")
print(f"Questions             : {len(VERSION_B_EVAL_QUESTIONS)}")
print(f"Model                 : {LLM_MODEL}")
print(f"Architecture          : {VERSION_B_RETRIEVAL_ARCHITECTURE}")
print(f"Top-K / Search        : {TOP_K_RESULTS}")
print(f"Max MCP Searches      : {MAX_MCP_SEARCHES}")
print(f"Evidence / Source     : {MCP_EXCERPT_CHARS:,} chars")
print(f"Benchmark SHA-256     : {BENCHMARK_SHA256}")
print(f"JSON Artifact         : {VERSION_B_JSON_PATH}")
print(f"CSV Summary           : {VERSION_B_CSV_PATH}")

print("=" * 100)


# ============================================================
# RUN EVALUATION
# ============================================================

evaluation_start = time.perf_counter()


async with Client(
    mcp
) as eval_mcp_client:

    for item in VERSION_B_EVAL_QUESTIONS:

        question_id = item[
            "id"
        ]

        domain = item[
            "domain"
        ]

        knowledge_scope = item[
            "knowledge_scope"
        ]

        expected_route = item[
            "expected_route"
        ]

        expected_source_families = item[
            "expected_source_families"
        ]

        expected_tcc_collections = item[
            "expected_tcc_collections"
        ]

        expected_3gpp_specs = item[
            "expected_3gpp_specs"
        ]

        expected_elements = item[
            "expected_elements"
        ]

        question = item[
            "question"
        ]

        print(
            "\n" + "=" * 100
        )

        print(
            f"{question_id} | "
            f"{domain} | "
            f"Expected Route: {expected_route.upper()}"
        )

        print("=" * 100)
        print(f"QUESTION:\n{question}")

        try:

            result = await run_version_b_e2e(

                mcp_client=
                    eval_mcp_client,

                question=
                    question,

                verbose=
                    False
            )

            completed = bool(
                result.get(
                    "completed",
                    False
                )
            )

            stop_reason = result.get(
                "stop_reason"
            )

            raw_finish_reason = result.get(
                "raw_finish_reason"
            )

            completion_mode = result.get(
                "completion_mode"
            )

            execution_errors = _safe_list(
                result.get(
                    "errors",
                    []
                )
            )

            if (
                completed
                and
                stop_reason == "end_turn"
                and
                not execution_errors
            ):

                status = "COMPLETED"

            else:

                status = "CHECK"


            # ----------------------------------------------------
            # COMPLETE SEARCH TRACE
            # ----------------------------------------------------

            tool_trace = build_version_b_tool_trace(
                result
            )

            provenance = extract_observed_provenance(
                tool_trace
            )

            routes_used = provenance[
                "routes"
            ]

            observed_source_families = provenance[
                "source_families"
            ]

            observed_tcc_collections = provenance[
                "tcc_collections"
            ]

            observed_3gpp_specs = provenance[
                "gpp_specs"
            ]

            evidence_items = provenance[
                "evidence_items"
            ]

            evidence_items_with_text = provenance[
                "evidence_items_with_text"
            ]


            # ----------------------------------------------------
            # ROUTING / SOURCE VALIDATION
            # ----------------------------------------------------

            route_match = route_expectation_met(
                expected_route=
                    expected_route,

                observed_routes=
                    routes_used,

                observed_source_families=
                    observed_source_families
            )


            strict_route_adherence = (
                strict_route_adherence_met(
                    expected_route=
                        expected_route,

                    observed_routes=
                        routes_used,

                    observed_source_families=
                        observed_source_families
                )
            )


            source_family_match = set(
                expected_source_families
            ).issubset(
                set(
                    observed_source_families
                )
            )

            tcc_collection_match = (

                set(
                    expected_tcc_collections
                ).issubset(
                    set(
                        observed_tcc_collections
                    )
                )

                if expected_tcc_collections

                else True
            )

            evidence_text_coverage_pct = (

                evidence_items_with_text
                /
                evidence_items
                *
                100.0

                if evidence_items > 0

                else 0.0
            )


            # ----------------------------------------------------
            # ORCHESTRATION METRICS
            # ----------------------------------------------------

            mcp_searches = int(
                result.get(
                    "mcp_searches",
                    0
                )
                or 0
            )

            mcp_tool_requests = int(
                result.get(
                    "mcp_tool_requests",
                    mcp_searches
                )
                or 0
            )

            llm_turns = int(
                result.get(
                    "llm_calls",
                    result.get(
                        "claude_calls",
                        0
                    )
                )
                or 0
            )

            if mcp_searches == 1:
                search_classification = "IDEAL"
            elif mcp_searches == 2:
                search_classification = "ACCEPTABLE"
            elif mcp_searches == 3:
                search_classification = "MAXIMUM"
            else:
                search_classification = "UNEXPECTED"

            search_queries = _safe_list(
                result.get(
                    "tool_queries",
                    []
                )
            )

            sources_used = _safe_list(
                result.get(
                    "tool_sources",
                    []
                )
            )

            source_distributions = _safe_list(
                result.get(
                    "tool_source_distributions",
                    []
                )
            )


            candidate_terms_per_search = [

                _safe_list(
                    (
                        search.get(
                            "retrieval_trace",
                            {}
                        )
                        or
                        {}
                    ).get(
                        "candidate_terms",
                        []
                    )
                )

                for search
                in tool_trace
            ]


            # ----------------------------------------------------
            # SOURCE PAYLOAD
            # ----------------------------------------------------

            total_sources_returned = sum(
                int(
                    search.get(
                        "result_count",
                        0
                    )
                    or 0
                )
                for search
                in tool_trace
            )

            mean_sources_per_search = (

                total_sources_returned
                /
                mcp_searches

                if mcp_searches > 0

                else 0.0
            )


            # ----------------------------------------------------
            # LATENCY
            # ----------------------------------------------------

            retrieval_latency = float(
                result.get(
                    "total_retrieval_time_s",
                    0.0
                )
                or 0.0
            )

            total_mcp_roundtrip = float(
                result.get(
                    "total_mcp_roundtrip_s",
                    0.0
                )
                or 0.0
            )

            total_mcp_overhead = float(
                result.get(
                    "total_mcp_overhead_s",
                    0.0
                )
                or 0.0
            )

            total_latency = float(
                result.get(
                    "e2e_time_s",
                    0.0
                )
                or 0.0
            )

            mean_retrieval_latency_per_search = (

                retrieval_latency
                /
                mcp_searches

                if mcp_searches > 0

                else 0.0
            )

            non_retrieval_latency = max(
                0.0,
                total_latency
                -
                retrieval_latency
            )

            retrieval_latency_share_pct = (

                retrieval_latency
                /
                total_latency
                *
                100.0

                if total_latency > 0

                else 0.0
            )


            # ----------------------------------------------------
            # TOKENS / ANSWER
            # ----------------------------------------------------

            input_tokens = int(
                result.get(
                    "input_tokens",
                    0
                )
                or 0
            )

            output_tokens = int(
                result.get(
                    "output_tokens",
                    0
                )
                or 0
            )

            total_tokens = int(
                result.get(
                    "total_tokens",
                    input_tokens
                    +
                    output_tokens
                )
                or 0
            )

            final_answer_tokens = int(
                result.get(
                    "final_answer_tokens",
                    0
                )
                or 0
            )

            answer_words = int(
                result.get(
                    "word_count",
                    0
                )
                or 0
            )

            answer = str(
                result.get(
                    "final_answer",
                    ""
                )
                or ""
            )


            # ----------------------------------------------------
            # SAVE COMPLETE QUESTION RESULT
            # ----------------------------------------------------

            saved_result = {

                # Benchmark definition for this question.
                "id":
                    question_id,

                "domain":
                    domain,

                "knowledge_scope":
                    knowledge_scope,

                "question":
                    question,

                "expected_route":
                    expected_route,

                "expected_source_families":
                    expected_source_families,

                "expected_tcc_collections":
                    expected_tcc_collections,

                "expected_3gpp_specs":
                    expected_3gpp_specs,

                "expected_elements":
                    expected_elements,

                # Completion.
                "status":
                    status,

                "completion_mode":
                    completion_mode,

                "stop_reason":
                    stop_reason,

                "raw_finish_reason":
                    raw_finish_reason,

                # Response.
                "answer":
                    answer,

                "answer_words":
                    answer_words,

                "final_answer_tokens":
                    final_answer_tokens,

                # Routing / provenance observations.
                "routes_used":
                    routes_used,

                "route_match":
                    bool(
                        route_match
                    ),

                "strict_route_adherence":
                    bool(
                        strict_route_adherence
                    ),

                "observed_source_families":
                    observed_source_families,

                "source_family_match":
                    bool(
                        source_family_match
                    ),

                "observed_tcc_collections":
                    observed_tcc_collections,

                "tcc_collection_match":
                    bool(
                        tcc_collection_match
                    ),

                "observed_3gpp_specs":
                    observed_3gpp_specs,

                # Orchestration.
                "mcp_searches":
                    mcp_searches,

                "mcp_tool_requests":
                    mcp_tool_requests,

                "search_classification":
                    search_classification,

                # Compatibility alias retained for the shared
                # Version A/B analysis notebook.
                "claude_turns":
                    llm_turns,

                "llm_turns":
                    llm_turns,

                "search_queries":
                    search_queries,

                # Retain compatibility with Benchmark v1 /
                # Version B analysis fields.
                "profiles_used":
                    [],

                "candidate_terms_per_search":
                    candidate_terms_per_search,

                "sources_used":
                    sources_used,

                "source_distributions":
                    source_distributions,

                # CRITICAL: complete bounded evidence lives here.
                "tool_trace":
                    tool_trace,

                "turn_usage":
                    result.get(
                        "turn_usage",
                        []
                    ),

                # Evidence capture diagnostics.
                "evidence_items":
                    evidence_items,

                "evidence_items_with_text":
                    evidence_items_with_text,

                "evidence_text_coverage_pct":
                    round(
                        evidence_text_coverage_pct,
                        1
                    ),

                # Retrieval / MCP timing.
                "retrieval_latency_sec":
                    round(
                        retrieval_latency,
                        3
                    ),

                "mean_retrieval_latency_per_search_sec":
                    round(
                        mean_retrieval_latency_per_search,
                        3
                    ),

                "total_sources_returned":
                    total_sources_returned,

                "mean_sources_per_search":
                    round(
                        mean_sources_per_search,
                        2
                    ),

                "mcp_roundtrip_sec":
                    round(
                        total_mcp_roundtrip,
                        3
                    ),

                "mcp_overhead_sec":
                    round(
                        total_mcp_overhead,
                        3
                    ),

                "total_latency_sec":
                    round(
                        total_latency,
                        3
                    ),

                "non_retrieval_latency_sec":
                    round(
                        non_retrieval_latency,
                        3
                    ),

                "retrieval_latency_share_pct":
                    round(
                        retrieval_latency_share_pct,
                        1
                    ),

                # Tokens.
                "input_tokens":
                    input_tokens,

                "output_tokens":
                    output_tokens,

                "total_tokens":
                    total_tokens,

                # Errors.
                "errors":
                    execution_errors
            }

            evaluation_results.append(
                saved_result
            )


            # ----------------------------------------------------
            # SUMMARY ROW
            # ----------------------------------------------------

            summary_rows.append({

                "question_id":
                    question_id,

                "domain":
                    domain,

                "knowledge_scope":
                    knowledge_scope,

                "expected_route":
                    expected_route,

                "routes_used":
                    ",".join(
                        routes_used
                    ),

                "route_match":
                    bool(
                        route_match
                    ),

                "strict_route_adherence":
                    bool(
                        strict_route_adherence
                    ),

                "observed_source_families":
                    ",".join(
                        observed_source_families
                    ),

                "source_family_match":
                    bool(
                        source_family_match
                    ),

                "status":
                    status,

                "stop_reason":
                    stop_reason,

                "raw_finish_reason":
                    raw_finish_reason,

                "mcp_searches":
                    mcp_searches,

                "search_classification":
                    search_classification,

                "claude_turns":
                    llm_turns,

                "llm_turns":
                    llm_turns,

                "total_sources_returned":
                    total_sources_returned,

                "evidence_items":
                    evidence_items,

                "evidence_items_with_text":
                    evidence_items_with_text,

                "evidence_text_coverage_pct":
                    round(
                        evidence_text_coverage_pct,
                        1
                    ),

                "mean_sources_per_search":
                    round(
                        mean_sources_per_search,
                        2
                    ),

                "retrieval_latency_sec":
                    round(
                        retrieval_latency,
                        3
                    ),

                "mean_retrieval_latency_per_search_sec":
                    round(
                        mean_retrieval_latency_per_search,
                        3
                    ),

                "total_latency_sec":
                    round(
                        total_latency,
                        3
                    ),

                "non_retrieval_latency_sec":
                    round(
                        non_retrieval_latency,
                        3
                    ),

                "retrieval_latency_share_pct":
                    round(
                        retrieval_latency_share_pct,
                        1
                    ),

                "input_tokens":
                    input_tokens,

                "output_tokens":
                    output_tokens,

                "total_tokens":
                    total_tokens,

                "final_answer_tokens":
                    final_answer_tokens,

                "answer_words":
                    answer_words
            })


            # ----------------------------------------------------
            # DISPLAY
            # ----------------------------------------------------

            print("\nFINAL ANSWER")
            print("-" * 100)
            print(answer)

            print("\nMETRICS")
            print("-" * 100)
            print(f"Status                     : {status}")
            print(f"Raw Finish Reason          : {raw_finish_reason}")
            print(f"Expected Route             : {expected_route.upper()}")
            print(f"Observed Routes            : {routes_used}")
            print(f"Route Match                : {route_match}")
            print(f"Strict Route Adherence     : {strict_route_adherence}")
            print(f"Observed Sources           : {observed_source_families}")
            print(f"Source Match               : {source_family_match}")
            print(f"MCP Searches               : {mcp_searches}")
            print(f"Evidence Items             : {evidence_items}")
            print(f"Evidence With Text         : {evidence_items_with_text}")
            print(f"Evidence Text Coverage     : {evidence_text_coverage_pct:.1f}%")
            print(f"Retrieval Latency          : {retrieval_latency:.3f} sec")
            print(f"Total E2E Latency          : {total_latency:.3f} sec")
            print(f"Total Tokens               : {total_tokens:,}")
            print(f"Answer Words               : {answer_words:,}")
            print(f"Execution Errors           : {len(execution_errors)}")


        except Exception as exc:

            error_text = (
                f"{type(exc).__name__}: "
                f"{exc}"
            )

            evaluation_results.append({

                "id":
                    question_id,

                "domain":
                    domain,

                "knowledge_scope":
                    knowledge_scope,

                "question":
                    question,

                "expected_route":
                    expected_route,

                "expected_source_families":
                    expected_source_families,

                "expected_tcc_collections":
                    expected_tcc_collections,

                "expected_3gpp_specs":
                    expected_3gpp_specs,

                "expected_elements":
                    expected_elements,

                "status":
                    "ERROR",

                "error":
                    error_text
            })

            summary_rows.append({

                "question_id":
                    question_id,

                "domain":
                    domain,

                "knowledge_scope":
                    knowledge_scope,

                "expected_route":
                    expected_route,

                "routes_used":
                    "",

                "route_match":
                    False,

                "strict_route_adherence":
                    False,

                "observed_source_families":
                    "",

                "source_family_match":
                    False,

                "status":
                    "ERROR",

                "stop_reason":
                    None,

                "mcp_searches":
                    None,

                "search_classification":
                    None,

                "claude_turns":
                    None,

                "llm_turns":
                    None,

                "total_sources_returned":
                    None,

                "evidence_items":
                    None,

                "evidence_items_with_text":
                    None,

                "evidence_text_coverage_pct":
                    None,

                "mean_sources_per_search":
                    None,

                "retrieval_latency_sec":
                    None,

                "mean_retrieval_latency_per_search_sec":
                    None,

                "total_latency_sec":
                    None,

                "non_retrieval_latency_sec":
                    None,

                "retrieval_latency_share_pct":
                    None,

                "input_tokens":
                    None,

                "output_tokens":
                    None,

                "total_tokens":
                    None,

                "final_answer_tokens":
                    None,

                "answer_words":
                    None
            })

            print("\nERROR")
            print("-" * 100)
            print(error_text)


        # --------------------------------------------------------
        # CHECKPOINT AFTER EVERY QUESTION
        # --------------------------------------------------------

        save_version_b_evaluation_checkpoint(
            evaluation_results,
            summary_rows
        )

        print(
            f"\n✓ {question_id} checkpoint saved."
        )


# ============================================================
# FINAL AGGREGATES
# ============================================================

evaluation_elapsed = (
    time.perf_counter()
    -
    evaluation_start
)

version_b_evaluation_summary = pd.DataFrame(
    summary_rows
)

successful_df = (
    version_b_evaluation_summary[
        version_b_evaluation_summary[
            "status"
        ] != "ERROR"
    ]
    .copy()
)

total_questions = len(
    VERSION_B_EVAL_QUESTIONS
)

completed_count = int(
    version_b_evaluation_summary[
        "status"
    ]
    .eq(
        "COMPLETED"
    )
    .sum()
)

check_count = int(
    version_b_evaluation_summary[
        "status"
    ]
    .eq(
        "CHECK"
    )
    .sum()
)

error_count = int(
    version_b_evaluation_summary[
        "status"
    ]
    .eq(
        "ERROR"
    )
    .sum()
)

completion_rate_pct = (
    completed_count
    /
    total_questions
    *
    100.0
)

aggregate_metrics = {

    "questions":
        total_questions,

    "completed":
        completed_count,

    "check":
        check_count,

    "errors":
        error_count,

    "completion_rate_pct":
        round(
            completion_rate_pct,
            1
        ),

    "evaluation_wall_time_sec":
        round(
            evaluation_elapsed,
            3
        )
}


if not successful_df.empty:

    total_mcp_searches = int(
        successful_df[
            "mcp_searches"
        ].sum()
    )

    total_sources_returned = int(
        successful_df[
            "total_sources_returned"
        ].sum()
    )

    route_accuracy_pct = (
        successful_df[
            "route_match"
        ].astype(
            bool
        ).mean()
        *
        100.0
    )


    strict_route_adherence_pct = (
        successful_df[
            "strict_route_adherence"
        ].astype(
            bool
        ).mean()
        *
        100.0
    )


    source_family_accuracy_pct = (
        successful_df[
            "source_family_match"
        ].astype(
            bool
        ).mean()
        *
        100.0
    )

    evidence_text_coverage_pct = (
        successful_df[
            "evidence_items_with_text"
        ].sum()
        /
        successful_df[
            "evidence_items"
        ].sum()
        *
        100.0

        if successful_df[
            "evidence_items"
        ].sum() > 0

        else 0.0
    )

    one_search_count = int(
        successful_df[
            "mcp_searches"
        ].eq(
            1
        ).sum()
    )

    two_search_count = int(
        successful_df[
            "mcp_searches"
        ].eq(
            2
        ).sum()
    )

    three_search_count = int(
        successful_df[
            "mcp_searches"
        ].eq(
            3
        ).sum()
    )

    global_retrieval_latency_per_search = (

        successful_df[
            "retrieval_latency_sec"
        ].sum()
        /
        total_mcp_searches

        if total_mcp_searches > 0

        else 0.0
    )

    aggregate_metrics.update({

        "route_accuracy_pct":
            round(
                route_accuracy_pct,
                1
            ),

        "strict_route_adherence_pct":
            round(
                strict_route_adherence_pct,
                1
            ),

        "source_family_accuracy_pct":
            round(
                source_family_accuracy_pct,
                1
            ),

        "evidence_text_coverage_pct":
            round(
                evidence_text_coverage_pct,
                1
            ),

        "total_mcp_searches":
            total_mcp_searches,

        "mean_mcp_searches_per_question":
            round(
                successful_df[
                    "mcp_searches"
                ].mean(),
                3
            ),

        "one_search_questions":
            one_search_count,

        "two_search_questions":
            two_search_count,

        "three_search_questions":
            three_search_count,

        "total_sources_returned":
            total_sources_returned,

        "mean_sources_per_search":
            round(
                (
                    total_sources_returned
                    /
                    total_mcp_searches
                )
                if total_mcp_searches > 0
                else 0.0,
                2
            ),

        "total_retrieval_latency_sec":
            round(
                successful_df[
                    "retrieval_latency_sec"
                ].sum(),
                3
            ),

        "mean_retrieval_latency_sec":
            round(
                successful_df[
                    "retrieval_latency_sec"
                ].mean(),
                3
            ),

        "median_retrieval_latency_sec":
            round(
                successful_df[
                    "retrieval_latency_sec"
                ].median(),
                3
            ),

        "global_retrieval_latency_per_search_sec":
            round(
                global_retrieval_latency_per_search,
                3
            ),

        "total_e2e_latency_sec":
            round(
                successful_df[
                    "total_latency_sec"
                ].sum(),
                3
            ),

        "mean_e2e_latency_sec":
            round(
                successful_df[
                    "total_latency_sec"
                ].mean(),
                3
            ),

        "median_e2e_latency_sec":
            round(
                successful_df[
                    "total_latency_sec"
                ].median(),
                3
            ),

        "total_input_tokens":
            int(
                successful_df[
                    "input_tokens"
                ].sum()
            ),

        "total_output_tokens":
            int(
                successful_df[
                    "output_tokens"
                ].sum()
            ),

        "total_tokens":
            int(
                successful_df[
                    "total_tokens"
                ].sum()
            ),

        "mean_total_tokens_per_question":
            round(
                successful_df[
                    "total_tokens"
                ].mean(),
                1
            ),

        "mean_answer_words":
            round(
                successful_df[
                    "answer_words"
                ].mean(),
                1
            ),

        "median_answer_words":
            round(
                successful_df[
                    "answer_words"
                ].median(),
                1
            )
    })


# ============================================================
# FINAL SAVE
# ============================================================

evaluation_artifact = {

    "metadata":
        experiment_metadata,

    "aggregate_metrics":
        aggregate_metrics,

    "results":
        evaluation_results
}

with open(
    VERSION_B_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        evaluation_artifact,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

version_b_evaluation_summary.to_csv(
    VERSION_B_CSV_PATH,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "VERSION B — BENCHMARK V2 QUESTION-LEVEL SUMMARY"
)

print("=" * 100)

display(
    version_b_evaluation_summary
)

print(
    "\n" + "=" * 100
)

print(
    "VERSION B — BENCHMARK V2 AGGREGATE SUMMARY"
)

print("=" * 100)

print(f"Questions                    : {total_questions}")
print(f"Completed                    : {completed_count}")
print(f"Check                        : {check_count}")
print(f"Errors                       : {error_count}")
print(f"Completion Rate              : {completion_rate_pct:.1f}%")

if not successful_df.empty:

    print(
        f"Route Accuracy               : "
        f"{aggregate_metrics['route_accuracy_pct']:.1f}%"
    )

    print(
        f"Strict Route Adherence       : "
        f"{aggregate_metrics['strict_route_adherence_pct']:.1f}%"
    )

    print(
        f"Source-Family Accuracy       : "
        f"{aggregate_metrics['source_family_accuracy_pct']:.1f}%"
    )

    print(
        f"Evidence Text Coverage       : "
        f"{aggregate_metrics['evidence_text_coverage_pct']:.1f}%"
    )

    print(
        f"Total MCP Searches           : "
        f"{aggregate_metrics['total_mcp_searches']}"
    )

    print(
        f"Mean MCP Searches / Question : "
        f"{aggregate_metrics['mean_mcp_searches_per_question']:.2f}"
    )

    print(
        f"Mean Retrieval Latency       : "
        f"{aggregate_metrics['mean_retrieval_latency_sec']:.3f} sec"
    )

    print(
        f"Mean E2E Latency             : "
        f"{aggregate_metrics['mean_e2e_latency_sec']:.3f} sec"
    )

    print(
        f"Total Tokens                 : "
        f"{aggregate_metrics['total_tokens']:,}"
    )

    print(
        f"Mean Answer Length           : "
        f"{aggregate_metrics['mean_answer_words']:.1f} words"
    )

print(
    f"Evaluation Wall Time         : "
    f"{evaluation_elapsed:.3f} sec"
)

print(
    f"Canonical JSON              : "
    f"{VERSION_B_JSON_PATH}"
)

print(
    f"Summary CSV                 : "
    f"{VERSION_B_CSV_PATH}"
)

print("=" * 100)

print("✓ Benchmark v2 completed.")
print("✓ 5 × 3GPP, 2 × TCC and 1 × Hybrid questions evaluated.")
print("✓ Benchmark definition and SHA-256 preserved.")
print("✓ Expected and observed routing preserved.")
print("✓ Full bounded evidence excerpts preserved in tool_trace.")
print("✓ Retrieval provenance and timing preserved.")
print("✓ Artifact supports relevance and claim-level groundedness evaluation.")
print("✓ Benchmark v1 artifacts remain untouched.")


VERSION B — MODULE 3 EVALUATION BENCHMARK V2
Benchmark             : module3_eval_v2
Composition           : 5 × 3GPP | 2 × TCC | 1 × HYBRID
Questions             : 8
Model                 : deepseek/deepseek-v4-flash-0731
Architecture          : Persistent DuckDB BM25/FTS
Top-K / Search        : 5
Max MCP Searches      : 3
Evidence / Source     : 2,500 chars
Benchmark SHA-256     : d40c0090c371f0a99ea6057bbb3fa8024e6b7d4174e037e7a6cf9fef9b9667f5
JSON Artifact         : /content/version_b_evaluation/version_b_deepseek_v4_flash_0731_evaluation_v2.json
CSV Summary           : /content/version_b_evaluation/version_b_deepseek_v4_flash_0731_summary_v2.csv

Q1 | 5G Core | Expected Route: 3GPP
QUESTION:
Explain the role of the AMF in registration and mobility management procedures in a 5G Standalone network.

FINAL ANSWER
----------------------------------------------------------------------------------------------------
In a 5G Standalone (SA) network, the AMF (Access and Mobility Management

,question_id,domain,knowledge_scope,expected_route,routes_used,route_match,strict_route_adherence,observed_source_families,source_family_match,status,...,retrieval_latency_sec,mean_retrieval_latency_per_search_sec,total_latency_sec,non_retrieval_latency_sec,retrieval_latency_share_pct,input_tokens,output_tokens,total_tokens,final_answer_tokens,answer_words
0,Q1,5G Core,3GPP,3gpp,"3gpp,3gpp,3gpp",True,True,3GPP,True,CHECK,...,5.468,1.823,119.140,113.672,4.6,37268,2453,39721,1800,223
1,Q2,Mobility,3GPP,3gpp,"3gpp,3gpp,3gpp",True,True,3GPP,True,COMPLETED,...,5.290,1.763,90.840,85.551,5.8,36262,2264,38526,1387,459
2,Q3,RAN,3GPP,3gpp,3gpp,True,True,3GPP,True,COMPLETED,...,0.534,0.534,42.756,42.222,1.2,5519,1451,6970,1337,423
3,Q4,QoS,3GPP,3gpp,"3gpp,3gpp",True,True,3GPP,True,COMPLETED,...,3.565,1.783,104.993,101.428,3.4,18633,1475,20108,1114,466
4,Q5,Security,3GPP,3gpp,"3gpp,3gpp",True,True,3GPP,True,COMPLETED,...,3.270,1.635,21.949,18.679,14.9,17874,1467,19341,970,464
5,Q6,Internet Transport,TCC,tcc,tcc,True,True,TCC,True,COMPLETED,...,1.769,1.769,56.440,54.671,3.1,6261,1278,7539,1159,447
6,Q7,Telecom AI Research,TCC,tcc,"tcc,tcc,tcc",True,True,TCC,True,COMPLETED,...,1.288,0.429,30.138,28.850,4.3,23957,1213,25170,876,311
7,Q8,5G SBA + Internet Protocols,Hybrid,hybrid,"hybrid,3gpp,3gpp",True,True,"3GPP,TCC",True,COMPLETED,...,4.049,1.350,125.452,121.403,3.2,24419,2092,26511,1531,442



VERSION B — BENCHMARK V2 AGGREGATE SUMMARY
Questions                    : 8
Completed                    : 7
Check                        : 1
Errors                       : 0
Completion Rate              : 87.5%
Route Accuracy               : 100.0%
Strict Route Adherence       : 100.0%
Source-Family Accuracy       : 100.0%
Evidence Text Coverage       : 100.0%
Total MCP Searches           : 18
Mean MCP Searches / Question : 2.25
Mean Retrieval Latency       : 3.154 sec
Mean E2E Latency             : 73.964 sec
Total Tokens                 : 183,886
Mean Answer Length           : 404.4 words
Evaluation Wall Time         : 591.784 sec
Canonical JSON              : /content/version_b_evaluation/version_b_deepseek_v4_flash_0731_evaluation_v2.json
Summary CSV                 : /content/version_b_evaluation/version_b_deepseek_v4_flash_0731_summary_v2.csv
✓ Benchmark v2 completed.
✓ 5 × 3GPP, 2 × TCC and 1 × Hybrid questions evaluated.
✓ Benchmark definition and SHA-256 preserved.
✓ Expecte

**Observation — DeepSeek V4 Flash 0731 Version B Benchmark v2**

- **7/8** questions were `COMPLETED`, with **1 CHECK** outcome(s) and **0 execution errors**.
- Route accuracy: **100.0%**; strict route adherence: **100.0%**; source-family accuracy: **100.0%**; evidence-text coverage: **100.0%**.
- MCP searches: **18** (**2.25/question**).
- Mean retrieval latency: **3.154 s**; mean end-to-end latency: **73.964 s**.
- Total tokens: **183,886**; mean final response length: **404.4 words**.

Q1 remains correctly marked CHECK because final generation hit the frozen **1,800-token ceiling** after three valid 3GPP searches. Retrieval was relatively fast while mean end-to-end latency reached **73.964 s**, showing that most delay occurred after retrieval in the model/provider inference loop. Q7 missed IEEE/OpenAlex but abstained. This untuned baseline motivates later **model-specific runtime configuration**—for example reasoning/thinking mode, search policy and token budget—by use case.

***Key Finding:*** The common MCP interface and frozen retrieval service do not produce identical model behaviour. Query formulation, search discipline, latency and response behaviour remain model-dependent and must be evaluated separately from retrieval correctness.
